# DA price forecasting — thesis experiments

Day-ahead price forecasting on DK1, organised so each model family runs through every
feature set and writes its test predictions into a shared dataframe.

**Pipeline**

1. Setup & config
2. Load all signals (one `read_signals` call, point-in-time correct)
3. Align PT60M signals to the PT15M target grid
4. Time / calendar features
5. Price features (own zone + neighbouring zones)
6. Fundamentals features
7. DST repair + warm-up trim
8. Folds & final test split
9. Experiment engine — model registry by *family* → *specific model*
10. Run every (family × model × feature-set) combination

**Output**

A single `test_predictions_df` with columns

```
actual | baseline_persistence_24h | xgb_fs1 | xgb_fs2 | xgb_fs3 | lgbm_fs1 | ... | enet_fs3
```

plus a `results_df` table summarising MAE / RMSE / R² per run.

## 1. Setup & config

In [ ]:
from __future__ import annotations

import os
from collections import Counter
from dataclasses import dataclass
from datetime import time, timedelta
from pathlib import Path
from typing import Any, Callable

import holidays
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd

from optuna.trial import TrialState
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from signalforge.signals import S, lag, latest, read_signals, snapshot

import re

In [ ]:
# ----- Top-level config -----------------------------------------------------

# Package-relative paths for the cleaned public copy.
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PACKAGE_ROOT = NOTEBOOK_DIR.parent
else:
    PACKAGE_ROOT = NOTEBOOK_DIR
PROJECT_ROOT = PACKAGE_ROOT.parent

TZ              = "Europe/Copenhagen"
TARGET_ZONE     = "dk1"
NEIGHBOR_ZONES  = ("dk2", "se3", "no2", "de_lu")
TARGET_COL      = "entso_e.dk1.price_da_pt15m"

PERIOD_START    = "2025-10-01"
PERIOD_END      = "2026-04-15"
WARMUP_DAYS     = 7

# PIT vintage rule for fundamentals (snapshot D-N {time_of_day} {tz}).
#
# The strict bidder-at-gate convention is D-1 12:00 Brussels (SDAC gate
# closure). However, the existing signal_values data has `available_at` stamped
# at the *legal* publication deadline (D-1 17:00 Brussels for wind/solar
# forecasts), not the realistic TSO morning publication time (~06:00-09:00
# Brussels). Until `scripts/restamp_forecast_available_at.py` is run, the
# strict cutoff returns zero rows for the forecast covariates.
#
# Default below: D-1 23:00 Brussels — academic EPF convention, equivalent to
# "knowable before delivery day starts in UTC". Captures all current data.
# Switch to time(12, 0) once the restamp script has been applied.
DA_GATE = dict(days_ahead=1, time_of_day=time(23, 0), tz="Europe/Brussels")

# ----- CV config ------------------------------------------------------------
TEST_DAYS = 28
VAL_DAYS  = 7
N_SPLITS  = 5
GAP_DAYS  = 0

# ----- Optuna config --------------------------------------------------------
N_TRIALS = 100
SEED     = 42

# ----- Test-prediction persistence ------------------------------------------
PREDICTIONS_PARQUET = "test_predictions_df.parquet"
RESULTS_PARQUET     = "results_df.parquet"

## 2. Load all signals

Single `read_signals` call covers the target, neighbouring zones, fundamentals
forecasts (PIT-safe via `snapshot`), and 24h-lagged actuals (via `lag`).

Column naming follows `signalforge.signals.read_signals` conventions:
- `latest()` returns the column under the slug name.
- `lag(..., delta=24h)` appends `_lag_1d` to the slug name.
- `snapshot(...)` returns the column under the slug name.

PT60M signals (`load_*`, `dk1_to_gb_scheduled`) come back with NaNs in the 3
quarters following each natural hour — those are filled in section 3.

> **PIT caveat — `lag()` and revised data.** The `lag()` filter applies
> `latest()` semantics under the hood: it pulls the most-recent revision per
> `valid_from` and then pandas-shifts the series by the requested delta. For
> covariates that get revised post-publication (notably ENTSO-E `load_actual`
> — first publication a few hours after delivery, with possibly several
> revisions over subsequent days), this means a training row at delivery
> time `D` could be using a revision of `D-1` actuals that landed *after*
> our `DA_GATE` cutoff. The leakage is small in practice (ENTSO-E load
> revisions are typically minor and quick), but for strict thesis
> reporting either (a) restrict to forecast covariates only, (b) lag by ≥
> the maximum revision-arrival horizon, or (c) extend the `lag()` filter
> with a snapshot semantics. The PIT assertion in section 7 catches snapshot
> violations but does *not* test for late-arriving lag revisions.

In [ ]:
DA_SCHED_FLOWS = {
    "dk1_to_dk2":  S.energy.entso_e.flows.dk1_to_dk2.scheduled_pt15m,
    "dk1_to_se3":  S.energy.entso_e.flows.dk1_to_se3.scheduled_pt15m,
    "dk1_to_no2":  S.energy.entso_e.flows.dk1_to_no2.scheduled_pt15m,
    "dk1_to_nl":   S.energy.entso_e.flows.dk1_to_nl.scheduled_pt15m,
    "dk1_to_gb":   S.energy.entso_e.flows.dk1_to_gb.scheduled_pt60m,
    "de_lu_to_dk1": S.energy.entso_e.flows.de_lu_to_dk1.scheduled_pt15m,
}

# DK1 fundamentals (DA forecasts + DK1 total generation forecast).
DA_DK1_FORECASTS = [
    S.energy.entso_e.dk1.load_forecast_pt60m,
    S.energy.entso_e.dk1.wind_onshore_forecast_pt15m,
    S.energy.entso_e.dk1.wind_offshore_forecast_pt15m,
    S.energy.entso_e.dk1.solar_forecast_pt15m,
    S.energy.entso_e.dk1.actual_aggregated_forecast_pt60m,    # NEW: DK1 total gen forecast
]

# Neighbour-zone DA forecasts. SE3 is intentionally excluded: its PT15M
# forecast signals only start 2025-12-02 (Nordic 15-min market go-live),
# while the legacy PT60M variants are not exposed in the catalog. Including
# them would shrink the dev window by ~2 months. SE3 still contributes via
# its DA price.
DA_NEIGHBOUR_FORECASTS = [
    # ---- DK2 (PT60M for load + total gen, PT15M for wind/solar) ----
    S.energy.entso_e.dk2.load_forecast_pt60m,
    S.energy.entso_e.dk2.wind_onshore_forecast_pt15m,
    S.energy.entso_e.dk2.wind_offshore_forecast_pt15m,
    S.energy.entso_e.dk2.solar_forecast_pt15m,
    S.energy.entso_e.dk2.actual_aggregated_forecast_pt60m,
    # ---- NO2 (all PT15M) ----
    S.energy.entso_e.no2.load_forecast_pt15m,
    S.energy.entso_e.no2.wind_onshore_forecast_pt15m,
    S.energy.entso_e.no2.wind_offshore_forecast_pt15m,
    S.energy.entso_e.no2.solar_forecast_pt15m,
    S.energy.entso_e.no2.actual_aggregated_forecast_pt15m,
    # ---- DE-LU (all PT15M) ----
    S.energy.entso_e.de_lu.load_forecast_pt15m,
    S.energy.entso_e.de_lu.wind_onshore_forecast_pt15m,
    S.energy.entso_e.de_lu.wind_offshore_forecast_pt15m,
    S.energy.entso_e.de_lu.solar_forecast_pt15m,
    S.energy.entso_e.de_lu.actual_aggregated_forecast_pt15m,
]

# Tier-A Nordpool covariates (DK1 day-ahead microstructure).
# All slugs are PT15M — the live Nord Pool feeds (post 2024-Q3 Nordic 15-min
# market go-live). PT60M variants exist but only carry pre-2024 CSV data,
# which is outside our modelling window.
#
# - DA volumes: cleared at D-1 ~12:42 Brussels (same auction we're predicting).
#   We lag by 24h so yesterday's cleared volume becomes today's feature; using
#   today's would leak.
# - IDA1 / IDA2 volumes: published D-1 15:20 / 22:20 Brussels. Yesterday's IDA
#   is knowable for today's DA prediction. Lagged 24h for symmetry with DA.
# - IDA1 / IDA2 capacities (DK1 borders): published pre-auction. Under our
#   relaxed DA_GATE convention (D-1 23:00 Brussels) they are knowable, so
#   `snapshot` is correct. Reverts to `lag` if the strict 12:00 gate is reinstated.
# - Intraday OHLCV / VWAP-1h: continuous trading feed. Lag 24h ensures the
#   snapshot pre-dates the strict DA gate even before any catalog `available_at`
#   stamps are present.
# Net `da_volume_pt15m` is empty in the live feed (only `buy_volume` and
# `sell_volume` are populated post-2024). We derive net export = sell − buy
# in Section 6b, so net is not lost.
NORDPOOL_DA_VOLUMES_DK1 = [
    S.energy.nordpool.dk1.da_buy_volume_pt15m,
    S.energy.nordpool.dk1.da_sell_volume_pt15m,
]

NORDPOOL_IDA_VOLUMES_DK1 = [
    S.energy.nordpool.dk1.ida1_buy_volume_pt15m,
    S.energy.nordpool.dk1.ida1_sell_volume_pt15m,
    S.energy.nordpool.dk1.ida2_buy_volume_pt15m,
    S.energy.nordpool.dk1.ida2_sell_volume_pt15m,
]

# DK1 borders covered by Nord Pool capacity feeds: DK2, GER (=DE-LU), NL.
# Norway / Sweden are excluded because the live capacity feed routes them
# through the virtual DK1A zone and the PT15M data is not in our modelling
# window.
NORDPOOL_IDA_CAPACITIES = [
    S.energy.nordpool.flows.dk1_to_dk2.ida1_capacity_export_pt15m,
    S.energy.nordpool.flows.dk1_to_dk2.ida1_capacity_import_pt15m,
    S.energy.nordpool.flows.dk1_to_dk2.ida2_capacity_export_pt15m,
    S.energy.nordpool.flows.dk1_to_dk2.ida2_capacity_import_pt15m,
    S.energy.nordpool.flows.dk1_to_ger.ida1_capacity_export_pt15m,
    S.energy.nordpool.flows.dk1_to_ger.ida1_capacity_import_pt15m,
    S.energy.nordpool.flows.dk1_to_ger.ida2_capacity_export_pt15m,
    S.energy.nordpool.flows.dk1_to_ger.ida2_capacity_import_pt15m,
    S.energy.nordpool.flows.dk1_to_nl.ida1_capacity_export_pt15m,
    S.energy.nordpool.flows.dk1_to_nl.ida1_capacity_import_pt15m,
    S.energy.nordpool.flows.dk1_to_nl.ida2_capacity_export_pt15m,
    S.energy.nordpool.flows.dk1_to_nl.ida2_capacity_import_pt15m,
]

NORDPOOL_INTRADAY_DK1 = [
    S.energy.nordpool.dk1.intraday_close_pt15m,
    S.energy.nordpool.dk1.intraday_volume_pt15m,
    S.energy.nordpool.dk1.intraday_vwap_1h_pt15m,
]

filters = [
    # ---- Target ---------------------------------------------------------
    latest(S.energy.entso_e.dk1.price_da_pt15m),

    # ---- Neighbour zone DA prices (latest; we will lag in pandas to avoid
    #      same-auction leakage) -----------------------------------------
    latest(S.energy.entso_e.dk2.price_da_pt15m),
    latest(S.energy.entso_e.se3.price_da_pt15m),
    latest(S.energy.entso_e.no2.price_da_pt15m),
    latest(S.energy.entso_e.de_lu.price_da_pt15m),

    # ---- DA forecasts: DK1 + neighbour zones (PIT-safe snapshot) -------
    *[snapshot(sig, **DA_GATE) for sig in DA_DK1_FORECASTS],
    *[snapshot(sig, **DA_GATE) for sig in DA_NEIGHBOUR_FORECASTS],

    # ---- Cross-border scheduled exchanges (PIT-safe snapshot) ----------
    *[snapshot(sig, **DA_GATE) for sig in DA_SCHED_FLOWS.values()],

    # ---- 24h-lagged actuals (forecast error / persistence proxies) -----
    lag(S.energy.entso_e.dk1.load_actual_pt60m,         delta=timedelta(hours=24)),
    lag(S.energy.entso_e.dk1.wind_onshore_actual_pt15m, delta=timedelta(hours=24)),
    lag(S.energy.entso_e.dk1.wind_offshore_actual_pt15m, delta=timedelta(hours=24)),
    lag(S.energy.entso_e.dk1.solar_actual_pt15m,        delta=timedelta(hours=24)),

    # ---- Tier-1 fundamentals: TTF gas + KRBN carbon (yfinance, daily) --
    # Daily-resolution commodity proxies broadcast over multiple delivery
    # days at write time so the PIT-safe snapshot picks the latest vintage
    # available at the gate (typically T-2 close due to 22:00-UTC publication).
    snapshot(S.commodities.gas.ttf.price_settlement_pt1d, **DA_GATE),
    snapshot(S.carbon.eua.krbn_proxy_close_pt1d,          **DA_GATE),

    # ---- Tier-1 fundamentals: Nordic hydro reservoirs (ENTSO-E, weekly) -
    # Weekly P7D observations published with a 5-day post-week offset.
    # Latest revision is sufficient — these series do not get revised.
    latest(S.energy.entso_e.no2.hydro_storage_p1d),
    latest(S.energy.entso_e.se3.hydro_storage_p1d),
    latest(S.energy.entso_e.se2.hydro_storage_p1d),

    # ---- Tier-A Nordpool fundamentals (DK1 microstructure) ----------------
    *[lag(sig, delta=timedelta(hours=24)) for sig in NORDPOOL_DA_VOLUMES_DK1],
    *[lag(sig, delta=timedelta(hours=24)) for sig in NORDPOOL_IDA_VOLUMES_DK1],
    *[snapshot(sig, **DA_GATE) for sig in NORDPOOL_IDA_CAPACITIES],
    *[lag(sig, delta=timedelta(hours=24)) for sig in NORDPOOL_INTRADAY_DK1],
]

raw_df = read_signals(filters, start=PERIOD_START, end=PERIOD_END, tz=TZ)
raw_df = raw_df.loc[
    (raw_df.index >= pd.Timestamp(PERIOD_START + " 00:00", tz=TZ))
    & (raw_df.index <  pd.Timestamp(PERIOD_END   + " 00:00", tz=TZ))
]

print(f"raw_df shape: {raw_df.shape}")
print(f"date range:   {raw_df.index.min()} -> {raw_df.index.max()}")
print()
print("NaN ratio per signal (sorted):")
print(raw_df.isna().mean().round(3).sort_values().to_string())

## 3. Resolution alignment — PT60M → PT15M

Several signals are PT60M (DK1 load + total generation forecast, DK2 load +
total generation forecast, DK1 lagged load actual, DK1→GB scheduled flow)
and arrive with NaNs in the 3 quarters after each natural hour. They show
up as a 0.75 NaN ratio in the section-2 summary above — the underlying
hourly grid is fully populated; what we see is purely a resolution mismatch
against the PT15M index.

The hourly value is constant across its 4 contained quarters by definition
of PT60M, so a forward-fill restricted to *within* each natural hour is
exact — there is no leakage and no information added.

In [ ]:
def align_pt60m_to_pt15m(
    df: pd.DataFrame,
    cols: list[str],
    *,
    tz: str = TZ,
) -> pd.DataFrame:
    """
    Forward-fill PT60M values across each natural hour's 4 quarters.

    Implementation: group by (local_date, hour) and ffill within each group.
    No cross-hour fill — preserves natural NaNs at hour boundaries.
    """
    out = df.copy()
    idx_local = out.index.tz_convert(tz)
    hour_key = pd.Series(
        idx_local.strftime("%Y-%m-%d %H"),
        index=out.index,
        name="_hour",
    )
    for c in cols:
        out[c] = out.groupby(hour_key, sort=False)[c].ffill()
    return out


def align_daily_to_pt15m(
    df: pd.DataFrame,
    cols: list[str],
    *,
    tz: str = TZ,
) -> pd.DataFrame:
    """
    Forward-fill P1D values across each local-day's 96 quarters.

    Used for daily commodity prices (TTF, KRBN) where read_signals returns
    a single value per delivery day at 00:00 UTC and we want it broadcast
    to every PT15M slot of the local-tz day. No leakage: the value for
    delivery day D is constant across day D by definition.
    """
    out = df.copy()
    idx_local = out.index.tz_convert(tz)
    day_key = pd.Series(
        idx_local.strftime("%Y-%m-%d"),
        index=out.index,
        name="_day",
    )
    for c in cols:
        out[c] = out.groupby(day_key, sort=False)[c].ffill()
    return out


def align_weekly_to_pt15m(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Forward-fill P7D weekly observations across all subsequent quarters
    until the next observation. ENTSO-E publishes hydro_storage every
    Sunday for the prior week; the value is the reservoir filling level
    as of that observation moment, which we propagate forward.
    """
    out = df.copy()
    for c in cols:
        out[c] = out[c].ffill()
    return out


PT60M_COLS = [
    "entso_e.dk1.load_forecast_pt60m",
    "entso_e.dk1.load_actual_pt60m_lag_1d",
    "entso_e.dk1.actual_aggregated_forecast_pt60m",
    "entso_e.dk2.load_forecast_pt60m",
    "entso_e.dk2.actual_aggregated_forecast_pt60m",
    "entso_e.flows.dk1_to_gb.scheduled_pt60m",
]
PT60M_COLS = [c for c in PT60M_COLS if c in raw_df.columns]

# P1D commodity columns (one row per delivery day; ffill across local-day).
P1D_COLS = [
    "gas.ttf.price_settlement_pt1d",
    "eua.krbn_proxy_close_pt1d",
]
P1D_COLS = [c for c in P1D_COLS if c in raw_df.columns]

# Weekly hydro reservoir columns (ffill until next observation, no day cap).
WEEKLY_COLS = [
    "entso_e.no2.hydro_storage_p1d",
    "entso_e.se3.hydro_storage_p1d",
    "entso_e.se2.hydro_storage_p1d",
]
WEEKLY_COLS = [c for c in WEEKLY_COLS if c in raw_df.columns]

df = align_pt60m_to_pt15m(raw_df, PT60M_COLS)
df = align_daily_to_pt15m(df, P1D_COLS)
df = align_weekly_to_pt15m(df, WEEKLY_COLS)

print("After PT60M->PT15M alignment, NaN ratios:")
print(df[PT60M_COLS].isna().mean().round(3).to_string())
print()
print("After P1D commodity alignment, NaN ratios:")
print(df[P1D_COLS].isna().mean().round(3).to_string())
print()
print("After weekly hydro alignment, NaN ratios:")
print(df[WEEKLY_COLS].isna().mean().round(3).to_string())

## 3b. Comprehensive NaN fill on raw signals

After the explicit PT60M alignment, a few small holes still exist in the raw
signal frame: the NL flow column is hourly-source for Oct 1-7 (pre-15-min-
market era), some neighbour zones are missing one day on isolated dates,
and DK1 solar/load actual have a few publication holes. We fill them
**before** building lag/rolling features so those engineered features are
NaN-free in the modelling window.

Multi-stage fill, applied in order. Each stage only fills cells that are
still NaN after previous stages. The target column (`TARGET_COL`) is never
touched.

| Stage | Operation | Fixes |
|---|---|---|
| 1 | ffill within `(local_date, hour)` | hourly-source-on-PT15M-grid (e.g. NL flow Oct 1-7) |
| 2 | ffill / bfill within `(local_date, tod_min)` | DST fall-back duplicates |
| 3 | linear interpolate within `local_date` | DST spring-forward + scattered intraday gaps |
| 4 | shift(96) (t-24h) | full-day forecast holes |
| 5 | shift(96*7) (t-7d) | two-or-more consecutive missing days |
| 6 | `ffill` | most-recent-known-value, last resort |
| 7 | `bfill` | only relevant for the very first row of the period |

**PIT note.** Every stage only uses past-or-same-row data relative to the
row being filled. Stages 4-7 explicitly use historical values (which were
knowable at the row's gate cutoff). Stage 3 (interpolation within local
day) uses other PT15M values from the same delivery day — all of which
came from snapshots gated at the same D-1 23:00 cutoff, so they too were
PIT-known at gate time. No look-ahead.

In [ ]:
def fill_signal_nans(
    df: pd.DataFrame,
    *,
    target_col: str,
    tz: str = TZ,
    return_diagnostics: bool = False,
    verbose: bool = True,
) -> pd.DataFrame | tuple[pd.DataFrame, dict]:
    """Multi-stage PIT-safe NaN fill for raw signal columns. Never modifies target_col."""
    out = df.copy()
    feat_cols = [c for c in out.columns if c != target_col]
    if not feat_cols:
        return (out, {}) if return_diagnostics else out

    idx_local = out.index.tz_convert(tz)
    local_date = pd.Series(pd.to_datetime(idx_local.date), index=out.index)
    hour       = pd.Series(idx_local.hour, index=out.index)
    tod_min    = pd.Series(idx_local.hour * 60 + idx_local.minute, index=out.index)

    initial_total = int(out[feat_cols].isna().sum().sum())
    initial_per_col = out[feat_cols].isna().sum()
    initial_per_col = initial_per_col[initial_per_col > 0]

    diag = {"initial_total": initial_total, "stages": []}

    def _record(name):
        rem = int(out[feat_cols].isna().sum().sum())
        prev = diag["stages"][-1]["remaining"] if diag["stages"] else initial_total
        diag["stages"].append({"name": name, "remaining": rem, "filled": prev - rem})
        return rem

    # Stage 1: ffill within (local_date, hour) — hourly-on-PT15M-grid
    out[feat_cols] = out.groupby([local_date, hour], sort=False)[feat_cols].ffill()
    _record("1_ffill_within_hour")

    # Stage 2: ffill/bfill within (local_date, tod_min) — DST fall-back duplicates
    out[feat_cols] = out.groupby([local_date, tod_min], sort=False)[feat_cols].ffill()
    out[feat_cols] = out.groupby([local_date, tod_min], sort=False)[feat_cols].bfill()
    _record("2_dst_fallback")

    # Stage 3: linear interpolation within local_date
    interp = (
        out[feat_cols]
        .groupby(local_date, group_keys=False, sort=False)
        .apply(lambda g: g.interpolate(method="linear", axis=0, limit_direction="both"))
    )
    if isinstance(interp.index, pd.MultiIndex):
        interp.index = interp.index.get_level_values(-1)
    interp = interp.reindex(out.index)
    out[feat_cols] = out[feat_cols].fillna(interp)
    _record("3_interp_within_day")

    # Stage 4: t-24h fallback
    out[feat_cols] = out[feat_cols].fillna(out[feat_cols].shift(96))
    _record("4_t_minus_24h")

    # Stage 5: t-7d fallback
    out[feat_cols] = out[feat_cols].fillna(out[feat_cols].shift(96 * 7))
    _record("5_t_minus_7d")

    # Stage 6: ffill — uses any past value
    out[feat_cols] = out[feat_cols].ffill()
    _record("6_ffill")

    # Stage 7: bfill — only kicks in at the very first row of the period
    out[feat_cols] = out[feat_cols].bfill()
    _record("7_bfill")

    if verbose:
        print(f"NaN fill — initial: {initial_total:,} cells across {len(initial_per_col)} cols")
        for s in diag["stages"]:
            if s["filled"] > 0:
                print(f"  {s['name']:<22s}: filled {s['filled']:>6,}, remaining {s['remaining']:,}")
        final_per_col = out[feat_cols].isna().sum()
        final_per_col = final_per_col[final_per_col > 0]
        if len(final_per_col):
            print("  WARNING: residual NaN remains:")
            print(final_per_col.to_string())
        else:
            print("  All feature columns are NaN-free.")

    if return_diagnostics:
        return out, diag
    return out


df = fill_signal_nans(df, target_col=TARGET_COL, tz=TZ, verbose=True)

# Sanity: must be 0 NaN across all non-target columns now.
n_remaining = int(df.drop(columns=[TARGET_COL]).isna().sum().sum())
print(f"\nTotal NaN cells in feature columns: {n_remaining:,}")
print(f"df shape: {df.shape}")

## 4. Time / calendar features

Imported verbatim from `little_Test_organized.ipynb`. Generates ~80 features
in 16 named groups, all DST-aware in Europe/Copenhagen. The feature set is
deterministic from the timestamp — no leakage possible.

In [ ]:
def add_time_features(
    df: pd.DataFrame,
    *,
    datetime_col: str | None = None,
    tz: str = "Europe/Copenhagen",
    prefix: str = "time_",
    holiday_countries: str | list[str] | tuple[str, ...] | None = "DK",
    observed_holidays: bool = True,
    auction_close_hour: int = 12,
    assume_tz_if_naive: str | None = None,
    drop_existing_time_features: bool = True,
    copy: bool = True,
    return_diagnostics: bool = False,
) -> pd.DataFrame | tuple[pd.DataFrame, dict]:
    """
    Calendar/time feature builder for PT15M day-ahead price models.

    Two intraday position concepts:
      1. clock_settlement_period — wall-clock 1..96 (DST-naive).
      2. sequence_in_day         — actual local-day sequence 1..92/96/100.
    """
    result = df.copy() if copy else df

    if drop_existing_time_features:
        existing = [c for c in result.columns if c.startswith(prefix)]
        if existing:
            result = result.drop(columns=existing)

    if datetime_col is None:
        if not isinstance(result.index, pd.DatetimeIndex):
            raise TypeError("df.index must be a DatetimeIndex when datetime_col=None.")
        raw_dt = result.index
    else:
        raw_dt = result[datetime_col]

    idx = pd.DatetimeIndex(pd.to_datetime(raw_dt))
    if idx.tz is None:
        if assume_tz_if_naive is None:
            raise ValueError("Datetime values are timezone-naive.")
        idx = idx.tz_localize(assume_tz_if_naive, ambiguous="infer", nonexistent="shift_forward")

    local = idx.tz_convert(tz)
    local_utc = local.tz_convert("UTC")
    feature_index = result.index
    features = pd.DataFrame(index=feature_index)

    hour    = np.asarray(local.hour, dtype=np.int16)
    minute  = np.asarray(local.minute, dtype=np.int16)
    qoh     = (minute // 15).astype(np.int8)
    mod     = (hour * 60 + minute).astype(np.int16)
    csp     = (hour * 4 + qoh + 1).astype(np.int16)
    dow     = np.asarray(local.dayofweek, dtype=np.int8)
    month   = np.asarray(local.month, dtype=np.int8)
    doy     = np.asarray(local.dayofyear, dtype=np.int16)
    leap    = np.asarray(local.is_leap_year, dtype=bool)
    diy     = np.where(leap, 366, 365).astype(np.int16)

    features[prefix + "clock_settlement_period"] = csp
    features[prefix + "hour"]            = hour
    features[prefix + "quarter_of_hour"] = qoh
    features[prefix + "minute_of_day"]   = mod
    features[prefix + "day_of_week"]     = dow
    features[prefix + "is_weekend"]      = (dow >= 5).astype(np.int8)
    features[prefix + "month"]           = month
    features[prefix + "day_of_year"]     = doy
    features[prefix + "is_month_start"]  = np.asarray(local.is_month_start, dtype=np.int8)
    features[prefix + "is_month_end"]    = np.asarray(local.is_month_end, dtype=np.int8)

    # DST-aware position
    date_keys = np.array([ts.date().isoformat() for ts in local], dtype=object)
    unique_dates = pd.unique(date_keys)
    midnight_utc = {}
    periods_in_day_lookup = {}
    for d0 in unique_dates:
        d = str(d0)
        s = pd.Timestamp(d).tz_localize(tz)
        e = (pd.Timestamp(d) + pd.Timedelta(days=1)).tz_localize(tz)
        midnight_utc[d] = s.tz_convert("UTC")
        periods_in_day_lookup[d] = int((e.tz_convert("UTC") - s.tz_convert("UTC")) / pd.Timedelta(minutes=15))

    seq = np.empty(len(local), dtype=np.int16)
    pid = np.empty(len(local), dtype=np.int16)
    for i, d0 in enumerate(date_keys):
        d = str(d0)
        seq[i] = int((local_utc[i] - midnight_utc[d]) / pd.Timedelta(minutes=15)) + 1
        pid[i] = periods_in_day_lookup[d]

    features[prefix + "sequence_in_day"]        = seq
    features[prefix + "periods_in_day"]         = pid
    features[prefix + "is_dst_transition_day"]  = (pid != 96).astype(np.int8)

    local_naive = local.tz_localize(None)
    utc_naive   = local.tz_convert("UTC").tz_localize(None)
    offset_h    = (local_naive - utc_naive).total_seconds() / 3600.0
    features[prefix + "utc_offset_hours"] = np.asarray(offset_h, dtype="float32")
    features[prefix + "is_dst"] = (np.asarray(offset_h) > 1.0).astype(np.int8)

    # Holidays
    if holiday_countries is not None:
        if isinstance(holiday_countries, str):
            holiday_countries = [holiday_countries]
        local_dates = pd.Series(local.date, index=feature_index)
        years = range(int(local.year.min()), int(local.year.max()) + 1)
        combined = pd.Series(False, index=feature_index, dtype=bool)
        for country in holiday_countries:
            cal = holidays.country_holidays(country.upper(), years=years, observed=observed_holidays)
            is_h = local_dates.isin(set(cal.keys()))
            features[f"{prefix}is_holiday_{country.lower()}"] = is_h.astype("int8")
            combined = combined | is_h
        features[prefix + "is_holiday"] = combined.astype("int8")
    else:
        features[prefix + "is_holiday"] = np.int8(0)

    features[prefix + "is_business_day"] = (
        (features[prefix + "is_weekend"] == 0) & (features[prefix + "is_holiday"] == 0)
    ).astype("int8")

    # Cyclical encodings
    csp_a = 2 * np.pi * (csp - 1) / 96.0
    features[prefix + "clock_settlement_period_sin"] = np.sin(csp_a).astype("float32")
    features[prefix + "clock_settlement_period_cos"] = np.cos(csp_a).astype("float32")
    dp_a = 2 * np.pi * (seq - 1) / pid
    features[prefix + "day_progress_sin"] = np.sin(dp_a).astype("float32")
    features[prefix + "day_progress_cos"] = np.cos(dp_a).astype("float32")
    mod_a = 2 * np.pi * mod / (24 * 60.0)
    features[prefix + "minute_of_day_sin"] = np.sin(mod_a).astype("float32")
    features[prefix + "minute_of_day_cos"] = np.cos(mod_a).astype("float32")
    dow_a = 2 * np.pi * dow / 7.0
    features[prefix + "day_of_week_sin"] = np.sin(dow_a).astype("float32")
    features[prefix + "day_of_week_cos"] = np.cos(dow_a).astype("float32")
    m_a = 2 * np.pi * (month - 1) / 12.0
    features[prefix + "month_sin"] = np.sin(m_a).astype("float32")
    features[prefix + "month_cos"] = np.cos(m_a).astype("float32")
    y_a = 2 * np.pi * (doy - 1) / diy
    features[prefix + "day_of_year_sin"] = np.sin(y_a).astype("float32")
    features[prefix + "day_of_year_cos"] = np.cos(y_a).astype("float32")

    # Auction horizon
    auction_times = []
    for ts in local:
        d = ts.date()
        ac = (pd.Timestamp(d) - pd.Timedelta(days=1) + pd.Timedelta(hours=auction_close_hour)).tz_localize(tz)
        auction_times.append(ac)
    auction_idx = pd.DatetimeIndex(auction_times)
    features[prefix + "hours_from_da_auction_close"] = (
        (local.tz_convert("UTC") - auction_idx.tz_convert("UTC")).total_seconds() / 3600.0
    ).astype("float32")

    result = result.join(features)

    if return_diagnostics:
        return result, {"n_added": len(features.columns), "added_cols": list(features.columns)}
    return result

In [ ]:
df = add_time_features(df, tz=TZ, holiday_countries="DK")
n_time_cols = sum(1 for c in df.columns if c.startswith("time_"))
print(f"Added {n_time_cols} time/calendar features.")

## 5. Price features

Two pieces:

- **Own-zone DA price features** — same-period lags (1d/2d/3d/7d), lag
  differences, neighbouring-period lags (±15/±60 min × {1,2,3,7}d),
  rolling stats (2d/3d/7d), previous-day distribution summaries (8 stats ×
  {1,2,3,7}d). Identical to the function in `little_Test_organized.ipynb`.

- **Neighbour-zone DA prices** — 24h-lagged DK2/SE3/NO2/DE-LU prices. The
  raw `latest()` columns are lagged in pandas (24h is the minimum safe lag —
  same-auction prices for D would leak).

Naive baselines (`price_baseline_naive_d1` etc.) are extracted into a
separate dataframe and dropped from the modelling matrix.

In [ ]:
def add_day_ahead_price_features(
    df: pd.DataFrame,
    *,
    price_col: str,
    tz: str = "Europe/Copenhagen",
    prefix: str = "price_",
    price_lags_days: tuple[int, ...] = (1, 2, 3, 7),
    rolling_windows_days: tuple[int, ...] = (2, 3, 7),
    daily_summary_lags_days: tuple[int, ...] = (1, 2, 3, 7),
    neighbor_offsets_minutes: tuple[int, ...] = (-60, -15, 15, 60),
    include_baselines: bool = True,
    copy: bool = True,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Same-period lags, neighbouring-period lags, rolling stats, and daily
    summaries derived from a single price column. DST-aware via local-date
    + tod-min + occurrence merge keys.
    """
    out = df.copy() if copy else df
    if not isinstance(out.index, pd.DatetimeIndex) or out.index.tz is None:
        raise ValueError("df must have a tz-aware DatetimeIndex.")
    out = out.sort_index()
    if price_col not in out.columns:
        raise KeyError(f"price_col {price_col!r} missing.")

    idx_local = out.index.tz_convert(tz)
    meta = pd.DataFrame(index=out.index)
    meta["_row_id"]     = np.arange(len(out))
    meta["_local_date"] = pd.to_datetime(idx_local.date)
    meta["_dow"]        = meta["_local_date"].dt.dayofweek
    meta["_tod_min"]    = idx_local.hour * 60 + idx_local.minute
    meta["_tod_occ"]    = meta.groupby(["_local_date", "_tod_min"]).cumcount() + 1
    meta["_price"]      = out[price_col].astype(float).to_numpy()

    work = meta[["_row_id", "_local_date", "_dow", "_tod_min", "_tod_occ"]].copy()
    period_keys = ["_local_date", "_tod_min", "_tod_occ"]

    groups: dict[str, list[str]] = {
        "baselines": [], "same_period_lags": [], "neighbor_lags": [],
        "same_period_rolling": [], "daily_summaries": [],
    }

    def merge_lag(w, lag_days, name):
        src = meta[["_local_date", "_tod_min", "_tod_occ", "_price"]].copy()
        src["_local_date"] = src["_local_date"] + pd.Timedelta(days=lag_days)
        src = src.rename(columns={"_price": name})
        return w.merge(src, on=period_keys, how="left")

    # 1. Same-period lags + naive baselines
    for L in price_lags_days:
        col = f"{prefix}lag_{L}d_same_period"
        work = merge_lag(work, L, col)
        groups["same_period_lags"].append(col)

    if include_baselines:
        d1c = f"{prefix}baseline_naive_d1"
        w1c = f"{prefix}baseline_naive_w1"
        hyc = f"{prefix}baseline_naive_epf_hybrid"
        work[d1c] = work[f"{prefix}lag_1d_same_period"]
        work[w1c] = work[f"{prefix}lag_7d_same_period"]
        tue_to_fri = work["_dow"].between(1, 4)
        work[hyc] = np.where(tue_to_fri, work[d1c], work[w1c])
        groups["baselines"].extend([d1c, w1c, hyc])

    # 2. Lag differences
    diff_specs = []
    if {f"{prefix}lag_1d_same_period", f"{prefix}lag_2d_same_period"} <= set(work.columns):
        diff_specs.append((f"{prefix}lag_1d_same_period", f"{prefix}lag_2d_same_period",
                           f"{prefix}diff_lag1d_minus_lag2d_same_period"))
    if {f"{prefix}lag_1d_same_period", f"{prefix}lag_3d_same_period"} <= set(work.columns):
        diff_specs.append((f"{prefix}lag_1d_same_period", f"{prefix}lag_3d_same_period",
                           f"{prefix}diff_lag1d_minus_lag3d_same_period"))
    if {f"{prefix}lag_1d_same_period", f"{prefix}lag_7d_same_period"} <= set(work.columns):
        diff_specs.append((f"{prefix}lag_1d_same_period", f"{prefix}lag_7d_same_period",
                           f"{prefix}diff_lag1d_minus_lag7d_same_period"))
    for a, b, name in diff_specs:
        work[name] = work[a] - work[b]
        groups["same_period_lags"].append(name)

    # 3. Neighbour-period lags
    for L in price_lags_days:
        for off in neighbor_offsets_minutes:
            if off == 0:
                continue
            sign = "plus" if off > 0 else "minus"
            name = f"{prefix}lag_{L}d_period_{sign}_{abs(off)}min"
            src = meta[["_local_date", "_tod_min", "_tod_occ", "_price"]].copy()
            src["_local_date"] = src["_local_date"] + pd.Timedelta(days=L)
            src["_tod_min"] = (src["_tod_min"] - off) % 1440
            src = src.rename(columns={"_price": name})
            work = work.merge(src, on=period_keys, how="left")
            groups["neighbor_lags"].append(name)

    # 4. Rolling stats over previous N days, same period
    roll_base = meta.copy().sort_values(["_tod_min", "_tod_occ", "_local_date"])
    for w in rolling_windows_days:
        gk = ["_tod_min", "_tod_occ"]
        mc = f"{prefix}roll_same_period_mean_{w}d"
        sc = f"{prefix}roll_same_period_std_{w}d"
        nc = f"{prefix}roll_same_period_min_{w}d"
        xc = f"{prefix}roll_same_period_max_{w}d"
        roll_base[mc] = roll_base.groupby(gk)["_price"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=w).mean())
        roll_base[sc] = roll_base.groupby(gk)["_price"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=w).std())
        roll_base[nc] = roll_base.groupby(gk)["_price"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=w).min())
        roll_base[xc] = roll_base.groupby(gk)["_price"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=w).max())
        groups["same_period_rolling"].extend([mc, sc, nc, xc])
    roll_cols = groups["same_period_rolling"]
    work = work.merge(roll_base[["_row_id"] + roll_cols], on="_row_id", how="left")

    # 5. Previous-day distribution summaries
    daily = (
        meta.groupby("_local_date")["_price"]
            .agg(mean="mean", median="median", std="std", min="min", max="max")
            .reset_index()
    )
    daily["range"] = daily["max"] - daily["min"]
    q10 = meta.groupby("_local_date")["_price"].quantile(0.10).rename("q10").reset_index()
    q90 = meta.groupby("_local_date")["_price"].quantile(0.90).rename("q90").reset_index()
    daily = daily.merge(q10, on="_local_date").merge(q90, on="_local_date")
    for L in daily_summary_lags_days:
        src = daily.copy()
        src["_local_date"] = src["_local_date"] + pd.Timedelta(days=L)
        rename_map = {c: f"{prefix}daily_lag_{L}d_{c}" for c in
                      ["mean", "median", "std", "min", "max", "range", "q10", "q90"]}
        src = src.rename(columns=rename_map)
        cols = list(rename_map.values())
        work = work.merge(src[["_local_date"] + cols], on="_local_date", how="left")
        groups["daily_summaries"].extend(cols)

    # Stitch back
    generated = []
    for k, cs in groups.items():
        generated.extend(cs)
    generated = list(dict.fromkeys(generated))
    work = work.sort_values("_row_id")
    new_df = pd.DataFrame(work[generated].to_numpy(), columns=generated, index=out.index)
    out = pd.concat([out, new_df], axis=1)
    return out, groups

In [ ]:
def add_neighbor_price_lags(
    df: pd.DataFrame,
    *,
    target_zone: str,
    neighbor_zones: tuple[str, ...],
    lags_hours: tuple[int, ...] = (24, 168),
    prefix: str = "neighbor_price_",
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Shift each neighbour zone's price column by N hours so the value at
    timestamp T is the neighbour's price from N hours ago. 24h is the minimum
    safe lag for a DA-price target — same-auction prices would leak.

    Each PT15M shift = lag_hours * 4 quarters.
    """
    out = df.copy()
    groups: dict[str, list[str]] = {}
    for L in lags_hours:
        n_qtr = L * 4
        cols = []
        for z in neighbor_zones:
            src_col = f"entso_e.{z}.price_da_pt15m"
            if src_col not in out.columns:
                continue
            new_col = f"{prefix}{z}_lag_{L}h"
            out[new_col] = out[src_col].shift(n_qtr)
            cols.append(new_col)
        if cols:
            groups[f"neighbor_prices_lag_{L}h"] = cols
    return out, groups


def add_neighbor_price_spread_features(
    df: pd.DataFrame,
    *,
    target_col: str,
    neighbor_zones: tuple[str, ...],
    lags_hours: tuple[int, ...] = (24, 168),
    rolling_days: tuple[int, ...] = (7,),
    spread_prefix: str = "price_spread_dk1_minus_",
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    For each (neighbour zone, lag), compute the lagged spread

        spread_dk1_minus_{z}_lag_{L}h(T) = own_price.shift(L*4)(T)
                                          - neighbor_price_{z}_lag_{L}h(T)

    Both sides of the subtraction use the same absolute-quarter shift so DST
    days do not introduce a bias. Spreads are the dominant cross-zone signal
    in coupled markets — DK1 typically trades at a premium / discount to a
    neighbour driven by congestion, hydro state, or wind regime — and lag-24h
    captures yesterday's regime as a forecast of today's.

    Then for each `rolling_days` window W, take a W-day mean of the lag-24h
    spread series (shifted by 1 quarter to keep the rolling at-or-before the
    feature row, no current-row leak). These rolling means are slow regime
    trackers that complement the raw lag-24h spreads.
    """
    out = df.copy()
    groups: dict[str, list[str]] = {}
    if target_col not in out.columns:
        raise KeyError(f"target_col {target_col!r} missing.")

    # Per-lag spreads.
    for L in lags_hours:
        n_qtr = L * 4
        own_lag = out[target_col].shift(n_qtr)
        spread_cols: list[str] = []
        for z in neighbor_zones:
            nbr_col = f"neighbor_price_{z}_lag_{L}h"
            if nbr_col not in out.columns:
                continue
            spread_col = f"{spread_prefix}{z}_lag_{L}h"
            out[spread_col] = own_lag - out[nbr_col]
            spread_cols.append(spread_col)
        if spread_cols:
            groups[f"price_neighbor_spreads_lag_{L}h"] = spread_cols

    # Rolling means on lag-24h spreads.
    if 24 in lags_hours:
        for W in rolling_days:
            n_roll = W * 96  # 96 quarters per day
            roll_cols: list[str] = []
            for z in neighbor_zones:
                base_col = f"{spread_prefix}{z}_lag_24h"
                if base_col not in out.columns:
                    continue
                roll_col = f"{spread_prefix}{z}_lag_24h_roll_{W}d_mean"
                out[roll_col] = (
                    out[base_col].shift(1).rolling(n_roll, min_periods=n_roll).mean()
                )
                roll_cols.append(roll_col)
            if roll_cols:
                groups[f"price_neighbor_spread_rolling_{W}d"] = roll_cols
    return out, groups


def extract_naive_baselines(
    df: pd.DataFrame,
    *,
    target_col: str,
    baseline_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Pull naive baseline columns + target into a side dataframe; drop them
    from the main modelling frame."""
    missing = [c for c in baseline_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing baseline cols: {missing}")
    df_baselines = df[[target_col, *baseline_cols]].copy()
    df_model = df.drop(columns=baseline_cols)
    return df_model, df_baselines

In [ ]:
df, price_groups = add_day_ahead_price_features(
    df, price_col=TARGET_COL, tz=TZ, prefix="price_"
)
df, neighbor_groups = add_neighbor_price_lags(
    df, target_zone=TARGET_ZONE, neighbor_zones=NEIGHBOR_ZONES,
    lags_hours=(24, 168), prefix="neighbor_price_",
)

# Cross-zone spread features (DK1 minus neighbour, at lag-24h and lag-168h, plus
# 7-day rolling on the 24h spreads). Spreads are the dominant cross-zone signal
# in coupled markets — congestion, hydro regime, wind asymmetry all show up as
# persistent spread bias that yesterday's spread is a strong forecast of.
df, spread_groups = add_neighbor_price_spread_features(
    df, target_col=TARGET_COL, neighbor_zones=NEIGHBOR_ZONES,
    lags_hours=(24, 168), rolling_days=(7,),
    spread_prefix="price_spread_dk1_minus_",
)

baseline_cols = price_groups["baselines"]
df, df_baselines = extract_naive_baselines(df, target_col=TARGET_COL, baseline_cols=baseline_cols)

# Drop the raw same-day neighbour DA prices. They were loaded via latest()
# as the source for the lag-24h / lag-168h feature columns above; the unshifted
# values are same-auction-clearing leakage and have no feature group, so we
# remove them from df entirely to make exp["feature_cols"] safe to use.
raw_neighbor_price_cols = [
    f"entso_e.{z}.price_da_pt15m" for z in NEIGHBOR_ZONES
    if f"entso_e.{z}.price_da_pt15m" in df.columns
]
if raw_neighbor_price_cols:
    df = df.drop(columns=raw_neighbor_price_cols)
    print(f"Dropped raw same-day neighbour prices (leakage hazard): {raw_neighbor_price_cols}")

print(f"price feature groups:    {[(k, len(v)) for k, v in price_groups.items() if k != 'baselines']}")
print(f"neighbour feature groups: {[(k, len(v)) for k, v in neighbor_groups.items()]}")
print(f"spread feature groups:    {[(k, len(v)) for k, v in spread_groups.items()]}")
print(f"baselines extracted:     {baseline_cols}")
print(f"df shape after price:     {df.shape}")

## 6. Fundamentals features

Three layers:

1. **DK1 snapshot covariates** — load + wind on/offshore + solar forecasts,
   plus the new total-generation forecast (`actual_aggregated_forecast`),
   plus per-border scheduled exchanges (and a derived net-import sum).
   All published before the DA gate, with `available_at` cutoff applied at
   `read_signals()` time.
2. **Neighbour-zone DA forecasts** — DK2, NO2, DE-LU each contribute load,
   wind on/offshore, solar, and total-generation forecasts. SE3 is excluded
   because its PT15M forecasts only start 2025-12-02 (Nordic 15-min market
   go-live), which would shrink the dev window. From the neighbour rosters
   we derive: per-zone residual load (load − wind − solar), per-zone
   surplus (gen − load), and a Nordic+DE-LU system-wide aggregate
   (system load, system renewables, system residual load).
3. **DK1 lagged actuals** — load/wind/solar realised at
   delivery_time − 24h. Already shifted by the `lag()` filter; used to
   build forecast-error proxies (yesterday's actual minus yesterday's
   forecast for the same hour).

Every fundamentals column is renamed to a short `fund_` prefix so the DST
repair function in section 7 can target them by prefix.

In [ ]:
FUND_RENAME = {
    # ---- DK1 DA forecasts ----
    "entso_e.dk1.load_forecast_pt60m":              "fund_load_fc",
    "entso_e.dk1.wind_onshore_forecast_pt15m":      "fund_wind_onshore_fc",
    "entso_e.dk1.wind_offshore_forecast_pt15m":     "fund_wind_offshore_fc",
    "entso_e.dk1.solar_forecast_pt15m":             "fund_solar_fc",
    "entso_e.dk1.actual_aggregated_forecast_pt60m": "fund_dk1_gen_fc",
    # ---- Cross-border scheduled exchanges ----
    "entso_e.flows.dk1_to_dk2.scheduled_pt15m":   "fund_sched_dk1_to_dk2",
    "entso_e.flows.dk1_to_se3.scheduled_pt15m":   "fund_sched_dk1_to_se3",
    "entso_e.flows.dk1_to_no2.scheduled_pt15m":   "fund_sched_dk1_to_no2",
    "entso_e.flows.dk1_to_nl.scheduled_pt15m":    "fund_sched_dk1_to_nl",
    "entso_e.flows.dk1_to_gb.scheduled_pt60m":    "fund_sched_dk1_to_gb",
    "entso_e.flows.de_lu_to_dk1.scheduled_pt15m": "fund_sched_de_lu_to_dk1",
    # ---- DK1 24h-lagged actuals ----
    "entso_e.dk1.load_actual_pt60m_lag_1d":          "fund_load_lag_24h",
    "entso_e.dk1.wind_onshore_actual_pt15m_lag_1d":  "fund_wind_onshore_lag_24h",
    "entso_e.dk1.wind_offshore_actual_pt15m_lag_1d": "fund_wind_offshore_lag_24h",
    "entso_e.dk1.solar_actual_pt15m_lag_1d":         "fund_solar_lag_24h",
    # ---- DK2 neighbour-zone DA forecasts ----
    "entso_e.dk2.load_forecast_pt60m":              "fund_dk2_load_fc",
    "entso_e.dk2.wind_onshore_forecast_pt15m":      "fund_dk2_wind_onshore_fc",
    "entso_e.dk2.wind_offshore_forecast_pt15m":     "fund_dk2_wind_offshore_fc",
    "entso_e.dk2.solar_forecast_pt15m":             "fund_dk2_solar_fc",
    "entso_e.dk2.actual_aggregated_forecast_pt60m": "fund_dk2_gen_fc",
    # ---- NO2 neighbour-zone DA forecasts ----
    "entso_e.no2.load_forecast_pt15m":              "fund_no2_load_fc",
    "entso_e.no2.wind_onshore_forecast_pt15m":      "fund_no2_wind_onshore_fc",
    "entso_e.no2.wind_offshore_forecast_pt15m":     "fund_no2_wind_offshore_fc",
    "entso_e.no2.solar_forecast_pt15m":             "fund_no2_solar_fc",
    "entso_e.no2.actual_aggregated_forecast_pt15m": "fund_no2_gen_fc",
    # ---- DE-LU neighbour-zone DA forecasts ----
    "entso_e.de_lu.load_forecast_pt15m":              "fund_de_lu_load_fc",
    "entso_e.de_lu.wind_onshore_forecast_pt15m":      "fund_de_lu_wind_onshore_fc",
    "entso_e.de_lu.wind_offshore_forecast_pt15m":     "fund_de_lu_wind_offshore_fc",
    "entso_e.de_lu.solar_forecast_pt15m":             "fund_de_lu_solar_fc",
    "entso_e.de_lu.actual_aggregated_forecast_pt15m": "fund_de_lu_gen_fc",
    # ---- Tier-1 commodities ----
    "gas.ttf.price_settlement_pt1d":                 "fund_ttf_da",
    "eua.krbn_proxy_close_pt1d":                     "fund_eua_carbon_proxy",
    # ---- Tier-1 Nordic hydro reservoirs (weekly) ----
    "entso_e.no2.hydro_storage_p1d":                 "fund_no2_hydro_reservoir",
    "entso_e.se3.hydro_storage_p1d":                 "fund_se3_hydro_reservoir",
    "entso_e.se2.hydro_storage_p1d":                 "fund_se2_hydro_reservoir",
}

NEIGHBOUR_FUND_ZONES = ("dk2", "no2", "de_lu")


def add_fundamentals_features(
    df: pd.DataFrame,
    *,
    rename_map: dict[str, str] = FUND_RENAME,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Rename signal columns to short fund_* names and derive:
      - residual_load_fc = load_fc - wind_onshore_fc - wind_offshore_fc - solar_fc
      - net_sched_imports = de_lu_to_dk1 - sum(dk1_to_X) (DK1 net import from neighbours)
      - 24h forecast-error proxies (lagged actual - lagged forecast at same valid_at)
    """
    out = df.copy()
    actually_renamed = {k: v for k, v in rename_map.items() if k in out.columns}
    out = out.rename(columns=actually_renamed)

    groups: dict[str, list[str]] = {}

    # ---- raw forecast levels --------------------------------------------
    fc_cols = [c for c in ["fund_load_fc", "fund_wind_onshore_fc",
                           "fund_wind_offshore_fc", "fund_solar_fc"] if c in out.columns]
    if "fund_load_fc" in out.columns:
        groups["fund_load_fc"] = ["fund_load_fc"]
    ws_cols = [c for c in ["fund_wind_onshore_fc", "fund_wind_offshore_fc",
                           "fund_solar_fc"] if c in out.columns]
    if ws_cols:
        groups["fund_wind_solar_fc"] = ws_cols

    # ---- residual load forecast -----------------------------------------
    if all(c in out.columns for c in ["fund_load_fc", "fund_wind_onshore_fc",
                                       "fund_wind_offshore_fc", "fund_solar_fc"]):
        out["fund_residual_load_fc"] = (
            out["fund_load_fc"]
            - out["fund_wind_onshore_fc"].fillna(0)
            - out["fund_wind_offshore_fc"].fillna(0)
            - out["fund_solar_fc"].fillna(0)
        )
        groups["fund_residual_load_fc"] = ["fund_residual_load_fc"]

    # ---- scheduled exchanges (per border) -------------------------------
    sched_outbound = [c for c in [
        "fund_sched_dk1_to_dk2", "fund_sched_dk1_to_se3", "fund_sched_dk1_to_no2",
        "fund_sched_dk1_to_nl", "fund_sched_dk1_to_gb",
    ] if c in out.columns]
    sched_inbound = [c for c in ["fund_sched_de_lu_to_dk1"] if c in out.columns]
    sched_all = sched_outbound + sched_inbound
    if sched_all:
        groups["fund_scheduled_exchanges"] = sched_all

    # ---- net imports (signed: positive = net import to DK1) ------------
    if sched_outbound or sched_inbound:
        out["fund_net_sched_imports"] = (
            sum((out[c].fillna(0) for c in sched_inbound), pd.Series(0, index=out.index))
            - sum((out[c].fillna(0) for c in sched_outbound), pd.Series(0, index=out.index))
        )
        groups["fund_net_imports"] = ["fund_net_sched_imports"]

    # ---- 24h-lagged actuals ---------------------------------------------
    if "fund_load_lag_24h" in out.columns:
        groups["fund_load_lag_24h"] = ["fund_load_lag_24h"]
    ws_lag = [c for c in ["fund_wind_onshore_lag_24h", "fund_wind_offshore_lag_24h",
                          "fund_solar_lag_24h"] if c in out.columns]
    if ws_lag:
        groups["fund_wind_solar_lag_24h"] = ws_lag

    # ---- forecast-error proxies (lag_24h actual - lag_24h forecast) ----
    # The lag_24h forecast is yesterday's forecast for yesterday's same hour,
    # which we approximate as the current forecast shifted by 96 quarters.
    n_qtr_24h = 96
    err_cols = []
    for fc, act, name in [
        ("fund_load_fc",          "fund_load_lag_24h",          "fund_load_fc_err_lag_24h"),
        ("fund_wind_onshore_fc",  "fund_wind_onshore_lag_24h",  "fund_wind_onshore_fc_err_lag_24h"),
        ("fund_wind_offshore_fc", "fund_wind_offshore_lag_24h", "fund_wind_offshore_fc_err_lag_24h"),
        ("fund_solar_fc",         "fund_solar_lag_24h",         "fund_solar_fc_err_lag_24h"),
    ]:
        if fc in out.columns and act in out.columns:
            out[name] = out[act] - out[fc].shift(n_qtr_24h)
            err_cols.append(name)
    if err_cols:
        groups["fund_forecast_error_24h"] = err_cols

    # ---- residual load forecast rolling stats ---------------------------
    if "fund_residual_load_fc" in out.columns:
        # 96 quarters = 24h
        rl = out["fund_residual_load_fc"]
        out["fund_residual_load_fc_mean_24h"] = rl.shift(1).rolling(96, min_periods=96).mean()
        out["fund_residual_load_fc_std_24h"]  = rl.shift(1).rolling(96, min_periods=96).std()
        out["fund_residual_load_fc_max_24h"]  = rl.shift(1).rolling(96, min_periods=96).max()
        groups["fund_residual_load_rolling"] = [
            "fund_residual_load_fc_mean_24h",
            "fund_residual_load_fc_std_24h",
            "fund_residual_load_fc_max_24h",
        ]

    # ---- DK1 total generation forecast + surplus -----------------------
    # gen_fc - load_fc = expected DK1 net surplus before flows. A high value
    # implies DK1 will be exporting and prices should track neighbours' lower
    # marginal cost; a low value implies DK1 is importing and prices firm up.
    dk1_gen_cols = []
    if "fund_dk1_gen_fc" in out.columns:
        dk1_gen_cols.append("fund_dk1_gen_fc")
        if "fund_load_fc" in out.columns:
            out["fund_dk1_gen_minus_load_fc"] = out["fund_dk1_gen_fc"] - out["fund_load_fc"]
            dk1_gen_cols.append("fund_dk1_gen_minus_load_fc")
    if dk1_gen_cols:
        groups["fund_dk1_gen_fc"] = dk1_gen_cols

    # ---- Per-neighbour fundamentals -----------------------------------
    # For each neighbouring zone (dk2, no2, de_lu), build:
    #   * residual_load_fc = load - wind - solar (drives that zone's price)
    #   * gen_minus_load_fc (if both are present)
    # Group the raw forecasts by family across neighbours so Optuna can
    # turn whole families on/off together (load, wind/solar, total-gen).
    neighbour_load_cols = []
    neighbour_wind_solar_cols = []
    neighbour_gen_cols = []
    neighbour_residual_cols = []
    for z in ("dk2", "no2", "de_lu"):
        load_c   = f"fund_{z}_load_fc"
        won_c    = f"fund_{z}_wind_onshore_fc"
        woff_c   = f"fund_{z}_wind_offshore_fc"
        sol_c    = f"fund_{z}_solar_fc"
        gen_c    = f"fund_{z}_gen_fc"
        if load_c in out.columns:
            neighbour_load_cols.append(load_c)
        for c in (won_c, woff_c, sol_c):
            if c in out.columns:
                neighbour_wind_solar_cols.append(c)
        if gen_c in out.columns:
            neighbour_gen_cols.append(gen_c)
            if load_c in out.columns:
                surplus_c = f"fund_{z}_gen_minus_load_fc"
                out[surplus_c] = out[gen_c] - out[load_c]
                neighbour_gen_cols.append(surplus_c)
        # Residual load forecast for this zone
        if load_c in out.columns and (won_c in out.columns or woff_c in out.columns
                                       or sol_c in out.columns):
            res_c = f"fund_{z}_residual_load_fc"
            out[res_c] = (
                out[load_c]
                - (out[won_c].fillna(0) if won_c in out.columns else 0)
                - (out[woff_c].fillna(0) if woff_c in out.columns else 0)
                - (out[sol_c].fillna(0) if sol_c in out.columns else 0)
            )
            neighbour_residual_cols.append(res_c)

    if neighbour_load_cols:
        groups["fund_neighbours_load_fc"] = neighbour_load_cols
    if neighbour_wind_solar_cols:
        groups["fund_neighbours_wind_solar_fc"] = neighbour_wind_solar_cols
    if neighbour_gen_cols:
        groups["fund_neighbours_gen_fc"] = neighbour_gen_cols
    if neighbour_residual_cols:
        groups["fund_neighbours_residual_load_fc"] = neighbour_residual_cols

    # ---- System-wide aggregates (DK1 + DK2 + NO2 + DE-LU) -------------
    # Sum fundamentals across DK1 + neighbours we collect to get a "Nordic+
    # central-Europe" view of system tightness. DK1's price is increasingly
    # coupled to this aggregate via the cross-border flows.
    sys_load_parts = [c for c in ("fund_load_fc", "fund_dk2_load_fc",
                                   "fund_no2_load_fc", "fund_de_lu_load_fc")
                      if c in out.columns]
    sys_renew_parts = []
    for c in ("fund_wind_onshore_fc", "fund_wind_offshore_fc", "fund_solar_fc",
              "fund_dk2_wind_onshore_fc", "fund_dk2_wind_offshore_fc", "fund_dk2_solar_fc",
              "fund_no2_wind_onshore_fc", "fund_no2_wind_offshore_fc", "fund_no2_solar_fc",
              "fund_de_lu_wind_onshore_fc", "fund_de_lu_wind_offshore_fc", "fund_de_lu_solar_fc"):
        if c in out.columns:
            sys_renew_parts.append(c)
    sys_cols = []
    if sys_load_parts:
        out["fund_system_load_fc"] = sum(out[c].fillna(0) for c in sys_load_parts)
        sys_cols.append("fund_system_load_fc")
    if sys_renew_parts:
        out["fund_system_renewable_fc"] = sum(out[c].fillna(0) for c in sys_renew_parts)
        sys_cols.append("fund_system_renewable_fc")
    if sys_load_parts and sys_renew_parts:
        out["fund_system_residual_load_fc"] = (
            out["fund_system_load_fc"] - out["fund_system_renewable_fc"]
        )
        sys_cols.append("fund_system_residual_load_fc")
    if sys_cols:
        groups["fund_system_aggregates"] = sys_cols

    # ---- System stress / share ratios (DK1) -----------------------------
    # Marginal-clearing intuition: the sign and magnitude of the price
    # response to a forecast surprise scales with how "tight" the system is.
    # A unit of extra wind is worth more (lowers price more) when residual
    # load is already a big share of total load. These ratios let a tree
    # split on regime in a way that the raw GW levels can't.
    stress_cols: list[str] = []
    if all(c in out.columns for c in ["fund_load_fc", "fund_residual_load_fc"]):
        out["fund_residual_load_ratio_fc"] = (
            out["fund_residual_load_fc"] / out["fund_load_fc"].replace(0, np.nan)
        )
        stress_cols.append("fund_residual_load_ratio_fc")
    wind_solar_components = [c for c in (
        "fund_wind_onshore_fc", "fund_wind_offshore_fc", "fund_solar_fc",
    ) if c in out.columns]
    if "fund_load_fc" in out.columns and wind_solar_components:
        out["fund_renewable_share_fc"] = (
            sum(out[c].fillna(0) for c in wind_solar_components)
            / out["fund_load_fc"].replace(0, np.nan)
        )
        stress_cols.append("fund_renewable_share_fc")
    wind_components = [c for c in (
        "fund_wind_onshore_fc", "fund_wind_offshore_fc",
    ) if c in out.columns]
    if "fund_load_fc" in out.columns and wind_components:
        out["fund_wind_share_fc"] = (
            sum(out[c].fillna(0) for c in wind_components)
            / out["fund_load_fc"].replace(0, np.nan)
        )
        stress_cols.append("fund_wind_share_fc")
    if stress_cols:
        groups["fund_system_stress"] = stress_cols

    # ---- Tier-1: TTF gas, EUA carbon, Nordic hydro reservoirs ----------
    # Marginal-cost (gas + carbon) and regime (hydro) drivers that no
    # ENTSO-E electricity-system signal captures. Hydro values are MWh
    # equivalents; aggregate Nordic reservoir state is the relevant
    # regime variable for NO2-coupled DK1 days.
    if "fund_ttf_da" in out.columns:
        groups["fund_ttf_da"] = ["fund_ttf_da"]
    if "fund_eua_carbon_proxy" in out.columns:
        groups["fund_eua_carbon"] = ["fund_eua_carbon_proxy"]

    hydro_cols = [c for c in ("fund_no2_hydro_reservoir",
                              "fund_se3_hydro_reservoir",
                              "fund_se2_hydro_reservoir") if c in out.columns]
    if hydro_cols:
        groups["fund_nordic_hydro_levels"] = hydro_cols
        # Aggregate Nordic reservoir total — the regime variable.
        out["fund_nordic_hydro_total"] = sum(out[c].fillna(0) for c in hydro_cols)
        groups["fund_nordic_hydro_aggregates"] = ["fund_nordic_hydro_total"]

        # Hydro regime tracker: fractional deviation from the trailing 30d
        # mean = (current - 30d_mean) / 30d_mean. Window = 30d (2880 quarters);
        # min_periods = 7d (672 quarters) so the feature is non-NaN as soon as
        # WARMUP_DAYS=7 of trailing data exists. We avoid z-scoring (current
        # - mu) / sigma because hydro publishes weekly: a 7-day trailing
        # window can have a *constant* sigma=0 if no new weekly value landed
        # in the window, which would NaN the z-score. Fractional deviation
        # is well-defined as long as mu>0 (always true for reservoir levels).
        # With only ~7 months of data we cannot do a proper same-week-of-year
        # seasonal anomaly (would need multi-year history), so this is a
        # regime-change tracker — reservoirs falling rapidly vs. recent trend
        # are predictive of higher prices in NO2-coupled days.
        n_roll = 30 * 96
        n_min = 7 * 96
        regime_cols: list[str] = []
        for c in hydro_cols + ["fund_nordic_hydro_total"]:
            base = out[c].shift(1)
            mu = base.rolling(n_roll, min_periods=n_min).mean()
            r_col = f"{c}_regime_dev_30d"
            out[r_col] = (out[c] - mu) / mu.replace(0, np.nan)
            regime_cols.append(r_col)
        if regime_cols:
            groups["fund_hydro_regime_dev"] = regime_cols

    return out, groups

In [ ]:
df, fund_groups = add_fundamentals_features(df)
print(f"fundamentals feature groups: {[(k, len(v)) for k, v in fund_groups.items()]}")
n_fund_cols = sum(1 for c in df.columns if c.startswith("fund_"))
print(f"total fund_* columns: {n_fund_cols}")
print(f"df shape after fundamentals: {df.shape}")

## 6b. Nordpool fundamentals features (Tier-A microstructure)

DK1 day-ahead **microstructure** signals from Nord Pool, complementary to the
ENTSO-E TSO-published fundamentals above:

- **DA volumes (lag 24h)** — yesterday's cleared buy / sell / net volume.
  Volume scarcity tracks regime shifts in supply / demand depth.
- **IDA1 / IDA2 volumes (lag 24h)** — yesterday's intraday-auction cleared
  volume. Captures intraday rebalancing pressure post-DA.
- **IDA1 / IDA2 capacities (snapshot)** — pre-auction interconnector capacity
  for DK1 ↔ DK2, GER (DE-LU), NL. Tightening capacity per border tightens
  the local price; net (import − export) is a price-direction proxy.
- **Intraday close / volume / VWAP-1h (lag 24h)** — yesterday's intraday
  trading discovery. Spread vs DA price is a known leading indicator of
  next-day DA movements (information value for the DA model).

All columns are renamed to `fund_np_*` so they are picked up automatically by
the existing DST repair and warm-up trim (which target the `fund_` prefix).

In [ ]:
def add_nordpool_fundamentals_features(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """Rename Nordpool covariates to short fund_np_* and derive aggregates.

    Returns: (df, group_dict). Idempotent — only renames columns that exist.
    """
    out = df.copy()

    NP_RENAME = {
        # ---- DA volumes (lag 24h, PT15M) ----
        "nordpool.dk1.da_buy_volume_pt15m_lag_1d":  "fund_np_da_buy_vol_lag_24h",
        "nordpool.dk1.da_sell_volume_pt15m_lag_1d": "fund_np_da_sell_vol_lag_24h",
        # ---- IDA volumes (lag 24h, PT15M) ----
        "nordpool.dk1.ida1_buy_volume_pt15m_lag_1d":  "fund_np_ida1_buy_vol_lag_24h",
        "nordpool.dk1.ida1_sell_volume_pt15m_lag_1d": "fund_np_ida1_sell_vol_lag_24h",
        "nordpool.dk1.ida2_buy_volume_pt15m_lag_1d":  "fund_np_ida2_buy_vol_lag_24h",
        "nordpool.dk1.ida2_sell_volume_pt15m_lag_1d": "fund_np_ida2_sell_vol_lag_24h",
        # ---- IDA capacities (snapshot pre-gate, PT15M) ----
        "nordpool.flows.dk1_to_dk2.ida1_capacity_export_pt15m": "fund_np_ida1_cap_exp_dk2",
        "nordpool.flows.dk1_to_dk2.ida1_capacity_import_pt15m": "fund_np_ida1_cap_imp_dk2",
        "nordpool.flows.dk1_to_dk2.ida2_capacity_export_pt15m": "fund_np_ida2_cap_exp_dk2",
        "nordpool.flows.dk1_to_dk2.ida2_capacity_import_pt15m": "fund_np_ida2_cap_imp_dk2",
        "nordpool.flows.dk1_to_ger.ida1_capacity_export_pt15m": "fund_np_ida1_cap_exp_ger",
        "nordpool.flows.dk1_to_ger.ida1_capacity_import_pt15m": "fund_np_ida1_cap_imp_ger",
        "nordpool.flows.dk1_to_ger.ida2_capacity_export_pt15m": "fund_np_ida2_cap_exp_ger",
        "nordpool.flows.dk1_to_ger.ida2_capacity_import_pt15m": "fund_np_ida2_cap_imp_ger",
        "nordpool.flows.dk1_to_nl.ida1_capacity_export_pt15m":  "fund_np_ida1_cap_exp_nl",
        "nordpool.flows.dk1_to_nl.ida1_capacity_import_pt15m":  "fund_np_ida1_cap_imp_nl",
        "nordpool.flows.dk1_to_nl.ida2_capacity_export_pt15m":  "fund_np_ida2_cap_exp_nl",
        "nordpool.flows.dk1_to_nl.ida2_capacity_import_pt15m":  "fund_np_ida2_cap_imp_nl",
        # ---- Intraday OHLCV / VWAP (lag 24h, PT15M) ----
        "nordpool.dk1.intraday_close_pt15m_lag_1d":   "fund_np_intraday_close_lag_24h",
        "nordpool.dk1.intraday_volume_pt15m_lag_1d":  "fund_np_intraday_vol_lag_24h",
        "nordpool.dk1.intraday_vwap_1h_pt15m_lag_1d": "fund_np_intraday_vwap_1h_lag_24h",
    }
    actually_renamed = {k: v for k, v in NP_RENAME.items() if k in out.columns}
    out = out.rename(columns=actually_renamed)

    groups: dict[str, list[str]] = {}

    # ---- DA volumes ----
    da_vol_cols = [c for c in (
        "fund_np_da_buy_vol_lag_24h",
        "fund_np_da_sell_vol_lag_24h",
    ) if c in out.columns]
    if da_vol_cols:
        # Sell − buy is a net-export proxy: positive = DK1 was an exporter
        # at the cleared price. Useful direction signal.
        if {"fund_np_da_sell_vol_lag_24h", "fund_np_da_buy_vol_lag_24h"}.issubset(out.columns):
            out["fund_np_da_net_export_lag_24h"] = (
                out["fund_np_da_sell_vol_lag_24h"]
                - out["fund_np_da_buy_vol_lag_24h"]
            )
            da_vol_cols.append("fund_np_da_net_export_lag_24h")
        groups["fund_np_da_volumes"] = da_vol_cols

    # ---- IDA volumes ----
    ida_vol_cols = [c for c in (
        "fund_np_ida1_buy_vol_lag_24h", "fund_np_ida1_sell_vol_lag_24h",
        "fund_np_ida2_buy_vol_lag_24h", "fund_np_ida2_sell_vol_lag_24h",
    ) if c in out.columns]
    if ida_vol_cols:
        groups["fund_np_ida_volumes"] = ida_vol_cols

    # ---- Per-border IDA capacity + per-border net ----
    for border in ("dk2", "ger", "nl"):
        border_cols: list[str] = []
        for prefix in ("ida1_cap_exp", "ida1_cap_imp", "ida2_cap_exp", "ida2_cap_imp"):
            c = f"fund_np_{prefix}_{border}"
            if c in out.columns:
                border_cols.append(c)
        if not border_cols:
            continue
        # Net IDA capacity for this border (import − export across IDA1+IDA2).
        zero = pd.Series(0.0, index=out.index)
        net = (
            out.get(f"fund_np_ida1_cap_imp_{border}", zero).fillna(0)
            - out.get(f"fund_np_ida1_cap_exp_{border}", zero).fillna(0)
            + out.get(f"fund_np_ida2_cap_imp_{border}", zero).fillna(0)
            - out.get(f"fund_np_ida2_cap_exp_{border}", zero).fillna(0)
        )
        net_col = f"fund_np_ida_cap_net_{border}"
        out[net_col] = net
        border_cols.append(net_col)
        groups[f"fund_np_ida_capacity_{border}"] = border_cols

    # ---- Total IDA capacity across all DK1 borders ----
    imp_cols = [c for c in out.columns
                if c.startswith("fund_np_ida") and "_cap_imp_" in c]
    exp_cols = [c for c in out.columns
                if c.startswith("fund_np_ida") and "_cap_exp_" in c]
    if imp_cols and exp_cols:
        out["fund_np_ida_total_cap_import"] = sum(out[c].fillna(0) for c in imp_cols)
        out["fund_np_ida_total_cap_export"] = sum(out[c].fillna(0) for c in exp_cols)
        out["fund_np_ida_total_cap_net"] = (
            out["fund_np_ida_total_cap_import"]
            - out["fund_np_ida_total_cap_export"]
        )
        groups["fund_np_ida_capacity_aggregates"] = [
            "fund_np_ida_total_cap_import",
            "fund_np_ida_total_cap_export",
            "fund_np_ida_total_cap_net",
        ]

    # ---- Intraday OHLCV / VWAP ----
    intraday_cols = [c for c in (
        "fund_np_intraday_close_lag_24h",
        "fund_np_intraday_vol_lag_24h",
        "fund_np_intraday_vwap_1h_lag_24h",
    ) if c in out.columns]
    if intraday_cols:
        groups["fund_np_intraday"] = intraday_cols

    return out, groups


df, np_fund_groups = add_nordpool_fundamentals_features(df)
fund_groups.update(np_fund_groups)

print(f"Nordpool fundamentals feature groups: "
      f"{[(k, len(v)) for k, v in np_fund_groups.items()]}")
n_np_cols = sum(1 for c in df.columns if c.startswith("fund_np_"))
print(f"total fund_np_* columns: {n_np_cols}")
print(f"df shape after Nordpool fundamentals: {df.shape}")

## 6c. Weather features (FS4) — ECMWF IFS 12z forecasts

DK1-relevant weather features from the **ECMWF IFS 12z run** (issued at
D-1 12:00 UTC, available ~T+5h ≈ D-1 17:00 UTC, well before the 23:00
Brussels gate). Pre-extracted by `_build/extract_weather_dk1_ifs.py` from
the Hetzner Bronze Zarr store and cached as a single parquet artefact —
keeps the notebook focused on modelling instead of S3/xarray plumbing.

Per delivery day D, the extractor:

1. Opens the D-1 12z Zarr from `bronze-v2/ecmwf_ifs/grid/surface/...`.
2. Slices to a wider DK1 + offshore bbox (lat 54.0-58.5, lon 6.5-13.0).
3. Slices to lead steps T+9..T+36 (covers any local delivery day across
   DST: 23h spring-forward, 24h normal, 25h fall-back).
4. Differences accumulated `shortwave_radiation` (J/m² → 3h-mean W/m²).
5. Computes 19 features in 9 groups (see registry in §9): bbox-mean / std
   / p90 / max wind at 100m and 10m, an onshore/offshore split (lon split
   at 8.5°E since the IFS Zarr lacks an LSM mask), vector-mean wind
   direction (sin/cos), wind shear ratio, temperature stats, dewpoint and
   apparent-humidity proxy, mean SSRD, MSL mean and spatial std.
6. Forward-fills the PT3H native cadence to PT15M, restricted to the
   Europe/Copenhagen-local UTC range of delivery day D — gives 96 quarters
   on a normal day, 100 on fall-back, 92 on spring-forward.

The parquet stores one PT15M-aligned row per local delivery quarter, so
the join below is a single `df.join(...)` with no overlap and no resample.

> **Dropped from v1:** `total_precipitation` is corrupt at the Bronze
> layer (cumulative values go backward, single cells reach 1.6 m at T+51).
> Open as a Bronze ingestion bug to investigate later. Precip is a weak
> DA-price feature anyway — the mechanism is via wind/load.

In [ ]:
WEATHER_PARQUET = PACKAGE_ROOT / "_build/cache/weather_dk1_ifs_12z.parquet"

WEATHER_FEATURE_COLS = [
    # wind 100m bbox spatial
    "weather_wind100m_mean", "weather_wind100m_std",
    "weather_wind100m_p90", "weather_wind100m_max",
    # wind 100m onshore/offshore split (lon split at 8.5°E)
    "weather_wind100m_onshore_mean", "weather_wind100m_offshore_mean",
    # wind 100m direction (vector mean -> sin/cos of bearing)
    "weather_wind100m_dir_sin", "weather_wind100m_dir_cos",
    # wind 10m
    "weather_wind10m_mean", "weather_wind10m_std",
    # wind shear (100m / 10m ratio)
    "weather_wind_shear_ratio",
    # temperature
    "weather_temp_mean", "weather_temp_min", "weather_temp_max",
    "weather_temp_dewpoint_diff",
    # solar (3h-mean W/m^2, differenced from cumulative J/m^2)
    "weather_ssrd_mean",
    # pressure
    "weather_msl_mean", "weather_msl_std",
    # humidity
    "weather_dewpoint_mean",
]

weather_pt15m = pd.read_parquet(WEATHER_PARQUET)
print(f"Weather parquet loaded: {weather_pt15m.shape}, "
      f"{weather_pt15m['delivery_date'].nunique()} delivery days, "
      f"{WEATHER_PARQUET.stat().st_size/1024/1024:.2f} MB")

# PIT validation: every available_at must be < gate (D-1 23:00 Brussels).
# Gate UTC offset is +1h winter (CET) or +2h summer (CEST) — check tz-aware.
_brussels_gate = (
    pd.to_datetime(weather_pt15m["delivery_date"])
      .dt.tz_localize("Europe/Brussels")
      - pd.Timedelta(days=1)
      + pd.Timedelta(hours=23)
).dt.tz_convert("UTC")
_violations = (weather_pt15m["available_at"] > _brussels_gate).sum()
assert _violations == 0, f"PIT violation: {_violations} weather rows have available_at > gate"
_min_headroom = (_brussels_gate - weather_pt15m["available_at"]).min()
print(f"PIT check: 0 violations, min headroom = {_min_headroom}")

# Sanity-check coverage against df.index range before joining
_df_dates = pd.Index(df.index.tz_convert("Europe/Copenhagen").date).unique()
_w_dates = pd.Index(weather_pt15m["delivery_date"]).unique()
_missing_in_weather = set(_df_dates) - set(_w_dates)
if _missing_in_weather:
    print(f"WARNING: {len(_missing_in_weather)} delivery dates in df not covered by weather parquet")
    print(f"  examples: {sorted(_missing_in_weather)[:5]}")

# Join only the feature cols (drop metadata cols from the merged df).
n_before = df.shape[1]
df = df.join(weather_pt15m[WEATHER_FEATURE_COLS], how="left")
n_added = df.shape[1] - n_before
n_new_nan = int(df[WEATHER_FEATURE_COLS].isna().sum().sum())
print(f"Weather joined: +{n_added} cols, NaN cells in weather cols: {n_new_nan}")
print(f"df shape after weather join: {df.shape}")

## 6d. Weather forecast-error features (FS4 enrichment)

ERA5 reanalysis *actuals* lagged 24h, paired with the same-quarter IFS
forecast lagged 24h, give a clean per-feature **forecast error** family:

```
weather_<feat>_err_lag_24h(T) = era5_<feat>(T - 24h)            (actual yesterday)
                              - weather_<feat>(T).shift(96)     (forecast made yesterday-of-yesterday for yesterday)
```

Mirror of the `fund_*_fc_err_lag_24h` family: yesterday's forecast bias is
the strongest single predictor of today's forecast bias for the same feature
(persistence in upstream NWP errors is well-documented). Five errors —
temperature, all-DK1 wind100m, on/offshore wind100m split, SSRD — chosen
because each maps to a distinct DA-price driver (load, wind generation,
solar generation).

**PIT note:** ERA5 final reanalysis has ~5-day publication latency, so it is
*not* literally knowable at the D-1 23:00 gate. We use it as a proxy for
ECMWF's operational analysis (HRES initial-condition analysis, ~6h latency)
and for SYNOP/METAR observations available in real time. This is a
documented thesis caveat, not a leakage-free claim.

The ERA5 parquet is pre-extracted by `_build/extract_era5_dk1.py`, sharing
the bbox + onshore/offshore split + aggregation logic with the IFS extractor
so the subtraction is dimensionally clean.

In [ ]:
ERA5_PARQUET = PACKAGE_ROOT / "_build/cache/weather_dk1_era5_actual.parquet"

# Subset of weather features for which we compute forecast errors. Chosen
# for distinct DA-price drivers; each error has clear physical meaning.
WEATHER_ERR_FEATS = [
    "weather_temp_mean",                 # load-demand error (heating / cooling)
    "weather_wind100m_mean",             # bulk wind generation error
    "weather_wind100m_onshore_mean",     # onshore wind generation error
    "weather_wind100m_offshore_mean",    # offshore wind generation error
    "weather_ssrd_mean",                 # solar generation error
]

era5_hourly = pd.read_parquet(ERA5_PARQUET)
print(f"ERA5 parquet loaded: {era5_hourly.shape}, "
      f"{era5_hourly['utc_date'].nunique()} UTC dates, "
      f"{ERA5_PARQUET.stat().st_size/1024/1024:.2f} MB")

# Sanity checks before resampling.
assert isinstance(era5_hourly.index, pd.DatetimeIndex), "ERA5 index must be DatetimeIndex"
assert era5_hourly.index.tz is not None, "ERA5 index must be tz-aware"
assert era5_hourly.index.is_monotonic_increasing, "ERA5 index must be monotonically increasing"
era5_dt = era5_hourly.index.to_series().diff().dropna().value_counts()
assert era5_dt.index[0] == pd.Timedelta(hours=1) and len(era5_dt) == 1, (
    f"ERA5 must be strictly hourly: {era5_dt}"
)

# 1. Resample hourly UTC -> PT15M UTC by forward-fill (the value at hour H is
#    held across H:00, H:15, H:30, H:45). Build a complete PT15M index over
#    the full ERA5 range so the reindex below is a no-op for non-DST days.
era5_idx_15 = pd.date_range(
    era5_hourly.index.min(),
    era5_hourly.index.max() + pd.Timedelta(minutes=45),
    freq="15min", tz="UTC",
)
era5_pt15m_full = (
    era5_hourly[WEATHER_ERR_FEATS]
    .reindex(era5_idx_15)
    .ffill(limit=3)  # ffill at most 3 quarters (one hour) — defensive
)

# 2. Shift 96 quarters on the FULL ERA5 series first (which starts 2 days
#    before PERIOD_START — see extract_era5_dk1.py), THEN reindex to df.index.
#    This way, every row of df.index gets a valid lag-24h actual from the
#    pre-period buffer; reversing the order would NaN-out the first 96 df rows.
era5_lag24h_full = era5_pt15m_full.shift(96)
era5_lag24h = era5_lag24h_full.reindex(df.index)
era5_lag24h.columns = [f"era5_{c}_lag_24h" for c in WEATHER_ERR_FEATS]

# 3. Compute weather_<feat>_err_lag_24h = actual_lag_24h - fc.shift(96)
err_cols: list[str] = []
for feat in WEATHER_ERR_FEATS:
    actual_col = f"era5_{feat}_lag_24h"
    err_col = f"{feat}_err_lag_24h"
    if actual_col not in era5_lag24h.columns:
        continue
    if feat not in df.columns:
        continue
    df[err_col] = era5_lag24h[actual_col] - df[feat].shift(96)
    err_cols.append(err_col)

print(f"\nWeather forecast-error features added: {len(err_cols)}")
for c in err_cols:
    s = df[c]
    print(f"  {c:48s}  rows={len(s)}  NaN={int(s.isna().sum())}  "
          f"mean={s.mean():+8.3f}  std={s.std():7.3f}  "
          f"min={s.min():+8.3f}  max={s.max():+8.3f}")
print(f"\ndf shape after weather forecast-error join: {df.shape}")

## 7. DST repair & warm-up trim

1. Trim the first `WARMUP_DAYS` (=7) so every row has its longest lag
   populated — this is the deepest lookback across all three feature
   families.
2. Apply DST repair to columns prefixed `price_`, `fund_`, and
   `neighbor_price_` — the spring/fall-back transitions create NaNs in
   lagged + rolling features in every family that uses lags.
3. Apply the same DST repair to the naive-baseline side table
   (`price_baseline_*`). These are pure persistence lags and the events
   already happened in reality, so filling DST gaps within the local day
   is not look-ahead — it just keeps the baseline defined on every row of
   the modelling window.

In [ ]:
def fill_dst_nans(
    df: pd.DataFrame,
    *,
    target_col: str,
    prefixes: tuple[str, ...] = ("price_",),
    tz: str = "Europe/Copenhagen",
    add_imputation_flag: bool = True,
    flag_col: str = "dst_feature_imputed",
    recompute_diff_features: bool = True,
    interpolation_method: str = "linear",
    raise_on_remaining_nans: bool = False,
    copy: bool = True,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict]:
    """
    Generalised DST repair that handles multiple feature prefixes in one pass.

    Repair logic (applied per column):
      1. Fall-back DST (autumn): copy values within (local_date, tod_min)
         group via ffill/bfill.
      2. Spring-forward DST: interpolate remaining NaNs within the local day.
      3. Diff features named "<prefix>diff_lagAd_minus_lagBd_same_period"
         are recomputed from repaired lag columns.
    """
    if not isinstance(df.index, pd.DatetimeIndex) or df.index.tz is None:
        raise ValueError("df.index must be tz-aware DatetimeIndex.")
    out = df.copy() if copy else df
    out = out.sort_index()

    repair_cols = [
        c for c in out.columns
        if isinstance(c, str)
        and any(c.startswith(p) for p in prefixes)
        and c != target_col
        and c != flag_col
    ]
    if not repair_cols:
        raise ValueError(f"No columns found for prefixes={prefixes}")

    target_before = out[target_col].copy()
    for c in repair_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").astype("float64")

    pre = out[repair_cols].copy()
    missing_before = pre.isna()
    rows_with_missing = missing_before.any(axis=1)
    n_missing_before = int(missing_before.sum().sum())

    if add_imputation_flag:
        out[flag_col] = rows_with_missing.astype("int8")

    if n_missing_before == 0:
        diag = {"n_missing_before": 0, "n_missing_after": 0,
                "rows_flagged": 0, "flag_col": flag_col}
        if verbose:
            print("DST repair: nothing to fix.")
        return out, diag

    idx_local = out.index.tz_convert(tz)
    local_date = pd.Series(pd.to_datetime(idx_local.date), index=out.index)
    tod_min    = pd.Series(idx_local.hour * 60 + idx_local.minute, index=out.index)

    # Identify diff columns we will recompute from repaired lags
    diff_specs = []
    if recompute_diff_features:
        for prefix in prefixes:
            pat = re.compile(rf"^{re.escape(prefix)}diff_lag(\d+)d_minus_lag(\d+)d_same_period$")
            for c in repair_cols:
                m = pat.match(c)
                if m:
                    a = f"{prefix}lag_{m.group(1)}d_same_period"
                    b = f"{prefix}lag_{m.group(2)}d_same_period"
                    if a in repair_cols and b in repair_cols:
                        diff_specs.append((c, a, b))
    diff_set = {x[0] for x in diff_specs}

    base_cols = [c for c in repair_cols if c not in diff_set and missing_before[c].any()]

    if base_cols:
        before = out[base_cols].copy()
        miss = before.isna()
        same_wall = (
            before.groupby([local_date, tod_min], sort=False).ffill()
                  .groupby([local_date, tod_min], sort=False).bfill()
        )
        repaired = before.mask(miss, same_wall)
        # Spring-forward: interpolate within local day
        rem_cols = repaired.columns[repaired.isna().any(axis=0)].tolist()
        if rem_cols:
            interp = (
                repaired[rem_cols]
                .groupby(local_date, group_keys=False, sort=False)
                .apply(lambda g: g.interpolate(method=interpolation_method, axis=0,
                                               limit_direction="both"))
            )
            if isinstance(interp.index, pd.MultiIndex):
                interp.index = interp.index.get_level_values(-1)
            interp = interp.reindex(repaired.index)
            rem_miss = repaired[rem_cols].isna()
            repaired.loc[:, rem_cols] = repaired[rem_cols].mask(rem_miss, interp)
        out.loc[:, base_cols] = out.loc[:, base_cols].mask(miss, repaired)

    # Fill diff columns from repaired lags
    for diff_col, a, b in diff_specs:
        if diff_col in out.columns and out[diff_col].isna().any():
            computed = out[a] - out[b]
            mask = out[diff_col].isna() & computed.notna()
            if mask.any():
                out.loc[mask, diff_col] = computed.loc[mask]

    pd.testing.assert_series_equal(out[target_col], target_before, check_names=True,
                                   obj="target column")

    after = out[repair_cols]
    n_missing_after = int(after.isna().sum().sum())
    rows_after = int(after.isna().any(axis=1).sum())

    if verbose:
        print("DST feature repair")
        print("------------------")
        print(f"  cols repaired:         {len(repair_cols)}")
        print(f"  missing cells before:  {n_missing_before:,}")
        print(f"  missing cells after:   {n_missing_after:,}")
        print(f"  rows flagged:          {int(rows_with_missing.sum()):,}")

    if raise_on_remaining_nans and n_missing_after > 0:
        raise ValueError(f"{n_missing_after} NaNs remain after DST repair.")

    diag = {
        "n_missing_before": n_missing_before,
        "n_missing_after":  n_missing_after,
        "rows_flagged":     int(rows_with_missing.sum()),
        "flag_col":         flag_col if add_imputation_flag else None,
    }
    return out, diag

In [ ]:
warmup_cutoff = pd.Timestamp(PERIOD_START, tz=TZ) + pd.Timedelta(days=WARMUP_DAYS)
df = df.loc[df.index >= warmup_cutoff].copy()
print(f"After warm-up trim ({WARMUP_DAYS}d): df.shape = {df.shape}, start = {df.index.min()}")

df, dst_diag = fill_dst_nans(
    df,
    target_col=TARGET_COL,
    prefixes=("price_", "fund_", "neighbor_price_"),
    tz=TZ,
    raise_on_remaining_nans=False,   # some neighbour zones may have legitimate gaps
)

# Apply the same DST repair to the side-table of naive baselines.
# These columns are pure lag/hybrid persistence (price_baseline_naive_d1 = lag-1d,
# price_baseline_naive_w1 = lag-7d, price_baseline_naive_epf_hybrid = mix). The
# spring-forward day produces NaNs (e.g. 2026-03-30 02:00 in naive_d1 because
# 2026-03-29 02:00 does not exist in local time), and the fall-back day produces
# duplicates that ffill/bfill can resolve. The event has already happened in
# reality, so filling within the local day is fair.
df_baselines = df_baselines.loc[df.index].copy()
df_baselines, dst_baselines_diag = fill_dst_nans(
    df_baselines,
    target_col=TARGET_COL,
    prefixes=("price_baseline_",),
    tz=TZ,
    add_imputation_flag=False,
    raise_on_remaining_nans=False,
    verbose=True,
)

# Drop rows where any feature is still NaN (legitimate data gaps after DST repair)
n_before = len(df)
df = df.dropna()
print(f"After dropna: removed {n_before - len(df):,} rows. df.shape = {df.shape}")

# Sanity: baselines must have no remaining NaNs in the windows we use.
remaining_baseline_nans = df_baselines.loc[df.index].isna().sum()
remaining_baseline_nans = remaining_baseline_nans[remaining_baseline_nans > 0]
if len(remaining_baseline_nans) > 0:
    print("WARNING: baseline NaNs remain after DST repair:")
    print(remaining_baseline_nans)
else:
    print("Baselines have zero NaNs across the modelling window.")

### 7b. PIT-safety check

For every fundamentals snapshot signal, re-derive the cutoff in SQL and
verify two things:

1. **Picked-vintage safety:** the row chosen by the snapshot SQL satisfies
   `available_at <= cutoff` (tautology by construction of the snapshot SQL,
   but worth confirming).
2. **Cutoff headroom:** distribution of `cutoff - available_at` per signal,
   so we can see how tight the cutoff is to actual publication. Negative
   values would be PIT violations and trigger an assertion.

This does *not* test the `lag()` filter — that uses `latest()` semantics
and is documented as an open caveat in section 2.

In [ ]:
from signalforge.data_access._query import _fetch
from signalforge.data_access.connection import _run

# fund_* column -> canonical signal_name in signal_values
SNAPSHOT_CANONICALS = {
    # ---- DK1 forecasts ----
    "fund_load_fc":            "ENTSO-E:energy:bidding_zone=DK1-dataset=load_forecast:load_forecast_day_ahead:MWh:PT60M",
    "fund_wind_onshore_fc":    "ENTSO-E:energy:bidding_zone=DK1-dataset=wind_and_solar_forecast-production_type=Wind Onshore:wind_onshore_forecast:MWh:PT15M",
    "fund_wind_offshore_fc":   "ENTSO-E:energy:bidding_zone=DK1-dataset=wind_and_solar_forecast-production_type=Wind Offshore:wind_offshore_forecast:MWh:PT15M",
    "fund_solar_fc":           "ENTSO-E:energy:bidding_zone=DK1-dataset=wind_and_solar_forecast-production_type=Solar:solar_forecast:MWh:PT15M",
    "fund_dk1_gen_fc":         "ENTSO-E:energy:bidding_zone=DK1-dataset=generation_forecast-production_type=Actual Aggregated:actual_aggregated_forecast:MWh:PT60M",
    # ---- Cross-border scheduled exchanges ----
    "fund_sched_dk1_to_dk2":   "ENTSO-E:energy:dataset=scheduled_exchanges-from_zone=DK1-to_zone=DK2:scheduled_exchange_day_ahead:MWh:PT15M",
    "fund_sched_dk1_to_se3":   "ENTSO-E:energy:dataset=scheduled_exchanges-from_zone=DK1-to_zone=SE3:scheduled_exchange_day_ahead:MWh:PT15M",
    "fund_sched_dk1_to_no2":   "ENTSO-E:energy:dataset=scheduled_exchanges-from_zone=DK1-to_zone=NO2:scheduled_exchange_day_ahead:MWh:PT15M",
    "fund_sched_dk1_to_nl":    "ENTSO-E:energy:dataset=scheduled_exchanges-from_zone=DK1-to_zone=NL:scheduled_exchange_day_ahead:MWh:PT15M",
    "fund_sched_dk1_to_gb":    "ENTSO-E:energy:dataset=scheduled_exchanges-from_zone=DK1-to_zone=GB:scheduled_exchange_day_ahead:MWh:PT60M",
    "fund_sched_de_lu_to_dk1": "ENTSO-E:energy:dataset=scheduled_exchanges-from_zone=DE-LU-to_zone=DK1:scheduled_exchange_day_ahead:MWh:PT15M",
    # ---- DK2 neighbour forecasts ----
    "fund_dk2_load_fc":          "ENTSO-E:energy:bidding_zone=DK2-dataset=load_forecast:load_forecast_day_ahead:MWh:PT60M",
    "fund_dk2_wind_onshore_fc":  "ENTSO-E:energy:bidding_zone=DK2-dataset=wind_and_solar_forecast-production_type=Wind Onshore:wind_onshore_forecast:MWh:PT15M",
    "fund_dk2_wind_offshore_fc": "ENTSO-E:energy:bidding_zone=DK2-dataset=wind_and_solar_forecast-production_type=Wind Offshore:wind_offshore_forecast:MWh:PT15M",
    "fund_dk2_solar_fc":         "ENTSO-E:energy:bidding_zone=DK2-dataset=wind_and_solar_forecast-production_type=Solar:solar_forecast:MWh:PT15M",
    "fund_dk2_gen_fc":           "ENTSO-E:energy:bidding_zone=DK2-dataset=generation_forecast-production_type=Actual Aggregated:actual_aggregated_forecast:MWh:PT60M",
    # ---- NO2 neighbour forecasts ----
    "fund_no2_load_fc":          "ENTSO-E:energy:bidding_zone=NO2-dataset=load_forecast:load_forecast_day_ahead:MWh:PT15M",
    "fund_no2_wind_onshore_fc":  "ENTSO-E:energy:bidding_zone=NO2-dataset=wind_and_solar_forecast-production_type=Wind Onshore:wind_onshore_forecast:MWh:PT15M",
    "fund_no2_wind_offshore_fc": "ENTSO-E:energy:bidding_zone=NO2-dataset=wind_and_solar_forecast-production_type=Wind Offshore:wind_offshore_forecast:MWh:PT15M",
    "fund_no2_solar_fc":         "ENTSO-E:energy:bidding_zone=NO2-dataset=wind_and_solar_forecast-production_type=Solar:solar_forecast:MWh:PT15M",
    "fund_no2_gen_fc":           "ENTSO-E:energy:bidding_zone=NO2-dataset=generation_forecast-production_type=Actual Aggregated:actual_aggregated_forecast:MWh:PT15M",
    # ---- DE-LU neighbour forecasts ----
    "fund_de_lu_load_fc":          "ENTSO-E:energy:bidding_zone=DE-LU-dataset=load_forecast:load_forecast_day_ahead:MWh:PT15M",
    "fund_de_lu_wind_onshore_fc":  "ENTSO-E:energy:bidding_zone=DE-LU-dataset=wind_and_solar_forecast-production_type=Wind Onshore:wind_onshore_forecast:MWh:PT15M",
    "fund_de_lu_wind_offshore_fc": "ENTSO-E:energy:bidding_zone=DE-LU-dataset=wind_and_solar_forecast-production_type=Wind Offshore:wind_offshore_forecast:MWh:PT15M",
    "fund_de_lu_solar_fc":         "ENTSO-E:energy:bidding_zone=DE-LU-dataset=wind_and_solar_forecast-production_type=Solar:solar_forecast:MWh:PT15M",
    "fund_de_lu_gen_fc":           "ENTSO-E:energy:bidding_zone=DE-LU-dataset=generation_forecast-production_type=Actual Aggregated:actual_aggregated_forecast:MWh:PT15M",
    # ---- Tier-1 commodity proxies (yfinance, daily) ----
    "fund_ttf_da":                 "ICE:commodities:commodity=gas-contract=front_month_future-hub=TTF:price_settlement:EUR/MWh:P1D",
    "fund_eua_carbon_proxy":       "KraneShares:carbon:contract=KRBN_global_carbon_etf-tenor=front_month_proxy-ticker=KRBN:price_settlement:USD:P1D",
}

PIT_CHECK_SQL = """
    WITH knowable AS (
        SELECT signal_name, valid_from, available_at, revision_id, entity_hash,
            (date_trunc('day', valid_from AT TIME ZONE $4)::timestamp
             - ($5 * INTERVAL '1 day')
             + $6::time) AT TIME ZONE $4 AS cutoff
        FROM signalforge.signal_values
        WHERE signal_name = $1
          AND valid_from >= $2
          AND valid_from <  $3
    ),
    passing AS (
        SELECT *, row_number() OVER (
            PARTITION BY signal_name, entity_hash, valid_from
            ORDER BY available_at DESC, revision_id DESC
        ) AS _rn
        FROM knowable
        WHERE available_at <= cutoff
    ),
    picked AS (
        SELECT * FROM passing WHERE _rn = 1
    )
    SELECT
        COUNT(*) AS n_picked,
        EXTRACT(EPOCH FROM MAX(available_at - cutoff)) AS max_violation_sec,
        EXTRACT(EPOCH FROM MIN(cutoff - available_at)) AS min_headroom_sec,
        EXTRACT(EPOCH FROM AVG(cutoff - available_at)) AS avg_headroom_sec,
        EXTRACT(EPOCH FROM MAX(cutoff - available_at)) AS max_headroom_sec
    FROM picked
"""


def assert_pit_safety(snapshot_canonicals, *, days_ahead, cutoff_time, tz, period_start, period_end):
    """Re-run snapshot cutoff math in SQL and verify max(available_at - cutoff) <= 0."""
    print(f"PIT-safety check: cutoff = D-{days_ahead} {cutoff_time:%H:%M} {tz}")
    print(f"Period: {period_start} -> {period_end}\n")
    print(f"{'fund column':<30s}  {'n_picked':>8s}  {'max_viol':>10s}  {'min_headroom':>14s}  {'avg_headroom':>14s}  {'max_headroom':>14s}")
    n_violations = 0
    rows_checked = 0
    for col, canonical in snapshot_canonicals.items():
        if col not in df.columns:
            continue
        rows = _run(_fetch(PIT_CHECK_SQL, canonical, period_start, period_end,
                           tz, days_ahead, cutoff_time))
        if not rows or rows[0]["n_picked"] == 0:
            print(f"{col:<30s}  no rows in DB for the period")
            continue
        r = rows[0]
        rows_checked += r["n_picked"]
        max_v = float(r["max_violation_sec"] or 0.0)
        min_h = float(r["min_headroom_sec"] or 0.0) / 3600.0
        avg_h = float(r["avg_headroom_sec"] or 0.0) / 3600.0
        max_h = float(r["max_headroom_sec"] or 0.0) / 3600.0
        print(f"{col:<30s}  {r['n_picked']:>8d}  {max_v:>9.0f}s  {min_h:>12.2f}h  {avg_h:>12.2f}h  {max_h:>12.2f}h")
        if max_v > 0:
            n_violations += 1
    print(f"\nrows checked across all signals: {rows_checked:,}")
    if n_violations > 0:
        raise AssertionError(f"PIT VIOLATION: {n_violations} signals had available_at > cutoff.")
    print("PIT safety: PASS  every picked vintage has available_at <= cutoff.")


_period_start_utc = pd.Timestamp(PERIOD_START + " 00:00", tz=TZ).tz_convert("UTC").to_pydatetime()
_period_end_utc   = pd.Timestamp(PERIOD_END   + " 00:00", tz=TZ).tz_convert("UTC").to_pydatetime()

assert_pit_safety(
    SNAPSHOT_CANONICALS,
    days_ahead=DA_GATE["days_ahead"],
    cutoff_time=DA_GATE["time_of_day"],
    tz=DA_GATE["tz"],
    period_start=_period_start_utc,
    period_end=_period_end_utc,
)

## 8. Folds & final test split

Walk-forward validation by complete local delivery days (DST-aware: 92/96/100
periods). Last 28 days = held-out test set; preceding days form the dev
set; 5 expanding folds × 7 val days carve out validation windows from the
end of dev.

`exp` dict produced here matches the contract used by the experiment engine:
`exp["folds"]` is a list of fold bundles with `X_train` / `y_train` / `X_val`
/ `y_val`, and `exp["final"]` holds the dev/test bundle.

In [ ]:
def validate_time_series_df(df, target_col, feature_cols=None, *, require_timezone=True):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("DatetimeIndex required.")
    if require_timezone and df.index.tz is None:
        raise ValueError("tz-aware index required.")
    if df.index.duplicated().any():
        raise ValueError("Duplicate timestamps in index.")
    df = df.sort_index().copy()
    if feature_cols is None:
        feature_cols = [c for c in df.columns if c != target_col]
    return df, feature_cols


def get_local_dates(df, *, tz=TZ):
    idx_local = df.index.tz_convert(tz)
    return pd.Series(idx_local.date, index=df.index, name="local_date")


def make_dev_test_split_by_days(df, target_col, feature_cols, *, test_days, tz=TZ):
    local_dates = get_local_dates(df, tz=tz)
    unique_dates = pd.Index(local_dates.drop_duplicates().to_list())
    if test_days >= len(unique_dates):
        raise ValueError(f"test_days={test_days} too large.")
    test_set = set(unique_dates[-test_days:])
    is_test = local_dates.isin(test_set)
    df_dev = df.loc[~is_test].copy()
    df_test = df.loc[is_test].copy()
    return {
        "df_dev": df_dev, "df_test": df_test,
        "X_dev": df_dev[feature_cols].copy(), "y_dev": df_dev[target_col].copy(),
        "X_test": df_test[feature_cols].copy(), "y_test": df_test[target_col].copy(),
        "feature_cols": feature_cols, "target_col": target_col,
        "test_dates": sorted(test_set),
    }


def materialize_expanding_day_folds(df_dev, target_col, feature_cols, *,
                                     n_splits, val_days, gap_days, tz=TZ):
    local_dates = get_local_dates(df_dev, tz=tz)
    unique_dates = pd.Index(local_dates.drop_duplicates().to_list())
    total_val = n_splits * val_days
    if total_val + gap_days >= len(unique_dates):
        raise ValueError("Not enough days for the requested folds.")
    first_val_pos = len(unique_dates) - total_val
    folds = []
    rows = []
    for f in range(1, n_splits + 1):
        v_start = first_val_pos + (f - 1) * val_days
        v_end   = v_start + val_days
        t_end   = v_start - gap_days
        if t_end <= 0:
            raise ValueError("Empty training window.")
        train_dates = set(unique_dates[:t_end])
        val_dates   = set(unique_dates[v_start:v_end])
        train_df = df_dev.loc[local_dates.isin(train_dates)].copy()
        val_df   = df_dev.loc[local_dates.isin(val_dates)].copy()
        folds.append({
            "fold": f,
            "train_df": train_df, "val_df": val_df,
            "X_train": train_df[feature_cols].copy(), "y_train": train_df[target_col].copy(),
            "X_val":   val_df[feature_cols].copy(),   "y_val":   val_df[target_col].copy(),
            "train_start": train_df.index.min(), "train_end": train_df.index.max(),
            "val_start":   val_df.index.min(),   "val_end":   val_df.index.max(),
            "n_train": len(train_df), "n_val": len(val_df),
        })
        rows.append({"fold": f, "n_train": len(train_df), "n_val": len(val_df),
                     "train_start": train_df.index.min(), "train_end": train_df.index.max(),
                     "val_start": val_df.index.min(), "val_end": val_df.index.max()})
    return folds, pd.DataFrame(rows)


def setup_time_series_experiment(df, target_col, feature_cols=None, *,
                                  test_days=TEST_DAYS, n_splits=N_SPLITS,
                                  val_days=VAL_DAYS, gap_days=GAP_DAYS, tz=TZ):
    df, feature_cols = validate_time_series_df(df, target_col, feature_cols)
    if df.empty:
        raise ValueError("Empty dataframe after validation.")
    split_bundle = make_dev_test_split_by_days(df, target_col, feature_cols,
                                                test_days=test_days, tz=tz)
    folds, fold_summary = materialize_expanding_day_folds(
        split_bundle["df_dev"], target_col, feature_cols,
        n_splits=n_splits, val_days=val_days, gap_days=gap_days, tz=tz,
    )
    final_bundle = {
        "train_df": split_bundle["df_dev"].copy(),
        "test_df":  split_bundle["df_test"].copy(),
        "X_train":  split_bundle["X_dev"].copy(),  "y_train": split_bundle["y_dev"].copy(),
        "X_test":   split_bundle["X_test"].copy(), "y_test":  split_bundle["y_test"].copy(),
    }
    return {
        "df": df, **split_bundle,
        "folds": folds, "fold_summary": fold_summary,
        "final": final_bundle, "tz": tz,
    }

In [ ]:
exp = setup_time_series_experiment(
    df, target_col=TARGET_COL,
    test_days=TEST_DAYS, n_splits=N_SPLITS,
    val_days=VAL_DAYS, gap_days=GAP_DAYS, tz=TZ,
)
print(exp["fold_summary"])
print(f"\nfeature_cols: {len(exp['feature_cols'])} columns")
print(f"dev:  {exp['df_dev'].index.min()} -> {exp['df_dev'].index.max()} ({len(exp['df_dev']):,} rows)")
print(f"test: {exp['df_test'].index.min()} -> {exp['df_test'].index.max()} ({len(exp['df_test']):,} rows)")

### 8b. Fold-integrity check

Before training anything, assert there is no leakage path between train,
val, and test slices: `train.max < val.min` per fold, `dev.max < test.min`,
all val windows sit inside the dev set, and no row appears in both train
and val of the same fold.

In [ ]:
def assert_fold_integrity(exp: dict) -> None:
    """Verify there's no train/val overlap per fold and no dev/test overlap."""
    issues = []
    # Per-fold: train must end strictly before val starts
    for fold in exp["folds"]:
        if fold["X_train"].index.max() >= fold["X_val"].index.min():
            issues.append(
                f"Fold {fold['fold']}: train ends {fold['X_train'].index.max()}, "
                f"val starts {fold['X_val'].index.min()}"
            )
        # Train/val should be non-empty
        if len(fold["X_train"]) == 0:
            issues.append(f"Fold {fold['fold']}: empty X_train")
        if len(fold["X_val"]) == 0:
            issues.append(f"Fold {fold['fold']}: empty X_val")
    # Final dev/test
    final = exp["final"]
    if final["X_train"].index.max() >= final["X_test"].index.min():
        issues.append(
            f"Dev/test overlap: dev ends {final['X_train'].index.max()}, "
            f"test starts {final['X_test'].index.min()}"
        )
    # Validation folds should sit inside the dev set, not extend into test
    test_start = final["X_test"].index.min()
    for fold in exp["folds"]:
        if fold["X_val"].index.max() >= test_start:
            issues.append(
                f"Fold {fold['fold']}: val extends to {fold['X_val'].index.max()} "
                f">= test start {test_start}"
            )
    # No row should appear in both train and val
    for fold in exp["folds"]:
        overlap = fold["X_train"].index.intersection(fold["X_val"].index)
        if len(overlap) > 0:
            issues.append(f"Fold {fold['fold']}: {len(overlap)} index overlap rows")

    if issues:
        for i in issues:
            print("FAIL:", i)
        raise AssertionError(f"Fold integrity check failed with {len(issues)} issue(s).")

    print("Fold integrity: PASS")
    print(f"  test window:  {final['X_test'].index.min()} -> {final['X_test'].index.max()}  ({len(final['X_test']):,} rows)")
    print(f"  dev window:   {final['X_train'].index.min()} -> {final['X_train'].index.max()}  ({len(final['X_train']):,} rows)")
    for fold in exp["folds"]:
        print(f"  fold {fold['fold']}: train -> {fold['X_train'].index.max()}  |  val: {fold['X_val'].index.min()} -> {fold['X_val'].index.max()}")


assert_fold_integrity(exp)

## 9. Experiment engine — model registry by family

### Feature groups

Time, price, neighbour-price, and fundamentals each contribute one or more
named groups; FS variants pick which families participate.

### Optuna objective

The grouped objective lets each trial choose (a) which feature *groups* to
include and (b) hyper-parameters. The score is mean validation MAE plus
optional group/feature penalties.

### Model registry

`ModelSpec(name, family, suggest_params_fn, build_model_fn, ...)` —
groupable by `family` so the runner can iterate
`tree → {xgb, lgbm}`, `linear → {ridge, lasso, enet}`, and trivially extend.

In [ ]:
TIME_FEATURE_GROUPS = {
    "intraday_position_raw": [
        "time_clock_settlement_period", "time_hour",
        "time_quarter_of_hour", "time_minute_of_day",
    ],
    "intraday_position_dst_aware": ["time_sequence_in_day", "time_periods_in_day"],
    "weekly_calendar_raw":          ["time_day_of_week"],
    "annual_calendar_raw":          ["time_month", "time_day_of_year"],
    "month_boundary_indicators":    ["time_is_month_start", "time_is_month_end"],
    "day_type_indicators":          ["time_is_weekend", "time_is_business_day"],
    "holiday_indicators":           ["time_is_holiday_dk", "time_is_holiday"],
    "dst_indicators":               ["time_is_dst_transition_day", "time_utc_offset_hours", "time_is_dst"],
    "intraday_settlement_period_cyclical": ["time_clock_settlement_period_sin", "time_clock_settlement_period_cos"],
    "intraday_day_progress_cyclical":      ["time_day_progress_sin", "time_day_progress_cos"],
    "minute_of_day_cyclical":              ["time_minute_of_day_sin", "time_minute_of_day_cos"],
    "weekly_calendar_cyclical":            ["time_day_of_week_sin", "time_day_of_week_cos"],
    "monthly_calendar_cyclical":           ["time_month_sin", "time_month_cos"],
    "annual_calendar_cyclical":            ["time_day_of_year_sin", "time_day_of_year_cos"],
    "auction_horizon":                     ["time_hours_from_da_auction_close"],
}


def _build_price_feature_groups(
    price_groups: dict[str, list[str]],
    neighbor_groups: dict[str, list[str]],
    spread_groups: dict[str, list[str]] | None = None,
) -> dict[str, list[str]]:
    """Convert add_day_ahead_price_features groups + neighbour lag groups +
    cross-zone spread groups into the same flat dict shape used by the Optuna
    engine."""
    out = {}
    # Same-period lags (incl. diffs)
    if price_groups.get("same_period_lags"):
        out["price_same_period_lags"] = price_groups["same_period_lags"]
    # Split neighbour-period lags by lag-days
    nbr = price_groups.get("neighbor_lags", [])
    for L in (1, 2, 3, 7):
        cols = [c for c in nbr if f"_lag_{L}d_period_" in c]
        if cols:
            out[f"price_neighbor_lags_{L}d"] = cols
    # Rolling stats
    rolling = price_groups.get("same_period_rolling", [])
    for w in (2, 3, 7):
        cols = [c for c in rolling if c.endswith(f"_{w}d")]
        if cols:
            out[f"price_rolling_same_period_{w}d"] = cols
    # Daily summaries by lag-days
    daily = price_groups.get("daily_summaries", [])
    for L in (1, 2, 3, 7):
        cols = [c for c in daily if f"_lag_{L}d_" in c]
        if cols:
            out[f"price_daily_summary_lag_{L}d"] = cols
    # Neighbour-zone price lags
    out.update(neighbor_groups)
    # Cross-zone spread features
    if spread_groups:
        out.update(spread_groups)
    return out


PRICE_FEATURE_GROUPS = _build_price_feature_groups(price_groups, neighbor_groups, spread_groups)
FUNDAMENTAL_FEATURE_GROUPS = fund_groups

# Weather feature groups (FS4) — defined here as the model-side counterpart
# to WEATHER_FEATURE_COLS in §6c. Nine semantically-distinct groups so the
# Optuna group-selector can keep, drop, or weight each independently.
WEATHER_FEATURE_GROUPS: dict[str, list[str]] = {
    "weather_wind100m_bulk": [
        "weather_wind100m_mean", "weather_wind100m_std",
        "weather_wind100m_p90", "weather_wind100m_max",
    ],
    "weather_wind100m_zones": [
        "weather_wind100m_onshore_mean", "weather_wind100m_offshore_mean",
    ],
    "weather_wind100m_direction": [
        "weather_wind100m_dir_sin", "weather_wind100m_dir_cos",
    ],
    "weather_wind10m": [
        "weather_wind10m_mean", "weather_wind10m_std",
    ],
    "weather_wind_shear": [
        "weather_wind_shear_ratio",
    ],
    "weather_temperature": [
        "weather_temp_mean", "weather_temp_min", "weather_temp_max",
    ],
    "weather_humidity": [
        "weather_dewpoint_mean", "weather_temp_dewpoint_diff",
    ],
    "weather_solar": [
        "weather_ssrd_mean",
    ],
    "weather_pressure": [
        "weather_msl_mean", "weather_msl_std",
    ],
    # Forecast errors from §6d: yesterday's IFS forecast vs ERA5 reanalysis
    # actuals at the same hour. Persistent NWP bias is the most consistent
    # single signal here — captures things like "model under-predicted
    # offshore wind for the past 36h" that the raw forecast level cannot.
    "weather_forecast_error_24h": [
        "weather_temp_mean_err_lag_24h",
        "weather_wind100m_mean_err_lag_24h",
        "weather_wind100m_onshore_mean_err_lag_24h",
        "weather_wind100m_offshore_mean_err_lag_24h",
        "weather_ssrd_mean_err_lag_24h",
    ],
}

# Add the DST imputation flag as its own group
if "dst_feature_imputed" in df.columns:
    PRICE_FEATURE_GROUPS["price_dst_imputation_indicator"] = ["dst_feature_imputed"]

print("price feature groups:")
for k, v in PRICE_FEATURE_GROUPS.items():
    print(f"  {k}: {len(v)} cols")
print("\nfundamentals feature groups:")
for k, v in FUNDAMENTAL_FEATURE_GROUPS.items():
    print(f"  {k}: {len(v)} cols")
print("\nweather feature groups:")
for k, v in WEATHER_FEATURE_GROUPS.items():
    print(f"  {k}: {len(v)} cols")

In [ ]:
FEATURE_GROUPS_BY_FAMILY = {
    "time":         TIME_FEATURE_GROUPS,
    "price":        PRICE_FEATURE_GROUPS,
    "fundamentals": FUNDAMENTAL_FEATURE_GROUPS,
    "weather":      WEATHER_FEATURE_GROUPS,
}

FEATURE_SET_VARIANTS = {
    "fs1": ["time"],
    "fs2": ["time", "price"],
    "fs3": ["time", "price", "fundamentals"],
    "fs4": ["time", "price", "fundamentals", "weather"],
}

ACTIVE_FS = ["fs1", "fs2", "fs3", "fs4"]

# ----------------------------------------------------------------------------
# Defensive checks: target must NOT appear in any feature group, and every
# non-target column in df must be accounted for by some group. This guards
# against accidental leakage and silently-dropped features.
# ----------------------------------------------------------------------------
_all_grouped_cols: set[str] = set()
for _fam_name, _fam_groups in FEATURE_GROUPS_BY_FAMILY.items():
    for _g_name, _g_cols in _fam_groups.items():
        if TARGET_COL in _g_cols:
            raise AssertionError(
                f"LEAKAGE: target column {TARGET_COL!r} found in "
                f"FEATURE_GROUPS_BY_FAMILY[{_fam_name!r}][{_g_name!r}]"
            )
        _all_grouped_cols.update(_g_cols)

_orphans = set(df.columns) - _all_grouped_cols - {TARGET_COL}
if _orphans:
    raise AssertionError(
        f"Unaccounted columns in df (not in any feature group, not the target): "
        f"{sorted(_orphans)}. Either add them to a group or drop them from df."
    )

print(f"Leakage / orphan check: PASS")
print(f"  target excluded:     {TARGET_COL}")
print(f"  grouped feature cols: {len(_all_grouped_cols)}")
print(f"  orphans:             0")

In [ ]:
# Private aliases — `time` in this notebook is `datetime.time` (from cell 2's
# `from datetime import time`), so we use the real time module under a private
# name to avoid clobbering it.
from time import monotonic as _now


def resolve_feature_groups_for_variant(
    variant_name, feature_set_variants, feature_groups_by_family, *, prefix_family=True,
):
    """Flatten a variant's family list into one prefixed group dict."""
    if variant_name not in feature_set_variants:
        raise KeyError(f"Unknown variant {variant_name!r}.")
    families = feature_set_variants[variant_name]
    unknown = [f for f in families if f not in feature_groups_by_family]
    if unknown:
        raise KeyError(f"Unknown families: {unknown}")
    out = {}
    for fam in families:
        for gname, cols in feature_groups_by_family[fam].items():
            key = f"{fam}__{gname}" if prefix_family else gname
            if key in out:
                raise ValueError(f"Duplicate group: {key}")
            out[key] = list(cols)
    return out


def validate_feature_groups(feature_groups, available_features, *, warn_unassigned=False):
    avail = set(available_features)
    rows, all_cols = [], []
    for g, cols in feature_groups.items():
        for c in cols:
            rows.append({"group": g, "feature": c, "exists_in_exp": c in avail})
            all_cols.append(c)
    missing = sorted(set(all_cols) - avail)
    counts = Counter(all_cols)
    dups = sorted(c for c, n in counts.items() if n > 1)
    if missing:
        raise ValueError(f"Grouped features not in exp['feature_cols']: {missing}")
    if dups:
        raise ValueError(f"Features in multiple groups: {dups}")
    if warn_unassigned:
        unassigned = sorted(avail - set(all_cols))
        if unassigned:
            print("Warning: unassigned features:", unassigned)
    return pd.DataFrame(rows)


def suggest_feature_groups(trial, feature_groups, *, min_groups=1, min_features=1):
    selected_groups, selected_features = [], []
    for gname, cols in feature_groups.items():
        if trial.suggest_categorical(f"use_group__{gname}", [False, True]):
            selected_groups.append(gname)
            selected_features.extend(cols)
    selected_features = list(dict.fromkeys(selected_features))
    if len(selected_groups) < min_groups:
        raise optuna.TrialPruned()
    if len(selected_features) < min_features:
        raise optuna.TrialPruned()
    return selected_features, selected_groups


def make_grouped_model_optuna_objective(
    exp, feature_groups, *, suggest_model_params_fn, build_model_fn,
    min_groups=1, min_features=1, group_penalty=0.0, feature_penalty=0.0,
    fixed_features: list[str] | None = None,
):
    """Build an Optuna objective for a grouped feature-selection + HP search.

    If `fixed_features` is provided, the group-selection step is bypassed:
    every trial uses exactly that feature list and only the model
    hyperparameters are tuned. Penalty terms are forced to 0 in that case
    (FS isn't being chosen, so penalising it is meaningless).
    """
    folds = exp["folds"]
    available = set(exp["feature_cols"])

    if fixed_features is not None:
        missing = [c for c in fixed_features if c not in available]
        if missing:
            raise ValueError(f"fixed_features not in exp['feature_cols']: {missing}")
        n_total_groups = 0
        n_total_features = len(fixed_features)
        # Use the fixed list; FS regularization is structurally off.
        group_penalty = 0.0
        feature_penalty = 0.0
    else:
        grouped = [c for cs in feature_groups.values() for c in cs]
        missing = [c for c in grouped if c not in available]
        if missing:
            raise ValueError(f"Grouped features not in exp['feature_cols']: {missing}")
        n_total_groups = len(feature_groups)
        n_total_features = len(set(grouped))

    def objective(trial: optuna.Trial) -> float:
        if fixed_features is not None:
            sf = list(fixed_features)
            sg: list[str] = []  # no group selection in fixed mode
        else:
            sf, sg = suggest_feature_groups(trial, feature_groups,
                                              min_groups=min_groups, min_features=min_features)
        params = suggest_model_params_fn(trial)
        maes, rmses, fold_seconds = [], [], []
        for fold in folds:
            X_tr, y_tr = fold["X_train"][sf], fold["y_train"]
            X_va, y_va = fold["X_val"][sf],   fold["y_val"]
            m = build_model_fn(params)
            _t0 = _now()
            m.fit(X_tr, y_tr)
            fold_seconds.append(float(_now() - _t0))
            pv = m.predict(X_va)
            maes.append(mean_absolute_error(y_va, pv))
            rmses.append(np.sqrt(mean_squared_error(y_va, pv)))
        mae, rmse = float(np.mean(maes)), float(np.mean(rmses))
        if fixed_features is None:
            score = mae + group_penalty * (len(sg) / n_total_groups) \
                        + feature_penalty * (len(sf) / n_total_features)
        else:
            score = mae
        trial.set_user_attr("selected_groups", sg)
        trial.set_user_attr("selected_features", sf)
        trial.set_user_attr("n_selected_groups", len(sg))
        trial.set_user_attr("n_selected_features", len(sf))
        trial.set_user_attr("mean_mae", mae)
        trial.set_user_attr("mean_rmse", rmse)
        trial.set_user_attr("fold_maes", maes)
        trial.set_user_attr("fold_rmses", rmses)
        trial.set_user_attr("fold_seconds", fold_seconds)
        trial.set_user_attr("trial_train_seconds", float(sum(fold_seconds)))
        trial.set_user_attr("model_params", params)
        return score
    return objective


In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    """Everything the experiment engine needs to run one model."""
    name: str
    family: str
    suggest_params_fn: Callable[[optuna.Trial], dict]
    build_model_fn: Callable[[dict], Any]
    min_groups: int = 1
    min_features: int = 1
    group_penalty: float = 0.0
    feature_penalty: float = 0.0


# ----- Tree family ----------------------------------------------------------
from xgboost import XGBRegressor
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("LightGBM not available — skipping LGBM_SPEC.")


def suggest_xgb_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    """Full-range XGBoost search space."""
    return {
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "tree_method": "hist",
        "random_state": random_state,
        "n_jobs": -1,
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.30, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 50.0, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-6, 50.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 200.0, log=True),
        "gamma":            trial.suggest_float("gamma", 1e-6, 20.0, log=True),
    }


def build_xgb_model(params: dict) -> XGBRegressor:
    return XGBRegressor(**params)


def suggest_lgbm_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    """Full-range LightGBM search space."""
    return {
        "objective": "regression_l1",
        "metric": "mae",
        "verbose": -1,
        "random_state": random_state,
        "n_jobs": -1,
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.30, log=True),
        "num_leaves":       trial.suggest_int("num_leaves", 15, 127, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq":   trial.suggest_int("subsample_freq", 1, 10),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-6, 50.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 200.0, log=True),
    }


def build_lgbm_model(params: dict):
    return LGBMRegressor(**params)


# ----- Linear family --------------------------------------------------------
def suggest_ridge_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    return {"alpha": trial.suggest_float("alpha", 1e-4, 1e4, log=True),
            "random_state": random_state}


def build_ridge_model(params: dict) -> Pipeline:
    return Pipeline([("scaler", StandardScaler()), ("model", Ridge(**params))])


def suggest_lasso_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    # alpha ≥ 1e-3 avoids the near-OLS regime where coordinate descent stalls
    # for tens of minutes; tol=1e-3 + max_iter=10_000 bounds the worst-case
    # convergence wall-time without measurable test-MAE loss in practice.
    return {"alpha": trial.suggest_float("alpha", 1e-3, 1e2, log=True),
            "max_iter": 10_000, "tol": 1e-3, "random_state": random_state}


def build_lasso_model(params: dict) -> Pipeline:
    return Pipeline([("scaler", StandardScaler()), ("model", Lasso(**params))])


def suggest_elasticnet_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    # See suggest_lasso_params for the alpha/max_iter/tol rationale.
    return {"alpha": trial.suggest_float("alpha", 1e-3, 1e2, log=True),
            "l1_ratio": trial.suggest_float("l1_ratio", 0.05, 0.95),
            "max_iter": 10_000, "tol": 1e-3, "random_state": random_state}


def build_elasticnet_model(params: dict) -> Pipeline:
    return Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(**params))])

In [ ]:
# ============================================================
# Tier 1 expansion — additional model specs
# ============================================================
# Tree family (added): CatBoost, HistGradientBoosting, RandomForest
# Neural family (added): MLP, TabNet
#
# Tree models receive raw features. Linear/neural models are wrapped in a
# StandardScaler pipeline. RF uses squared_error criterion (the MAE-aligned
# absolute_error variant is 10-50x slower with no measurable test-MAE gain
# in the literature for this data size); MAE remains the evaluation metric.

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not available — skipping CATBOOST_SPEC.")

try:
    from pytorch_tabnet.tab_model import TabNetRegressor
    HAS_TABNET = True
except ImportError:
    HAS_TABNET = False
    print("pytorch-tabnet not available — skipping TABNET_SPEC.")


# ----- CatBoost ------------------------------------------------------------
def suggest_catboost_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    return {
        "loss_function": "MAE",
        "eval_metric": "MAE",
        "verbose": False,
        "allow_writing_files": False,
        "random_state": random_state,
        "thread_count": -1,
        "iterations":         trial.suggest_int("iterations", 200, 1500),
        "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.30, log=True),
        "depth":              trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg":        trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "min_data_in_leaf":   trial.suggest_int("min_data_in_leaf", 1, 50),
        "subsample":          trial.suggest_float("subsample", 0.5, 1.0),
        "rsm":                trial.suggest_float("rsm", 0.5, 1.0),
        "bootstrap_type":     "Bernoulli",
    }


def build_catboost_model(params: dict):
    return CatBoostRegressor(**params)


# ----- HistGradientBoosting (sklearn) -------------------------------------
def suggest_hgbt_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    return {
        "loss": "absolute_error",
        "random_state": random_state,
        "max_iter":           trial.suggest_int("max_iter", 200, 1500),
        "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.30, log=True),
        "max_depth":          trial.suggest_int("max_depth", 4, 8),
        "max_leaf_nodes":     trial.suggest_int("max_leaf_nodes", 15, 127, log=True),
        "min_samples_leaf":   trial.suggest_int("min_samples_leaf", 5, 100, log=True),
        "l2_regularization":  trial.suggest_float("l2_regularization", 1e-6, 50.0, log=True),
        "max_bins": 255,
        "early_stopping": False,
    }


def build_hgbt_model(params: dict):
    return HistGradientBoostingRegressor(**params)


# ----- RandomForest -------------------------------------------------------
def suggest_rf_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    # Wave-1 tightened: RF wall-time was ~290s/trial at fs4 with 600 trees x depth 20.
    # Cap n_estimators to 250 and max_depth to 12 to keep per-trial budget under ~40s,
    # comparable to xgb/hgbt/catboost. Higher min_samples_leaf floor enforces practical
    # tree termination so small-leaf overfit branches do not dominate compute.
    return {
        "criterion": "squared_error",
        "n_jobs": -1,
        "random_state": random_state,
        "n_estimators":      trial.suggest_int("n_estimators", 100, 250),
        "max_depth":         trial.suggest_int("max_depth", 5, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 4, 50, log=True),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 5, 40, log=True),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", 0.3, 0.5, 0.7]),
        "bootstrap":         True,
    }


def build_rf_model(params: dict):
    return RandomForestRegressor(**params)


# ----- MLP (sklearn) ------------------------------------------------------
def suggest_mlp_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    arch = trial.suggest_categorical("arch", [
        "h64", "h128", "h256", "h128_64", "h256_128", "h256_128_64",
    ])
    arch_map = {
        "h64": (64,), "h128": (128,), "h256": (256,),
        "h128_64": (128, 64), "h256_128": (256, 128), "h256_128_64": (256, 128, 64),
    }
    return {
        "hidden_layer_sizes": arch_map[arch],
        "activation":         trial.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha":              trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        "batch_size":         trial.suggest_categorical("batch_size", [64, 128, 256]),
        "solver": "adam",
        "max_iter": 300,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 15,
        "random_state": random_state,
    }


def build_mlp_model(params: dict) -> Pipeline:
    return Pipeline([("scaler", StandardScaler()), ("model", MLPRegressor(**params))])


# ----- TabNet (pytorch-tabnet) -------------------------------------------
# TabNet's TabNetRegressor doesn't subclass sklearn cleanly. We wrap it in an
# sklearn-compatible adapter inheriting from BaseEstimator + RegressorMixin
# (required for sklearn 1.7+ which expects __sklearn_tags__).
class TabNetSklearnWrapper(BaseEstimator, RegressorMixin):
    """Adapter making TabNetRegressor work in sklearn pipelines.
       Reshapes y to 2D for fit; predict returns 1D."""
    def __init__(self, n_d=8, n_a=8, n_steps=3, gamma=1.3,
                 n_independent=2, n_shared=2, lambda_sparse=1e-4,
                 optimizer_params=None, verbose=0, seed=42,
                 max_epochs=80, patience=10, batch_size=1024,
                 virtual_batch_size=128):
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.n_independent = n_independent
        self.n_shared = n_shared
        self.lambda_sparse = lambda_sparse
        self.optimizer_params = optimizer_params or {"lr": 2e-2}
        self.verbose = verbose
        self.seed = seed
        self.max_epochs = max_epochs
        self.patience = patience
        self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size

    def _make_model(self):
        from pytorch_tabnet.tab_model import TabNetRegressor
        return TabNetRegressor(
            n_d=self.n_d, n_a=self.n_a, n_steps=self.n_steps, gamma=self.gamma,
            n_independent=self.n_independent, n_shared=self.n_shared,
            lambda_sparse=self.lambda_sparse,
            optimizer_params=dict(self.optimizer_params),
            verbose=self.verbose, seed=self.seed,
        )

    def fit(self, X, y):
        import numpy as np
        X_arr = np.asarray(X, dtype=np.float32)
        y_arr = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        self.model_ = self._make_model()
        self.model_.fit(
            X_arr, y_arr,
            max_epochs=self.max_epochs,
            patience=self.patience,
            batch_size=self.batch_size,
            virtual_batch_size=self.virtual_batch_size,
            num_workers=0,
            drop_last=False,
        )
        return self

    def predict(self, X):
        import numpy as np
        X_arr = np.asarray(X, dtype=np.float32)
        return self.model_.predict(X_arr).reshape(-1)


def suggest_tabnet_params(trial: optuna.Trial, *, random_state: int = 42) -> dict:
    n_d = trial.suggest_int("n_d", 8, 64, log=True)
    return {
        "n_d": n_d,
        "n_a": n_d,  # Common practice: n_a == n_d
        "n_steps":       trial.suggest_int("n_steps", 3, 10),
        "gamma":         trial.suggest_float("gamma", 1.0, 2.0),
        "n_independent": trial.suggest_int("n_independent", 1, 5),
        "n_shared":      trial.suggest_int("n_shared", 1, 5),
        "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-3, log=True),
        "optimizer_params": {"lr": trial.suggest_float("lr", 1e-3, 1e-1, log=True)},
        "verbose": 0,
        "seed": random_state,
    }


def build_tabnet_model(params: dict) -> Pipeline:
    if not HAS_TABNET:
        raise RuntimeError("pytorch-tabnet not installed.")
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", TabNetSklearnWrapper(**params)),
    ])
# ============================================================
# Tier 1 expansion — additional model specs
# ============================================================
# Tree family (added): CatBoost, HistGradientBoosting, RandomForest
# Neural family (added): MLP, TabNet
#
# Tree models receive raw features. Linear/neural models are wrapped in a
# StandardScaler pipeline. RF uses squared_error criterion (the MAE-aligned
# absolute_error variant is 10-50x slower with no measurable test-MAE gain
# in the literature for this data size); MAE remains the evaluation metric.

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not available — skipping CATBOOST_SPEC.")

try:
    from pytorch_tabnet.tab_model import TabNetRegressor
    HAS_TABNET = True
except ImportError:
    HAS_TABNET = False
    print("pytorch-tabnet not available — skipping TABNET_SPEC.")




In [ ]:
# Define the specs and group them by family.
XGB_SPEC = ModelSpec(
    name="xgb", family="tree",
    suggest_params_fn=suggest_xgb_params, build_model_fn=build_xgb_model,
    group_penalty=0.5,
)

if HAS_LGBM:
    LGBM_SPEC = ModelSpec(
        name="lgbm", family="tree",
        suggest_params_fn=suggest_lgbm_params, build_model_fn=build_lgbm_model,
        group_penalty=0.5,
    )

if HAS_CATBOOST:
    CATBOOST_SPEC = ModelSpec(
        name="catboost", family="tree",
        suggest_params_fn=suggest_catboost_params, build_model_fn=build_catboost_model,
        group_penalty=0.5,
    )

HGBT_SPEC = ModelSpec(
    name="hgbt", family="tree",
    suggest_params_fn=suggest_hgbt_params, build_model_fn=build_hgbt_model,
    group_penalty=0.5,
)

RF_SPEC = ModelSpec(
    name="rf", family="tree",
    suggest_params_fn=suggest_rf_params, build_model_fn=build_rf_model,
    group_penalty=0.5,
)

RIDGE_SPEC = ModelSpec(
    name="ridge", family="linear",
    suggest_params_fn=suggest_ridge_params, build_model_fn=build_ridge_model,
)
LASSO_SPEC = ModelSpec(
    name="lasso", family="linear",
    suggest_params_fn=suggest_lasso_params, build_model_fn=build_lasso_model,
)
ENET_SPEC = ModelSpec(
    name="enet", family="linear",
    suggest_params_fn=suggest_elasticnet_params, build_model_fn=build_elasticnet_model,
)

# MLP_SPEC and TABNET_SPEC are kept defined for reference but EXCLUDED from
# MODEL_FAMILIES on the CPU side. Neural models are too slow on CPU for the
# Wave-1 budget (one MLP/TabNet trial = 10-50 min CPU vs ~30 s on a 4090).
# TabNet runs separately in §10c via RunPod Flash, gated on GPU availability.
MLP_SPEC = ModelSpec(
    name="mlp", family="neural",
    suggest_params_fn=suggest_mlp_params, build_model_fn=build_mlp_model,
)

if HAS_TABNET:
    TABNET_SPEC = ModelSpec(
        name="tabnet", family="neural",
        suggest_params_fn=suggest_tabnet_params, build_model_fn=build_tabnet_model,
    )

MODEL_FAMILIES: dict[str, list[ModelSpec]] = {
    "tree": (
        [XGB_SPEC]
        + ([LGBM_SPEC] if HAS_LGBM else [])
        + ([CATBOOST_SPEC] if HAS_CATBOOST else [])
        + [HGBT_SPEC, RF_SPEC]
    ),
    "linear": [RIDGE_SPEC, LASSO_SPEC, ENET_SPEC],
    "neural": [],   # GPU-only — TabNet runs separately via §10c (RunPod Flash)
}

print("Registered model families (CPU sweep):")
for fam, specs in MODEL_FAMILIES.items():
    print(f"  {fam}: {[s.name for s in specs]}")


In [ ]:
# Private aliases — `time` in this notebook is `datetime.time` (from cell 2's
# `from datetime import time`), so we use the real time module under a private
# name. Likewise `subprocess` is imported here rather than touching cell 2.
from time import monotonic as _now


def regression_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "mae":  float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2":   float(r2_score(y_true, y_pred)),
    }


def short_variant_name(variant_name: str) -> str:
    return variant_name.split("_", 1)[0].lower()


def make_prediction_col_name(model_name: str, variant_name: str) -> str:
    return f"{model_name}_{short_variant_name(variant_name)}"


# --- Thesis training-environment helpers ---------------------------------
def _cpu_hardware_label() -> str:
    """Return CPU model + vCPU count, read from /proc/cpuinfo."""
    name = "unknown CPU"
    try:
        with open("/proc/cpuinfo") as _f:
            for _ln in _f:
                if _ln.startswith("model name"):
                    name = _ln.split(":", 1)[1].strip()
                    break
    except Exception:
        pass
    return f"{name} ({os.cpu_count()} vCPU)"


def _ledger_full(*, device: str = "cpu", hardware: str | None = None,
                 n_trials: int = 0, train_total_seconds: float = 0.0,
                 study_wall_seconds: float = 0.0,
                 final_fit_seconds: float = 0.0) -> dict:
    """Uniform thesis training-environment ledger for any model row."""
    if hardware is None:
        hardware = _cpu_hardware_label() if device == "cpu" else "unknown"
    return {
        "compute_device":       device,
        "hardware":             hardware,
        "n_trials":             int(n_trials),
        "train_total_seconds":  float(train_total_seconds),
        "study_wall_seconds":   float(study_wall_seconds),
        "per_trial_seconds":    float(train_total_seconds / n_trials) if n_trials > 0 else 0.0,
        "final_fit_seconds":    float(final_fit_seconds),
    }


def make_validation_predictions_from_best_trial(*, exp, model_spec, best_trial,
                                                  prediction_col, actual_col="actual"):
    sf = best_trial.user_attrs["selected_features"]
    params = best_trial.user_attrs["model_params"]
    val_frames, fold_rows = [], []
    for fold_id, fold in enumerate(exp["folds"]):
        X_tr, y_tr = fold["X_train"].loc[:, sf], fold["y_train"]
        X_va, y_va = fold["X_val"].loc[:, sf],   fold["y_val"]
        m = model_spec.build_model_fn(dict(params))
        m.fit(X_tr, y_tr)
        pv = m.predict(X_va)
        f_df = pd.DataFrame({actual_col: y_va, prediction_col: pv}, index=y_va.index)
        f_df["fold"] = fold_id
        f_df["error"] = f_df[prediction_col] - f_df[actual_col]
        f_df["abs_error"] = f_df["error"].abs()
        val_frames.append(f_df)
        mt = regression_metrics(y_va, pv)
        fold_rows.append({"fold": fold_id, "start": y_va.index.min(), "end": y_va.index.max(),
                          "n": len(y_va), **mt})
    return pd.concat(val_frames).sort_index(), pd.DataFrame(fold_rows)


def make_test_predictions_from_best_trial(*, exp, model_spec, best_trial,
                                           prediction_col, test_predictions_df=None,
                                           actual_col="actual"):
    sf = best_trial.user_attrs["selected_features"]
    params = best_trial.user_attrs["model_params"]
    X_tr, y_tr = exp["final"]["X_train"].loc[:, sf], exp["final"]["y_train"]
    X_te, y_te = exp["final"]["X_test"].loc[:, sf],  exp["final"]["y_test"]
    m = model_spec.build_model_fn(dict(params))
    _t0 = _now()
    m.fit(X_tr, y_tr)
    final_fit_seconds = float(_now() - _t0)
    p_te = m.predict(X_te)
    y_true = pd.Series(y_te, index=y_te.index, name=actual_col)
    p_ser = pd.Series(p_te, index=y_true.index, name=prediction_col)
    out = test_predictions_df.copy() if test_predictions_df is not None else pd.DataFrame(index=y_true.index)
    out = out.reindex(y_true.index)
    if actual_col not in out.columns:
        out[actual_col] = y_true
    out[prediction_col] = p_ser
    return out, m, regression_metrics(y_true, p_ser), final_fit_seconds


def run_grouped_model_study(*, exp, model_spec, variant_name,
                             feature_set_variants, feature_groups_by_family,
                             test_predictions_df=None, n_trials=100, seed=42,
                             show_progress_bar=True, actual_col="actual",
                             fixed_features: list[str] | None = None,
                             pred_col_override: str | None = None,
                             ablation_mode: str | None = None):
    """
    Optuna study for one (model, variant) pair.

    `fixed_features`: when provided, bypass group selection and use this exact
    feature list. Penalties are forced to 0 (no FS regularization).
    `pred_col_override`: name to use for the prediction column (e.g.
    "xgb_fs1_fix" instead of the default "xgb_fs1").
    `ablation_mode`: stored verbatim on summary_row for downstream pivot tables.
    """
    pred_col = pred_col_override or make_prediction_col_name(model_spec.name, variant_name)
    print("=" * 80)
    print(f"Running: {pred_col}  (family={model_spec.family}, "
          f"mode={ablation_mode or 'default'})")
    print("=" * 80)

    fg = resolve_feature_groups_for_variant(
        variant_name, feature_set_variants, feature_groups_by_family, prefix_family=True,
    )
    if fixed_features is None:
        validate_feature_groups(fg, exp["feature_cols"])
    else:
        # Validate fixed_features are in exp before paying the Optuna cost.
        avail = set(exp["feature_cols"])
        missing = [c for c in fixed_features if c not in avail]
        if missing:
            raise ValueError(f"fixed_features missing from exp['feature_cols']: {missing}")

    objective = make_grouped_model_optuna_objective(
        exp, fg,
        suggest_model_params_fn=model_spec.suggest_params_fn,
        build_model_fn=model_spec.build_model_fn,
        min_groups=model_spec.min_groups, min_features=model_spec.min_features,
        group_penalty=model_spec.group_penalty, feature_penalty=model_spec.feature_penalty,
        fixed_features=fixed_features,
    )
    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed),
        study_name=f"{pred_col}_grouped",
    )
    _study_t0 = _now()
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=show_progress_bar)
    study_wall_seconds = float(_now() - _study_t0)
    completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
    if not completed:
        raise RuntimeError(f"No completed trials for {pred_col}.")
    bt = study.best_trial

    val_pred_df, fold_metrics = make_validation_predictions_from_best_trial(
        exp=exp, model_spec=model_spec, best_trial=bt,
        prediction_col=pred_col, actual_col=actual_col,
    )
    combined_val = regression_metrics(val_pred_df[actual_col], val_pred_df[pred_col])

    test_predictions_df, final_model, test_metrics, final_fit_seconds = (
        make_test_predictions_from_best_trial(
            exp=exp, model_spec=model_spec, best_trial=bt,
            prediction_col=pred_col, test_predictions_df=test_predictions_df, actual_col=actual_col,
        )
    )

    sg = bt.user_attrs["selected_groups"]
    sf = bt.user_attrs["selected_features"]

    cv_train_total_seconds = float(sum(
        t.user_attrs.get("trial_train_seconds", 0.0) for t in completed
    ))
    train_total_seconds = cv_train_total_seconds + final_fit_seconds

    if fixed_features is not None:
        n_candidate_groups = 0
        n_candidate_features = len(fixed_features)
    else:
        n_candidate_groups = len(fg)
        n_candidate_features = len(set(c for cs in fg.values() for c in cs))

    summary = {
        "run_name": pred_col, "model": model_spec.name,
        "family": model_spec.family, "feature_variant": variant_name,
        "ablation_mode": ablation_mode or ("fixed" if fixed_features is not None
                                            else "no_penalty"),
        "best_trial": bt.number, "best_objective": float(bt.value),
        "mean_fold_val_mae":  bt.user_attrs["mean_mae"],
        "mean_fold_val_rmse": bt.user_attrs["mean_rmse"],
        "combined_val_mae":   combined_val["mae"],
        "combined_val_rmse":  combined_val["rmse"],
        "combined_val_r2":    combined_val["r2"],
        "test_mae":  test_metrics["mae"], "test_rmse": test_metrics["rmse"], "test_r2": test_metrics["r2"],
        "n_candidate_groups":   n_candidate_groups,
        "n_candidate_features": n_candidate_features,
        "n_selected_groups":    len(sg),
        "n_selected_features":  len(sf),
        "selected_groups":      sg,
        "selected_groups_str":  ", ".join(sg),
        "selected_features":    sf,
        "model_params":         bt.user_attrs["model_params"],
        **_ledger_full(
            device="cpu",
            n_trials=len(completed),
            train_total_seconds=train_total_seconds,
            study_wall_seconds=study_wall_seconds,
            final_fit_seconds=final_fit_seconds,
        ),
    }
    print(f"\nval MAE (mean fold): {summary['mean_fold_val_mae']:.4f}")
    print(f"val MAE (combined):  {summary['combined_val_mae']:.4f}")
    print(f"test MAE:            {summary['test_mae']:.4f}")
    if fixed_features is None:
        print(f"selected groups:     {summary['n_selected_groups']}/{summary['n_candidate_groups']}")
        print(f"selected features:   {summary['n_selected_features']}/{summary['n_candidate_features']}")
    else:
        print(f"fixed features:      {summary['n_selected_features']} (intuition set)")
    print(f"train wall:          {summary['study_wall_seconds']:.1f}s study, "
          f"{summary['train_total_seconds']:.1f}s pure-fit on {summary['hardware']}")

    return {
        "run_name": pred_col, "prediction_col": pred_col,
        "study": study, "best_trial": bt, "final_model": final_model,
        "test_predictions_df": test_predictions_df, "val_predictions_df": val_pred_df,
        "fold_metrics_df": fold_metrics, "summary_row": summary,
    }


## 10. Run all experiments

Initialise `test_predictions_df` with the actual target plus a 24h-persistence
benchmark, then loop `family → model → feature_set` and accumulate
predictions.

For a quick smoke run set `N_TRIALS_OVERRIDE = 5` and limit `ACTIVE_FS` to
`["fs1"]`. The full sweep at `N_TRIALS=150` × 5 models × 3 FS = 15 studies
is slow but embarrassingly parallel across cells if you split it.

In [ ]:
def initialize_test_predictions_df(exp, *, target_col, baseline_cols=None,
                                    baseline_source_df=None, actual_col="actual"):
    test_df = exp["final"]["test_df"]
    out = pd.DataFrame(index=test_df.index)
    if target_col in test_df.columns:
        out[actual_col] = test_df[target_col]
    else:
        out[actual_col] = exp["final"]["y_test"]
    if baseline_cols:
        for c in baseline_cols:
            if c in test_df.columns:
                out[c] = test_df[c]
            elif baseline_source_df is not None and c in baseline_source_df.columns:
                out[c] = baseline_source_df.reindex(out.index)[c]
            else:
                raise KeyError(f"Baseline {c!r} not found in test_df or baseline_source_df.")
    return out


# Persistence benchmark: 24h lag of the target — already exists in df_baselines
test_predictions_df = initialize_test_predictions_df(
    exp,
    target_col=TARGET_COL,
    baseline_cols=["price_baseline_naive_d1", "price_baseline_naive_w1",
                   "price_baseline_naive_epf_hybrid"],
    baseline_source_df=df_baselines,
)
print(test_predictions_df.head())


# ---------------------------------------------------------------------------
# Initialize global `results` list with persistence baselines. Subsequent
# baseline cells (climatology, LEAR) and the Optuna run-loop (cell 51) all
# extend this same list. Re-running cell 45 resets it; running 47/49/51
# without re-running 45 first would double-count baselines.
# ---------------------------------------------------------------------------
def _baseline_summary_row(*, run_name, model, feature_variant,
                           train_seconds=0.0, study_wall_seconds=None,
                           final_fit_seconds=None,
                           n_features=0, model_params=None,
                           family="baseline"):
    """Construct a `results`-shaped row for any non-Optuna model.
       train_seconds covers all fit work (folds + final). study_wall_seconds
       defaults to train_seconds; final_fit_seconds defaults to train_seconds."""
    if study_wall_seconds is None:
        study_wall_seconds = train_seconds
    if final_fit_seconds is None:
        final_fit_seconds = train_seconds
    y_true = test_predictions_df["actual"]
    y_pred = test_predictions_df[run_name]
    valid = ~y_pred.isna()
    if valid.any():
        m = regression_metrics(y_true[valid], y_pred[valid])
    else:
        m = {"mae": float("nan"), "rmse": float("nan"), "r2": float("nan")}
    return {
        "run_name": run_name, "model": model, "family": family, "feature_variant": feature_variant,
        "best_trial": -1, "best_objective": float("nan"),
        "mean_fold_val_mae": float("nan"), "mean_fold_val_rmse": float("nan"),
        "combined_val_mae": float("nan"), "combined_val_rmse": float("nan"), "combined_val_r2": float("nan"),
        "test_mae": m["mae"], "test_rmse": m["rmse"], "test_r2": m["r2"],
        "n_candidate_groups": 0, "n_candidate_features": int(n_features),
        "n_selected_groups": 0, "n_selected_features": int(n_features),
        "selected_groups": [], "selected_groups_str": "", "selected_features": [],
        "model_params": model_params or {},
        **_ledger_full(
            device="cpu", n_trials=0,
            train_total_seconds=train_seconds,
            study_wall_seconds=study_wall_seconds,
            final_fit_seconds=final_fit_seconds,
        ),
    }


results = []
for col in ["price_baseline_naive_d1", "price_baseline_naive_w1",
             "price_baseline_naive_epf_hybrid"]:
    if col in test_predictions_df.columns:
        results.append(_baseline_summary_row(
            run_name=col, model=col, feature_variant="lag",
            train_seconds=0.0, model_params={"kind": "persistence"},
        ))
print(f"Initialized `results` with {len(results)} persistence baseline rows.")


### 10a. Climatology baseline

Adds one more naive benchmark alongside the existing `naive_d1` (= D-1
persistence), `naive_w1` (= D-7 persistence), and `naive_epf_hybrid` (Lago's
2021 hybrid: D-7 on Mon/Sat/Sun, D-1 on Tue–Fri):

- `climatology_how` — **mean target by hour-of-week** computed on the dev
  partition, then broadcast to test rows by their hour-of-week index.

This is the lowest "structural-only" baseline — it captures *only* the
calendar pattern, no recent-price information at all. Any model worth its
compute has to clear it.


In [ ]:
# Climatology baseline: mean of target per hour-of-week computed on dev only,
# broadcast to test rows by their hour-of-week.
def hour_of_week(idx: pd.DatetimeIndex) -> np.ndarray:
    """Returns 0..167. Computed in the local TZ for calendar consistency."""
    local = idx.tz_convert(TZ)
    return (local.dayofweek * 24 + local.hour).to_numpy()


def add_climatology_baseline(test_predictions_df, *, exp, target_col, col_name="climatology_how"):
    y_dev = exp["final"]["y_train"]
    dev_idx = exp["final"]["X_train"].index
    test_idx = test_predictions_df.index

    how_dev = hour_of_week(dev_idx)
    clim_table = pd.Series(y_dev.values, index=how_dev).groupby(level=0).mean()
    how_test = hour_of_week(test_idx)
    test_predictions_df[col_name] = clim_table.reindex(how_test).to_numpy()
    return test_predictions_df


_clim_t0 = _now()
test_predictions_df = add_climatology_baseline(
    test_predictions_df, exp=exp, target_col=TARGET_COL,
)
_clim_seconds = float(_now() - _clim_t0)

print("Climatology baseline added.")
print(test_predictions_df.head())
print(f"\nclimatology_how NaN count: {test_predictions_df['climatology_how'].isna().sum()}")

results.append(_baseline_summary_row(
    run_name="climatology_how", model="climatology", feature_variant="hour_of_week",
    train_seconds=_clim_seconds, n_features=168,
    model_params={"method": "mean_per_hour_of_week", "n_classes": 168},
))
print(f"Climatology row appended (train_seconds={_clim_seconds:.4f}).")


### 10b. LEAR baseline (Lago 2021)

LASSO-Estimated AutoRegressive — the canonical "strong baseline" in the
day-ahead price forecasting literature (Lago, Marcjasz, De Schutter, Weron,
*Applied Energy* 2021). It is **not an Optuna-tuned model** — alpha is
auto-selected per delivery period via `LassoLarsIC(criterion='aic')`.

Recipe (mirrors the canonical `epftoolbox.models.LEAR`):

- One model per delivery period (96 quarter-hour models)
- Asinh-median scaling on Y and on non-dummy X
- Daily-profile features: target lags at D-1 / D-2 / D-3 / D-7
  (full 96-quarter profile of each), plus exogenous lags at D / D-1 / D-7
  (same daily-profile shape), plus 7 day-of-week dummies
- Lean exogenous set, 2 vars: `fund_load_fc`, `fund_residual_load_fc`
  (same minimalism as Lago's load + RES setup)

**Named deviations from canonical LEAR**, documented in thesis methodology:
1. **96 quarter-hourly sub-models** (Lago: 24 hourly). DK1 DA market is
   PT15M since 2025-10-01.
2. **No daily recalibration** (Lago: daily rolling-window refit). We use
   the same walk-forward folds as everyone else for clean comparison.

Compute is trivial (~30 s total) — each per-quarter Lasso fits on ~150 rows
× ~970 features.


In [ ]:
from sklearn.linear_model import Lasso, LassoLarsIC


# ---------------------------------------------------------------------------
# LEAR feature builder — daily-profile lags
# ---------------------------------------------------------------------------
def _pivot_to_daily_profile(s: pd.Series, *, periods_per_day: int = 96) -> pd.DataFrame:
    """Pivot a PT15M series to a (date, quarter) matrix in UTC. Quarter index 0..95."""
    df_p = s.to_frame("v").copy()
    df_p["date"] = df_p.index.normalize()
    df_p["q"] = df_p.index.hour * 4 + df_p.index.minute // 15
    return df_p.pivot_table(values="v", index="date", columns="q")


def build_lear_features(
    df: pd.DataFrame,
    *,
    target_col: str,
    exog_cols: list[str],
    lag_days: tuple[int, ...] = (1, 2, 3, 7),
    exog_lag_days: tuple[int, ...] = (0, 1, 7),
    periods_per_day: int = 96,
) -> pd.DataFrame:
    """Build LEAR-style daily-profile feature matrix.

    For each row at delivery time t:
      - target lags: periods_per_day x len(lag_days) features
      - exog lags:   periods_per_day x len(exog_lag_days) per exog var
      - DOW dummies: 7 features
    """
    out = pd.DataFrame(index=df.index)

    target_pivot = _pivot_to_daily_profile(df[target_col], periods_per_day=periods_per_day)
    for d in lag_days:
        past_dates = (df.index.normalize() - pd.Timedelta(days=d))
        block = target_pivot.reindex(past_dates).set_axis(df.index)
        block.columns = [f"lear_{target_col}_lag{d}d_q{q:02d}" for q in block.columns]
        out = pd.concat([out, block], axis=1)

    for exog in exog_cols:
        if exog not in df.columns:
            print(f"  WARN: exog {exog!r} missing - skipped")
            continue
        exog_pivot = _pivot_to_daily_profile(df[exog], periods_per_day=periods_per_day)
        for d in exog_lag_days:
            past_dates = (df.index.normalize() - pd.Timedelta(days=d))
            block = exog_pivot.reindex(past_dates).set_axis(df.index)
            block.columns = [f"lear_{exog}_lag{d}d_q{q:02d}" for q in block.columns]
            out = pd.concat([out, block], axis=1)

    dow_local = np.asarray(df.index.tz_convert(TZ).dayofweek)
    for d in range(7):
        out[f"lear_dow_{d}"] = (dow_local == d).astype(int)

    return out


# ---------------------------------------------------------------------------
# LEAR per-period regressor
# ---------------------------------------------------------------------------
class LEARRegressor:
    """Per-period Lasso with asinh-median scaling on Y and non-dummy X.
       Alpha auto-selected via LassoLarsIC. No hyperparameter sweep."""

    def __init__(self, criterion: str = "aic", periods_per_day: int = 96, max_iter: int = 2500):
        self.criterion = criterion
        self.periods_per_day = periods_per_day
        self.max_iter = max_iter
        self.models: dict[int, dict] = {}

    @staticmethod
    def _period_index(X: pd.DataFrame) -> np.ndarray:
        idx = X.index
        return idx.hour.values * 4 + idx.minute.values // 15

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "LEARRegressor":
        periods = self._period_index(X)
        dummy_cols = [c for c in X.columns if c.startswith("lear_dow_")]
        scaled_cols = [c for c in X.columns if c not in dummy_cols]

        for q in range(self.periods_per_day):
            mask = periods == q
            if int(mask.sum()) < 10:
                continue

            Xq, yq = X.loc[mask], y.loc[mask]

            x_med = Xq[scaled_cols].median()
            x_mad = (Xq[scaled_cols] - x_med).abs().median().replace(0, 1.0)
            y_med = float(yq.median())
            y_mad = float(max((yq - y_med).abs().median(), 1e-6))

            Xq_scaled = Xq.copy()
            Xq_scaled[scaled_cols] = np.arcsinh(
                (Xq[scaled_cols] - x_med) / x_mad
            )
            yq_scaled = np.arcsinh((yq - y_med) / y_mad)

            larsic = LassoLarsIC(
                criterion=self.criterion,
                max_iter=self.max_iter,
                noise_variance=1.0,
            )
            larsic.fit(Xq_scaled.values, yq_scaled.values)

            lasso = Lasso(alpha=max(larsic.alpha_, 1e-8), max_iter=self.max_iter)
            lasso.fit(Xq_scaled.values, yq_scaled.values)

            self.models[q] = dict(
                lasso=lasso, scaled_cols=scaled_cols,
                x_med=x_med, x_mad=x_mad,
                y_med=y_med, y_mad=y_mad,
                n_features=int((np.abs(lasso.coef_) > 0).sum()),
            )
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        out = np.full(len(X), np.nan)
        periods = self._period_index(X)
        for q, m in self.models.items():
            mask = periods == q
            if not mask.any():
                continue
            Xq = X.loc[mask]
            Xq_scaled = Xq.copy()
            Xq_scaled[m["scaled_cols"]] = np.arcsinh(
                (Xq[m["scaled_cols"]] - m["x_med"]) / m["x_mad"]
            )
            yps = m["lasso"].predict(Xq_scaled.values)
            out[mask] = m["y_med"] + m["y_mad"] * np.sinh(yps)
        return out


# ---------------------------------------------------------------------------
# LEAR runner — single fit per fold + final on dev (with thesis-ledger timing)
# ---------------------------------------------------------------------------
LEAR_EXOG_COLS = ["fund_load_fc", "fund_residual_load_fc"]

print("Building LEAR features...")
lear_X_full = build_lear_features(
    pd.concat([df[[TARGET_COL] + LEAR_EXOG_COLS]], axis=1),
    target_col=TARGET_COL,
    exog_cols=LEAR_EXOG_COLS,
)
print(f"  shape: {lear_X_full.shape}")
print(f"  NaN cells: {int(lear_X_full.isna().sum().sum())} (mostly the warm-up rows)")

# Slice to dev/test using exp's window
lear_X_dev = lear_X_full.reindex(exp["final"]["X_train"].index)
lear_X_test = lear_X_full.reindex(exp["final"]["test_df"].index)

# Drop any NaN rows that survive (early dev rows where lag-7d is unavailable
# despite warm-up trim - edge cases at fold boundaries).
dev_valid = ~lear_X_dev.isna().any(axis=1)
test_valid = ~lear_X_test.isna().any(axis=1)
print(f"  dev valid rows: {int(dev_valid.sum())} / {len(lear_X_dev)}")
print(f"  test valid rows: {int(test_valid.sum())} / {len(lear_X_test)}")


def run_lear_baseline(*, exp, lear_X_dev, lear_X_test, dev_valid, test_valid):
    fold_maes = []
    fold_seconds = []
    val_pred_frames = []
    for fold_idx, fold in enumerate(exp["folds"]):
        train_idx = fold["X_train"].index
        val_idx = fold["X_val"].index

        X_tr = lear_X_dev.loc[train_idx].dropna()
        y_tr = exp["final"]["y_train"].reindex(X_tr.index)
        X_va = lear_X_dev.loc[val_idx].dropna()
        y_va = exp["final"]["y_train"].reindex(X_va.index)

        _t0 = _now()
        m = LEARRegressor().fit(X_tr, y_tr)
        y_pred = m.predict(X_va)
        fold_seconds.append(float(_now() - _t0))
        valid = ~np.isnan(y_pred)
        mae = float(np.mean(np.abs(y_pred[valid] - y_va.values[valid])))
        fold_maes.append(mae)
        val_pred_frames.append(pd.DataFrame(
            {"actual": y_va.values, "lear": y_pred, "fold": fold_idx},
            index=y_va.index,
        ))
        print(f"  Fold {fold_idx + 1}/5: train={len(X_tr):>5d}  val={len(X_va):>5d}  MAE={mae:.3f}  fit={fold_seconds[-1]:.1f}s")

    # Final fit on all valid dev rows, predict on all test rows
    Xd = lear_X_dev.loc[dev_valid]
    yd = exp["final"]["y_train"].reindex(Xd.index)
    Xt = lear_X_test.loc[test_valid]
    yt = exp["final"]["y_test"].reindex(Xt.index)

    _t0 = _now()
    final = LEARRegressor().fit(Xd, yd)
    y_test_pred = final.predict(Xt)
    final_fit_seconds = float(_now() - _t0)

    test_mae = float(np.mean(np.abs(y_test_pred - yt.values)))
    test_rmse = float(np.sqrt(np.mean((y_test_pred - yt.values) ** 2)))
    print(f"\n  TEST  MAE={test_mae:.3f}  RMSE={test_rmse:.3f}  ({len(yt)} rows)  final_fit={final_fit_seconds:.1f}s")
    print(f"  Per-fold val MAE mean: {np.mean(fold_maes):.3f}  std: {np.std(fold_maes):.3f}")

    val_pred_df = pd.concat(val_pred_frames).sort_index() if val_pred_frames else pd.DataFrame()

    return {
        "model": "lear", "feature_set": "lear-fs",
        "n_features": Xd.shape[1],
        "fold_maes": fold_maes,
        "fold_seconds": fold_seconds,
        "final_fit_seconds": final_fit_seconds,
        "y_test_pred": y_test_pred,
        "test_index": Xt.index,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "val_pred_df": val_pred_df,
    }


print("\nRunning LEAR (no Optuna) - 5 folds + final fit on dev...")
_lear_study_t0 = _now()
lear_result = run_lear_baseline(
    exp=exp, lear_X_dev=lear_X_dev, lear_X_test=lear_X_test,
    dev_valid=dev_valid, test_valid=test_valid,
)
_lear_study_wall = float(_now() - _lear_study_t0)

# Add LEAR predictions to test_predictions_df (NaN-pad rows that LEAR couldn't predict)
lear_pred_series = pd.Series(np.nan, index=test_predictions_df.index, name="lear")
lear_pred_series.loc[lear_result["test_index"]] = lear_result["y_test_pred"]
test_predictions_df["lear"] = lear_pred_series
print(f"\nLEAR column added to test_predictions_df. NaN count: {int(test_predictions_df['lear'].isna().sum())}")

# --- Build LEAR summary row with thesis ledger ---
_lear_train_total = float(sum(lear_result["fold_seconds"])) + lear_result["final_fit_seconds"]
_lear_y_true = test_predictions_df["actual"]
_lear_y_pred = test_predictions_df["lear"]
_lear_valid = ~_lear_y_pred.isna()
_lear_metrics = regression_metrics(_lear_y_true[_lear_valid], _lear_y_pred[_lear_valid])
_lear_fold_mean = float(np.mean(lear_result["fold_maes"]))

results.append({
    "run_name": "lear", "model": "lear", "family": "linear", "feature_variant": "lear-fs",
    "best_trial": -1, "best_objective": float("nan"),
    "mean_fold_val_mae": _lear_fold_mean, "mean_fold_val_rmse": float("nan"),
    "combined_val_mae": _lear_fold_mean, "combined_val_rmse": float("nan"), "combined_val_r2": float("nan"),
    "test_mae": _lear_metrics["mae"], "test_rmse": _lear_metrics["rmse"], "test_r2": _lear_metrics["r2"],
    "n_candidate_groups": 0, "n_candidate_features": int(lear_result["n_features"]),
    "n_selected_groups": 0, "n_selected_features": int(lear_result["n_features"]),
    "selected_groups": ["lear-features"], "selected_groups_str": "lear-features", "selected_features": [],
    "model_params": {"criterion": "aic", "periods_per_day": 96, "scaling": "asinh-median"},
    **_ledger_full(
        device="cpu", n_trials=0,
        train_total_seconds=_lear_train_total,
        study_wall_seconds=_lear_study_wall,
        final_fit_seconds=lear_result["final_fit_seconds"],
    ),
})
print(f"LEAR row appended (train_total={_lear_train_total:.1f}s, study_wall={_lear_study_wall:.1f}s).")


In [ ]:
# Build baseline val-period predictions for §11 figures and §11b rMAE table.
# - Persistence baselines: pure lag, slice df_baselines on the val index.
# - Climatology: re-apply dev hour-of-week mean to val.
# - LEAR: walk-forward predictions collected by run_lear_baseline (cell 49).
# Build val_idx via Index.union to preserve tz-awareness (np.concatenate loses
# tz, which breaks the .loc lookup against tz-aware df_baselines below).
val_idx = exp["folds"][0]["X_val"].index
for fold in exp["folds"][1:]:
    val_idx = val_idx.union(fold["X_val"].index)
val_idx = val_idx.sort_values()

baseline_val_predictions: dict[str, pd.Series] = {}

# 1) Persistence baselines — direct slice of df_baselines on val index.
for col in ("price_baseline_naive_d1",
            "price_baseline_naive_w1",
            "price_baseline_naive_epf_hybrid"):
    if col in df_baselines.columns:
        s = df_baselines.loc[val_idx, col].rename(col).dropna()
        baseline_val_predictions[col] = s

# 2) Climatology — hour-of-week mean computed on dev, applied to val.
y_dev = exp["final"]["y_train"]
dev_idx = exp["final"]["X_train"].index
how_dev = hour_of_week(dev_idx)
how_dev_mean = pd.Series(y_dev.values, index=how_dev).groupby(level=0).mean()
how_val = hour_of_week(val_idx)
clim_val = pd.Series(how_dev_mean.reindex(how_val).values,
                      index=val_idx, name="climatology_how").dropna()
baseline_val_predictions["climatology_how"] = clim_val

# 3) LEAR — walk-forward predictions collected by run_lear_baseline.
if "val_pred_df" in lear_result and not lear_result["val_pred_df"].empty:
    baseline_val_predictions["lear"] = (
        lear_result["val_pred_df"]["lear"].rename("lear")
    )
else:
    print("WARNING: lear_result has no val_pred_df — LEAR val panels will be empty.")

print(f"baseline_val_predictions keys: {list(baseline_val_predictions.keys())}")
for k, s in baseline_val_predictions.items():
    print(f"  {k}: {len(s)} rows, NaN={int(s.isna().sum())}")


In [ ]:
# ============================================================
# Run mode config + GPU availability gate
# ============================================================
# RUN_SMOKE      = True  → fast iteration: N_TRIALS=2, restricted spec subset.
#                  False → full Wave-1 campaign (uses N_TRIALS from cell 3).
#
# INCLUDE_TABNET = True  → run TabNet on GPU via RunPod Flash (§10c).
#                          Requires configured GPU and object-store credentials,
#                          plus runpod_flash importability. If GPU is unavailable,
#                          TabNet is SKIPPED — it never falls back to CPU
#                          (would take >1 h per trial).
RUN_SMOKE      = False
INCLUDE_TABNET = True

# --- GPU gate (cheap probe — env vars + import check, no network call) -----
def _gpu_available() -> tuple[bool, str]:
    import os
    import importlib.util
    required = ["GPU_PROVIDER_AUTH", "OBJECT_STORE_ACCESS",
                "OBJECT_STORE_PRIVATE", "OBJECT_STORE_ENDPOINT"]
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        return False, f"missing env vars: {missing}"
    if importlib.util.find_spec("runpod_flash") is None:
        return False, "runpod_flash not importable"
    return True, "GPU and object-store credentials configured, runpod_flash importable"

GPU_AVAILABLE, _gpu_reason = _gpu_available()
INCLUDE_NEURAL_GPU = INCLUDE_TABNET and GPU_AVAILABLE
print(f"[gpu-gate] available={GPU_AVAILABLE} — {_gpu_reason}")
if INCLUDE_TABNET and not GPU_AVAILABLE:
    print("[gpu-gate] WARNING: INCLUDE_TABNET=True but GPU unavailable. "
          "TabNet will be SKIPPED in §10c (no CPU fallback by design).")

# --- Smoke override --------------------------------------------------------
if RUN_SMOKE:
    N_TRIALS   = 2
    ACTIVE_FS  = ["fs1", "fs2", "fs3", "fs4"]
    SMOKE_NAMES = {"xgb", "catboost", "hgbt", "rf", "ridge", "lasso", "enet"}
    MODEL_FAMILIES = {
        fam: [s for s in specs if s.name in SMOKE_NAMES]
        for fam, specs in MODEL_FAMILIES.items()
    }
    print("=== SMOKE CONFIG ACTIVE ===")
    print(f"  N_TRIALS   = {N_TRIALS}")
    print(f"  ACTIVE_FS  = {ACTIVE_FS}")

# --- Drop empty families (e.g. neural — always empty on CPU side) ----------
MODEL_FAMILIES = {fam: specs for fam, specs in MODEL_FAMILIES.items() if specs}

# --- Final summary ---------------------------------------------------------
print("\n=== Active sweep config ===")
print(f"  RUN_SMOKE          = {RUN_SMOKE}")
print(f"  INCLUDE_TABNET     = {INCLUDE_TABNET}")
print(f"  GPU_AVAILABLE      = {GPU_AVAILABLE}")
print(f"  INCLUDE_NEURAL_GPU = {INCLUDE_NEURAL_GPU}")
print(f"  N_TRIALS           = {N_TRIALS}")
print(f"  ACTIVE_FS          = {ACTIVE_FS}")
print("  CPU sweep models per family:")
for fam, specs in MODEL_FAMILIES.items():
    print(f"    {fam}: {[s.name for s in specs]}")
if INCLUDE_NEURAL_GPU:
    print(f"  GPU sweep (§10c):  tabnet × {len(ACTIVE_FS)} feature variants")


## 10c-pre. Ablation configuration

Three ablation modes per (model, FS) cell on the CPU side:

1. **fixed** — bypass Optuna feature selection; use a small intuition-based feature set (~5 cols/family, ~14 cols total at FS3). HP search only.
2. **no_penalty** — Optuna group-selection + HP search, no FS regularization. This is the framework's "out-of-the-box" mode.
3. **with_penalty** — Optuna group-selection + HP search, with `group_penalty=0.3` and `feature_penalty=0.02` added to fold MAE. Penalises over-selection in EUR/MWh units (4 EUR/MWh penalty for picking 10 groups + 50 features ≈ 5% of typical fold MAE).

TabNet (§10c) is GPU-only and runs once under the `no_penalty` mode for cost reasons; the rest of the GPU-side ablation is left for future work.

In [ ]:
# ============================================================
# Ablation configuration — three modes per (model, FS variant)
# ============================================================
# Mode A: "fixed"        → use a hand-picked intuition feature set, HP-tune only.
# Mode B: "no_penalty"   → Optuna group + HP search, no FS penalties.
# Mode C: "with_penalty" → Optuna group + HP search, regularize FS choice.
#
# All three modes run for every CPU-side spec × variant when ABLATION_MODES_ACTIVE
# contains all three. TabNet (§10c) is GPU-only and runs once under "no_penalty".

from dataclasses import replace as _replace_dc

# ----------------------------------------------------------------------------
# Intuition-based feature set — what a domain expert would pick without
# optimization. Chosen for distinct DA-price drivers per family. Keep small
# (~5 cols/family) so "fixed" is meaningfully tighter than the Optuna search
# space (which can pick all groups → 100+ features).
# ----------------------------------------------------------------------------
INTUITION_FEATURES: dict[str, list[str]] = {
    "time": [
        "time_hour",                          # intraday position
        "time_day_of_week",                   # weekly demand cycle
        "time_month",                         # annual cycle
        "time_is_weekend",                    # day-type
        "time_is_holiday_dk",                 # DK holidays
    ],
    "price": [
        "price_lag_1d_same_period",           # same period yesterday
        "price_lag_7d_same_period",           # same period last week
        "neighbor_price_de_lu_lag_24h",       # DE-LU coupling, 24h-safe
        "neighbor_price_se3_lag_24h",         # SE3 coupling, 24h-safe
    ],
    "fundamentals": [
        "fund_load_fc",                       # demand level (D-1 12:00)
        "fund_residual_load_fc",              # net demand after wind+solar
        "fund_wind_onshore_fc",               # onshore wind generation
        "fund_solar_fc",                      # solar generation
        "fund_net_sched_imports",             # cross-border net position
    ],
    "weather": [
        "weather_temp_mean",                  # heating/cooling driver
        "weather_wind100m_mean",              # bulk wind
        "weather_wind100m_offshore_mean",     # offshore split (DK1 heavy offshore)
        "weather_ssrd_mean",                  # solar irradiance
    ],
}

# Validate every intuition column exists in df. Fail loudly so a typo doesn't
# silently skew "fixed" mode.
_intuition_all = [c for cs in INTUITION_FEATURES.values() for c in cs]
_intuition_missing = [c for c in _intuition_all if c not in df.columns]
if _intuition_missing:
    raise AssertionError(
        f"INTUITION_FEATURES references {len(_intuition_missing)} columns "
        f"not in df: {_intuition_missing}. Fix the typo in the relevant family."
    )
print(f"Intuition features OK: {len(_intuition_all)} cols across "
      f"{len(INTUITION_FEATURES)} families.")


def intuition_features_for_variant(variant_name: str) -> list[str]:
    """Concatenate intuition columns for the families in a given FS variant."""
    families = FEATURE_SET_VARIANTS[variant_name]
    out: list[str] = []
    for fam in families:
        out.extend(INTUITION_FEATURES.get(fam, []))
    # de-duplicate while preserving order
    return list(dict.fromkeys(out))


# ----------------------------------------------------------------------------
# Ablation modes
# ----------------------------------------------------------------------------
ABLATION_MODES: list[str] = ["fixed", "no_penalty", "with_penalty"]

PENALTY_BY_MODE: dict[str, dict[str, float]] = {
    "fixed":        {"group_penalty": 0.0, "feature_penalty": 0.0},  # not used; FS bypassed
    "no_penalty":   {"group_penalty": 0.0, "feature_penalty": 0.0},
    "with_penalty": {"group_penalty": 0.3, "feature_penalty": 0.02},
}

MODE_TAG: dict[str, str] = {
    "fixed":        "fix",
    "no_penalty":   "np",
    "with_penalty": "wp",
}

# Runtime knob: which modes the §10 run loop should execute. Default is the
# full 3-way ablation. Override here for re-runs (e.g. ["with_penalty"]).
ABLATION_MODES_ACTIVE: list[str] = ABLATION_MODES.copy()

print(f"Ablation modes active: {ABLATION_MODES_ACTIVE}")
print(f"  with_penalty: group={PENALTY_BY_MODE['with_penalty']['group_penalty']}, "
      f"feature={PENALTY_BY_MODE['with_penalty']['feature_penalty']}")
print(f"  intuition cols by FS variant:")
for _v in ACTIVE_FS:
    print(f"    {_v}: {len(intuition_features_for_variant(_v))} cols")


### 10c-pre-ckpt. Checkpoint setup

Atomic per-run persistence under `_build/cache/ablation/`:

- `results_df.parquet` + `test_predictions_df.parquet` — atomic write after every run.
- `models/<run>.pkl` — final fitted estimator (for SHAP, residuals).
- `studies/<run>.pkl` — full Optuna study, every trial preserved.
- `val_predictions/<run>.parquet` — fold validation predictions.
- `df_final.parquet` — engineered features (saved once).
- `run_config.json` + `state.json` + `failed_runs.json` — methodology snapshot, manifest, failure ledger.

Resume: re-running this cell rebuilds `completed_run_names` from `results_df.parquet`. The §10 run loop skips any `pred_col` already in that set, so a kernel crash at hour N out of M only loses the in-flight run.

Reset everything: `import shutil; shutil.rmtree(CHECKPOINT_DIR); CHECKPOINT_DIR.mkdir()`.

In [ ]:
# ============================================================
# Checkpoint setup — atomic per-run persistence + resume support
# ============================================================
# Layout under _build/cache/ablation/:
#   results_df.parquet           master results table (atomic write per run)
#   test_predictions_df.parquet  all model prediction columns
#   df_final.parquet             engineered df (saved once)
#   run_config.json              methodology snapshot (penalties, intuition, seeds)
#   state.json                   manifest: started_at, last_completed_run, n_completed
#   failed_runs.json             per-run failures (preserved across restarts)
#   val_predictions/<run>.parquet
#   models/<run>.pkl             final fitted estimator
#   studies/<run>.pkl            full Optuna study object (all trials, not just best)
#
# Atomic writes: every parquet/json/pickle goes through `.tmp` + rename so a
# crash mid-write cannot corrupt the canonical file.
#
# Reset everything: run `import shutil; shutil.rmtree(CHECKPOINT_DIR)` and
# re-execute this cell.

from pathlib import Path
import json as _json
import pickle
import shutil
import subprocess
from datetime import datetime, timezone

CHECKPOINT_DIR      = PACKAGE_ROOT / "_build/cache/ablation"
RESULTS_PARQUET     = CHECKPOINT_DIR / "results_df.parquet"
TEST_PREDS_PARQUET  = CHECKPOINT_DIR / "test_predictions_df.parquet"
DF_FINAL_PARQUET    = CHECKPOINT_DIR / "df_final.parquet"
RUN_CONFIG_JSON     = CHECKPOINT_DIR / "run_config.json"
STATE_JSON          = CHECKPOINT_DIR / "state.json"
FAILED_RUNS_JSON    = CHECKPOINT_DIR / "failed_runs.json"
VAL_PREDS_DIR       = CHECKPOINT_DIR / "val_predictions"
MODELS_DIR          = CHECKPOINT_DIR / "models"
STUDIES_DIR         = CHECKPOINT_DIR / "studies"

for _d in (CHECKPOINT_DIR, VAL_PREDS_DIR, MODELS_DIR, STUDIES_DIR):
    _d.mkdir(parents=True, exist_ok=True)


# ---------- Atomic-write helpers ----------------------------------------
def _atomic_write_parquet(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp)
    tmp.replace(path)


def _atomic_write_json(obj, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        _json.dump(obj, f, indent=2, default=str)
    tmp.replace(path)


def _atomic_write_pickle(obj, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
    tmp.replace(path)


# ---------- Configuration snapshot --------------------------------------
def _git_sha() -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return "unknown"


run_config: dict = {
    "started_at":            datetime.now(timezone.utc).isoformat(),
    "code_version":          _git_sha(),
    "N_TRIALS":              int(N_TRIALS),
    "SEED":                  int(SEED),
    "TARGET_COL":            TARGET_COL,
    "TARGET_ZONE":           TARGET_ZONE,
    "PERIOD_START":          str(PERIOD_START),
    "PERIOD_END":            str(PERIOD_END),
    "ACTIVE_FS":             list(ACTIVE_FS),
    "ABLATION_MODES_ACTIVE": list(ABLATION_MODES_ACTIVE),
    "PENALTY_BY_MODE":       PENALTY_BY_MODE,
    "MODE_TAG":              MODE_TAG,
    "INTUITION_FEATURES":    INTUITION_FEATURES,
    "FEATURE_SET_VARIANTS":  FEATURE_SET_VARIANTS,
    "MODEL_FAMILIES":        {fam: [s.name for s in specs]
                              for fam, specs in MODEL_FAMILIES.items()},
    "INCLUDE_TABNET":        bool(INCLUDE_TABNET),
    "GPU_AVAILABLE":         bool(GPU_AVAILABLE),
    "INCLUDE_NEURAL_GPU":    bool(INCLUDE_NEURAL_GPU),
}

# Warn if a critical config field changed since the last persisted run, so
# you don't accidentally mix runs from incompatible configurations.
if RUN_CONFIG_JSON.exists() and RESULTS_PARQUET.exists():
    try:
        _prev = _json.loads(RUN_CONFIG_JSON.read_text())
        _critical = ["N_TRIALS", "SEED", "PENALTY_BY_MODE", "INTUITION_FEATURES",
                     "FEATURE_SET_VARIANTS", "MODEL_FAMILIES", "TARGET_COL"]
        _diffs = [k for k in _critical if _prev.get(k) != run_config.get(k)]
        if _diffs:
            print("=" * 70)
            print(f"WARNING: critical config changed since last run: {_diffs}")
            print( "  Persisted ablation runs may be inconsistent with the new config.")
            print( "  To start fresh:")
            print( "    import shutil; shutil.rmtree(CHECKPOINT_DIR); CHECKPOINT_DIR.mkdir()")
            print("=" * 70)
    except Exception:
        pass

_atomic_write_json(run_config, RUN_CONFIG_JSON)
print(f"run_config.json written ({len(run_config)} keys, "
      f"code_version={run_config['code_version']})")

# Snapshot the engineered df once. Cheap if it already exists with the same
# shape — letting a second notebook (or this notebook after a kernel restart
# without re-running cells 1-7) read the engineered features directly.
_should_write_df = (
    not DF_FINAL_PARQUET.exists()
    or pd.read_parquet(DF_FINAL_PARQUET).shape != df.shape
)
if _should_write_df:
    _atomic_write_parquet(df, DF_FINAL_PARQUET)
    print(f"df_final.parquet written: shape={df.shape}, "
          f"size={DF_FINAL_PARQUET.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print(f"df_final.parquet already up-to-date (shape={df.shape}).")


# ---------- Resume: rebuild completed_run_names from results parquet ----
completed_run_names: set[str] = set()
if RESULTS_PARQUET.exists():
    _existing = pd.read_parquet(RESULTS_PARQUET)
    if "ablation_mode" in _existing.columns and "run_name" in _existing.columns:
        # 'completed' = ablation rows on disk. Baselines (NaN ablation_mode) are
        # not 'completed' — they are rebuilt by cells 45/47/49 each session.
        _ablation_rows = _existing[_existing["ablation_mode"].notna()]
        completed_run_names = set(_ablation_rows["run_name"].astype(str))
        # Rehydrate the in-memory `results` list. Baselines were appended to
        # `results` by cells 45-49 already; we now extend with cached ablation
        # rows so the in-memory state matches the on-disk state.
        for _row in _ablation_rows.to_dict("records"):
            results.append(_row)

# Rehydrate test_predictions_df with persisted prediction columns (idempotent
# — if a column is already in the in-memory df we leave it alone).
if TEST_PREDS_PARQUET.exists():
    _existing_preds = pd.read_parquet(TEST_PREDS_PARQUET)
    for _col in completed_run_names:
        if _col in _existing_preds.columns and _col not in test_predictions_df.columns:
            test_predictions_df[_col] = _existing_preds[_col]

# Failed-runs ledger persists across restarts (so we can see what blew up
# in a prior session).
failed_runs: list[dict] = []
if FAILED_RUNS_JSON.exists():
    try:
        failed_runs = _json.loads(FAILED_RUNS_JSON.read_text())
    except Exception:
        failed_runs = []

print(f"Resume: {len(completed_run_names)} ablation runs already cached.")
if failed_runs:
    print(f"  ({len(failed_runs)} previously-failed runs in failed_runs.json — "
          f"will be retried.)")


# ---------- Per-run persistence helpers ---------------------------------
def _persist_run(res: dict) -> None:
    """Write all artifacts of a successful run to disk atomically.

    Called from the run loop after each `run_grouped_model_study` succeeds.
    Order matters: pickle/parquet first (large, slow), then state.json
    (small, fast) — so the state file always reflects work that's already
    fully on disk.
    """
    rn = res["run_name"]
    _atomic_write_parquet(res["val_predictions_df"], VAL_PREDS_DIR / f"{rn}.parquet")
    _atomic_write_pickle(res["final_model"],          MODELS_DIR    / f"{rn}.pkl")
    _atomic_write_pickle(res["study"],                STUDIES_DIR   / f"{rn}.pkl")
    _atomic_write_parquet(pd.DataFrame(results),      RESULTS_PARQUET)
    _atomic_write_parquet(test_predictions_df,        TEST_PREDS_PARQUET)
    _atomic_write_json({
        "last_completed_run": rn,
        "n_completed":        len(completed_run_names) + 1,
        "n_failed":           len(failed_runs),
        "updated_at":         datetime.now(timezone.utc).isoformat(),
    }, STATE_JSON)


def _persist_aggregate_only() -> None:
    """Write only results_df + test_predictions_df. Used by §10c TabNet
    (where we don't have a comparable per-run model.pkl / study.pkl)."""
    _atomic_write_parquet(pd.DataFrame(results), RESULTS_PARQUET)
    _atomic_write_parquet(test_predictions_df,   TEST_PREDS_PARQUET)
    _atomic_write_json({
        "last_completed_run": (results[-1].get("run_name") if results else "n/a"),
        "n_completed":        len(completed_run_names),
        "n_failed":           len(failed_runs),
        "updated_at":         datetime.now(timezone.utc).isoformat(),
    }, STATE_JSON)


def _record_failure(pred_col: str, mode: str, exc: BaseException) -> None:
    """Append a failure entry to failed_runs.json."""
    import traceback as _tb
    failed_runs.append({
        "run_name":      pred_col,
        "ablation_mode": mode,
        "error":         f"{type(exc).__name__}: {exc}",
        "traceback":     _tb.format_exc(),
        "failed_at":     datetime.now(timezone.utc).isoformat(),
    })
    _atomic_write_json(failed_runs, FAILED_RUNS_JSON)


In [ ]:
# Active sweep config — override here for smoke runs.
# `N_TRIALS_RUN` and `ACTIVE_FS` are defined upstream in the run-mode cell.
N_TRIALS_RUN = N_TRIALS

# Live-progress log: per-run lines written here as soon as a study starts and
# again when it finishes. Lets us `tail -f` from outside the kernel even though
# nbconvert buffers cell stdout. Path is fixed so monitors know where to look.
import time as _time_mod
PROGRESS_LOG = str(PACKAGE_ROOT / "_build/cache/ablation-progress.log")

def _progress(line: str) -> None:
    """Append a timestamped line to the progress log AND flush stdout."""
    ts = _time_mod.strftime("%Y-%m-%d %H:%M:%S")
    msg = f"[{ts}] {line}"
    print(msg, flush=True)
    try:
        with open(PROGRESS_LOG, "a") as f:
            f.write(msg + "\n")
    except Exception:
        pass

# Reset progress log at start of each notebook run.
try:
    with open(PROGRESS_LOG, "w") as _f:
        _f.write(f"=== ablation run-loop started at {_time_mod.strftime('%Y-%m-%d %H:%M:%S')} ===\n")
except Exception:
    pass

# In-memory caches for this session. Persisted artifacts on disk are the
# canonical source — these dicts are populated (a) by per-run completions in
# this session and (b) by the post-loop reconciliation cell after the loop
# (or on a kernel restart, when we reload from checkpoint).
trained_models: dict[str, object] = {}
val_predictions: dict[str, pd.DataFrame] = {}

n_total_runs = (
    len(ABLATION_MODES_ACTIVE)
    * sum(len(specs) for specs in MODEL_FAMILIES.values())
    * len(ACTIVE_FS)
)
_progress(f"plan: {len(ABLATION_MODES_ACTIVE)} modes × "
          f"{sum(len(s) for s in MODEL_FAMILIES.values())} specs × "
          f"{len(ACTIVE_FS)} variants = {n_total_runs} runs at N_TRIALS={N_TRIALS_RUN}")
_progress(f"resume: {len(completed_run_names)} runs already cached, "
          f"{n_total_runs - len(completed_run_names)} to do")

_run_idx = 0
for mode in ABLATION_MODES_ACTIVE:
    mode_tag = MODE_TAG[mode]
    pen = PENALTY_BY_MODE[mode]
    _progress(f"=== ablation mode: {mode!r} (tag={mode_tag!r}) ===")

    for family_name, specs in MODEL_FAMILIES.items():
        for spec in specs:
            # In "fixed" mode penalties are irrelevant (FS is bypassed); otherwise
            # stamp them onto a copy of the spec so run_grouped_model_study sees
            # the right values.
            if mode == "fixed":
                spec_for_run = spec
            else:
                spec_for_run = _replace_dc(
                    spec,
                    group_penalty=pen["group_penalty"],
                    feature_penalty=pen["feature_penalty"],
                )

            for variant in ACTIVE_FS:
                _run_idx += 1
                pred_col = f"{spec.name}_{short_variant_name(variant)}_{mode_tag}"

                # Skip if this run is already on disk from a prior session.
                if pred_col in completed_run_names:
                    _progress(f"[{_run_idx}/{n_total_runs}] skip {pred_col} (cached)")
                    continue

                fixed_feats = (intuition_features_for_variant(variant)
                               if mode == "fixed" else None)

                _t_run = _time_mod.monotonic()
                _progress(f"[{_run_idx}/{n_total_runs}] start {pred_col} "
                          f"(family={spec.family}, mode={mode})")

                # A single buggy spec must not kill the rest of the campaign.
                # On exception, log the failure and move on.
                try:
                    res = run_grouped_model_study(
                        exp=exp,
                        model_spec=spec_for_run,
                        variant_name=variant,
                        feature_set_variants=FEATURE_SET_VARIANTS,
                        feature_groups_by_family=FEATURE_GROUPS_BY_FAMILY,
                        test_predictions_df=test_predictions_df,
                        n_trials=N_TRIALS_RUN,
                        seed=SEED,
                        show_progress_bar=False,
                        fixed_features=fixed_feats,
                        pred_col_override=pred_col,
                        ablation_mode=mode,
                    )
                except BaseException as _exc:
                    _record_failure(pred_col, mode, _exc)
                    _progress(f"[{_run_idx}/{n_total_runs}] FAILED {pred_col}: "
                              f"{type(_exc).__name__}: {_exc}")
                    continue

                test_predictions_df = res["test_predictions_df"]
                trained_models[res["run_name"]] = res["final_model"]
                val_predictions[res["run_name"]] = res["val_predictions_df"]
                results.append(res["summary_row"])

                # Persist immediately so a later crash cannot lose this run.
                _persist_run(res)
                completed_run_names.add(pred_col)

                _elapsed = _time_mod.monotonic() - _t_run
                sr = res["summary_row"]
                _progress(f"[{_run_idx}/{n_total_runs}] done  {pred_col} "
                          f"test_mae={sr['test_mae']:.4f} "
                          f"({_elapsed:.1f}s wall)")

# Final dedupe + sort. drop_duplicates(keep='last') wins on accidental double-
# appends from resume + re-run, taking the freshest values.
results_df = (pd.DataFrame(results)
                .drop_duplicates(subset=["run_name"], keep="last")
                .sort_values(["family", "ablation_mode", "test_mae"], na_position="last")
                .reset_index(drop=True))
_atomic_write_parquet(results_df, RESULTS_PARQUET)

_progress(f"=== all {len(ABLATION_MODES_ACTIVE)} modes complete: "
          f"results_df has {len(results_df)} rows, "
          f"{len(failed_runs)} failed ===")

print("\n" + "=" * 80)
print(f"All {len(ABLATION_MODES_ACTIVE)} ablation modes complete. "
      f"results_df has {len(results_df)} rows.")
if failed_runs:
    print(f"  {len(failed_runs)} run(s) FAILED — see {FAILED_RUNS_JSON}")
print("=" * 80)
print(results_df[["run_name", "family", "feature_variant", "ablation_mode",
                   "mean_fold_val_mae", "test_mae", "test_rmse",
                   "n_selected_groups", "n_selected_features",
                   "compute_device", "n_trials", "train_total_seconds"]].to_string(index=False))


## 10c. Neural models on GPU via RunPod Flash

TabNet runs **GPU-only** — CPU training is prohibitively slow (>1 h per trial vs. ~30 s on an RTX 4090). When `INCLUDE_NEURAL_GPU=True` (set by the gate in §10's run-mode cell), this section:

1. **Uploads** the dev/test feature matrices once per FS variant to Hetzner S3 under `signalforge-ml/runpod-scratch/tabnet/<run_id>/`.
2. **Drives** an Optuna TPE study where each trial calls a RunPod RTX 4090 worker. The worker runs all 5 walk-forward folds in a single round-trip and returns mean MAE — `idle_timeout=300 s` keeps the worker warm between trials, giving ~30 s/trial after cold start.
3. **Refits** with the best params on full dev, downloads test predictions, and merges them into `test_predictions_df` and `results_df` so they appear in the §11 figures and §11b rMAE table alongside CPU runs.

The `@Endpoint` worker definitions live in `_build/runpod_tabnet_optuna.py` (kept out of the notebook to avoid 470 lines of decorators and worker bodies). This cell is the call-site only.

If `GPU_AVAILABLE=False` or `INCLUDE_TABNET=False`, this cell is a no-op.


In [ ]:
# §10c — TabNet GPU runs via RunPod Flash.
# No-op if GPU is unavailable or INCLUDE_TABNET is False (no CPU fallback).
# TabNet runs once under "no_penalty" (tag = "np") so it slots into the same
# results_df schema as the CPU ablation sweep.
#
# Persistence: TabNet only persists results_df + test_predictions_df (via
# _persist_aggregate_only). The TabNet final model lives on RunPod scratch
# S3 and is not standardised for local re-use; SHAP runs against tree models
# only. Skip-on-cached uses the same completed_run_names set as §10.
if INCLUDE_NEURAL_GPU:
    import sys as _sys
    _BUILD = str(PACKAGE_ROOT / "_build")
    if _BUILD not in _sys.path:
        _sys.path.insert(0, _BUILD)
    from runpod_tabnet_optuna import run_tabnet_gpu_study  # noqa: E402

    n_trials_neural = N_TRIALS  # follows smoke override
    _tabnet_mode = "no_penalty"
    _tabnet_tag = MODE_TAG[_tabnet_mode]

    for variant in ACTIVE_FS:
        family_keys = FEATURE_SET_VARIANTS[variant]
        variant_groups: dict[str, list[str]] = {}
        for fam in family_keys:
            for grp_name, cols in FEATURE_GROUPS_BY_FAMILY[fam].items():
                variant_groups[grp_name] = cols

        pred_col = f"tabnet_{short_variant_name(variant)}_{_tabnet_tag}"

        # Skip if cached. completed_run_names was populated from
        # results_df.parquet by the checkpoint-setup cell.
        if pred_col in completed_run_names:
            print(f"[tabnet-gpu] {variant}: skip {pred_col} (cached)")
            continue

        print(f"\n[tabnet-gpu] {variant} ({_tabnet_mode}): {len(variant_groups)} feature groups, "
              f"n_trials={n_trials_neural}")

        try:
            gpu_res = run_tabnet_gpu_study(
                exp=exp,
                feature_groups=variant_groups,
                suggest_params_fn=suggest_tabnet_params,
                suggest_groups_fn=suggest_feature_groups,
                n_trials=n_trials_neural,
                seed=SEED,
                variant_name=variant,
                test_predictions_df=test_predictions_df,
                pred_col_name=pred_col,
                cv_max_epochs=50, cv_patience=8,
                final_max_epochs=200, final_patience=20,
            )
        except BaseException as _exc:
            _record_failure(pred_col, _tabnet_mode, _exc)
            print(f"[tabnet-gpu] {variant}: FAILED {pred_col}: "
                  f"{type(_exc).__name__}: {_exc}")
            continue

        # Adapt the GPU summary_row to the CPU-sweep schema so it merges into
        # results_df cleanly. Compute test_rmse from the persisted predictions.
        sr = gpu_res["summary_row"]
        ya = test_predictions_df["actual"]
        yp = test_predictions_df[pred_col]
        mask = ya.notna() & yp.notna()
        test_rmse = float(np.sqrt(np.mean((ya[mask] - yp[mask]) ** 2)))

        results.append({
            "run_name":            pred_col,
            "family":              "neural",
            "feature_variant":     variant,
            "ablation_mode":       _tabnet_mode,
            "mean_fold_val_mae":   sr["best_cv_mae"],
            "test_mae":            sr["test_mae"],
            "test_rmse":           test_rmse,
            "n_selected_groups":   sr["n_groups"],
            "n_selected_features": sr["n_features"],
            "compute_device":      sr["compute_device"],
            "hardware":            sr["hardware"],
            "n_trials":            sr["n_trials"],
            "train_total_seconds": sr["train_total_seconds"],
            "study_wall_seconds":  sr["study_wall_seconds"],
            "per_trial_seconds":   sr["per_trial_seconds"],
            "final_fit_seconds":   sr["final_fit_seconds"],
            "code_version":        sr.get("code_version", ""),
        })
        completed_run_names.add(pred_col)
        _persist_aggregate_only()

        print(f"[tabnet-gpu] {variant} test_mae={sr['test_mae']:.4f}  "
              f"test_rmse={test_rmse:.4f}  best_cv_mae={sr['best_cv_mae']:.4f}")

    # Refresh results_df with TabNet rows merged in.
    results_df = (pd.DataFrame(results)
                    .drop_duplicates(subset=["run_name"], keep="last")
                    .sort_values(["family", "ablation_mode", "test_mae"],
                                 na_position="last")
                    .reset_index(drop=True))
    _atomic_write_parquet(results_df, RESULTS_PARQUET)
    print(f"\n[tabnet-gpu] all variants done. "
          f"results_df now has {len(results_df)} rows.")
elif INCLUDE_TABNET:
    print("[tabnet-gpu] SKIPPED — INCLUDE_TABNET=True but GPU unavailable. "
          "TabNet does not run on CPU.")
else:
    print("[tabnet-gpu] disabled (INCLUDE_TABNET=False).")


### 10c-post. Post-loop reconciliation

After a kernel restart, `trained_models` and `val_predictions` start empty even though every completed run is still on disk. This cell lazy-loads any missing artifacts so §11 figures and §13 SHAP work without re-running the loop.

If you ran the loop in this same kernel session, this cell is a near-no-op.

In [ ]:
# ============================================================
# Post-loop reconciliation
# ============================================================
# Lazy-load any persisted artifacts that aren't already in this session's
# in-memory dicts. Required for §11/§13 to work cleanly after a kernel
# restart, where `trained_models` and `val_predictions` start empty even
# though `results_df` and the on-disk pickles still hold every completed run.
#
# This is a no-op if the loop just completed in this session — every entry
# is already in memory.

_loaded_models = 0
_loaded_val    = 0
_skipped       = 0

for _run_name in sorted(completed_run_names):
    # TabNet runs don't have a per-run model.pkl (model lives on RunPod
    # scratch S3). Skip those silently.
    _is_neural = _run_name.startswith("tabnet_")

    if not _is_neural and _run_name not in trained_models:
        _model_path = MODELS_DIR / f"{_run_name}.pkl"
        if _model_path.exists():
            with open(_model_path, "rb") as _f:
                trained_models[_run_name] = pickle.load(_f)
            _loaded_models += 1
        else:
            _skipped += 1

    if _run_name not in val_predictions:
        _val_path = VAL_PREDS_DIR / f"{_run_name}.parquet"
        if _val_path.exists():
            val_predictions[_run_name] = pd.read_parquet(_val_path)
            _loaded_val += 1

print(f"reconciliation: loaded {_loaded_models} models, "
      f"{_loaded_val} val_predictions from disk.")
if _skipped:
    print(f"  ({_skipped} run(s) had no model.pkl on disk — likely TabNet/neural.)")
print(f"  trained_models entries:  {len(trained_models)}")
print(f"  val_predictions entries: {len(val_predictions)}")

_n_ablation = sum(1 for r in results if r.get("ablation_mode"))
_n_baselines = len(results) - _n_ablation
print(f"  results_df rows:         {len(results_df)} "
      f"({_n_ablation} ablation, {_n_baselines} baselines/lear)")


In [ ]:
# Heatmap of test MAE — ML model × FS variant + baseline floor rows.
# Baselines (persistence_naive_*, climatology_how, lear) are FS-agnostic; we
# replicate each baseline's test_mae across the 4 FS columns and separate the
# baseline block from ML rows with a horizontal divider.
ml_mask = results_df["feature_variant"].isin(ACTIVE_FS)
ml_rows = results_df[ml_mask]
nonml_rows = results_df[~ml_mask]

ml_pivot = ml_rows.pivot_table(index="model", columns="feature_variant",
                                 values="test_mae", aggfunc="first")
ml_pivot = ml_pivot.reindex(columns=ACTIVE_FS)
# Sort ML rows by mean MAE across FS variants (best to worst).
ml_pivot = (
    ml_pivot.assign(_mean=ml_pivot.mean(axis=1))
            .sort_values("_mean")
            .drop(columns="_mean")
)

# Baseline block: replicate test_mae across all FS columns.
baseline_pivot = pd.DataFrame(
    index=nonml_rows["model"].values, columns=ACTIVE_FS, dtype=float,
)
for _, row in nonml_rows.iterrows():
    baseline_pivot.loc[row["model"], :] = row["test_mae"]
baseline_pivot = baseline_pivot.sort_values(ACTIVE_FS[0])

heatmap = pd.concat([ml_pivot, baseline_pivot], axis=0)
print(heatmap.round(3))

vmin = float(np.nanmin(heatmap.values))
vmax = float(np.nanmax(heatmap.values))

fig, ax = plt.subplots(figsize=(8, max(4, 0.55 * len(heatmap.index))))
im = ax.imshow(heatmap.values, aspect="auto", cmap="RdYlGn_r", vmin=vmin, vmax=vmax)
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns)
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
for i in range(heatmap.shape[0]):
    for j in range(heatmap.shape[1]):
        v = heatmap.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="black", fontsize=8)
# Divider between ML block (top) and baseline block (bottom).
ax.axhline(y=len(ml_pivot) - 0.5, color="black", linewidth=1.5)
fig.colorbar(im, ax=ax, label="test MAE (EUR/MWh)")
ax.set_title("Test MAE — ML model × feature set, with baseline floor")
plt.tight_layout()
plt.show()


## 11. Actual vs predicted on the test period

Plot the held-out test period in EUR/MWh: actual DK1 day-ahead price overlaid with the best ML model per feature set, plus persistence and LEAR for context. The bottom panel zooms into a one-week slice so the daily wave-shape comparison is legible.


In [ ]:
# §11 — Actual vs predicted, one figure per run.
# Each figure has 2x2 panels: (test full, test zoom, val full, val zoom).
# Renders every ML run + every baseline + LEAR.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def _zoom_window(idx: pd.DatetimeIndex, *, frac: float = 1/3,
                 days: int = 7) -> tuple[pd.Timestamp, pd.Timestamp]:
    start = idx.min() + (idx.max() - idx.min()) * frac
    start = pd.Timestamp(start).normalize()
    return start, start + pd.Timedelta(days=days)


def _plot_actual_pred_panel(ax, df: pd.DataFrame, actual_col: str, pred_col: str,
                             title: str, *, zoom: tuple | None = None) -> None:
    if zoom is not None:
        df = df.loc[zoom[0]:zoom[1]]
    if df.empty:
        ax.set_title(f"{title} (no data)")
        ax.set_axis_off()
        return
    ax.plot(df.index, df[actual_col], label="actual",
            color="black", linewidth=1.5, alpha=0.9)
    ax.plot(df.index, df[pred_col], label=pred_col,
            color="C1", linewidth=1.0, alpha=0.85)
    ax.set_ylabel("EUR/MWh")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)
    if zoom is not None:
        ax.xaxis.set_major_locator(mdates.DayLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d-%b"))
    else:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(
            mdates.ConciseDateFormatter(ax.xaxis.get_major_locator())
        )


def plot_run_4panel(run_name: str) -> None:
    """Render a 2x2 figure for one run: test full, test zoom, val full, val zoom."""
    test_df = pd.DataFrame()
    if run_name in test_predictions_df.columns:
        test_df = test_predictions_df[["actual", run_name]].dropna()

    val_df = pd.DataFrame()
    if run_name in val_predictions:
        vp = val_predictions[run_name]
        if "actual" in vp.columns and run_name in vp.columns:
            val_df = vp[["actual", run_name]].dropna()
    elif run_name in baseline_val_predictions:
        s = baseline_val_predictions[run_name].dropna()
        actual_aligned = exp["final"]["y_train"].reindex(s.index)
        val_df = pd.DataFrame({"actual": actual_aligned, run_name: s}).dropna()

    if test_df.empty and val_df.empty:
        print(f"[skip] {run_name}: no test or val predictions available")
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharey=False)

    # Test full / zoom
    if not test_df.empty:
        _plot_actual_pred_panel(axes[0, 0], test_df, "actual", run_name,
                                 title=f"{run_name} — test (full)")
        zw = _zoom_window(test_df.index)
        _plot_actual_pred_panel(axes[0, 1], test_df, "actual", run_name,
                                 title=f"{run_name} — test ({zw[0]:%Y-%m-%d}–{zw[1]:%Y-%m-%d})",
                                 zoom=zw)
    else:
        axes[0, 0].set_axis_off(); axes[0, 1].set_axis_off()

    # Val full / zoom
    if not val_df.empty:
        _plot_actual_pred_panel(axes[1, 0], val_df, "actual", run_name,
                                 title=f"{run_name} — val (combined folds)")
        zw = _zoom_window(val_df.index)
        _plot_actual_pred_panel(axes[1, 1], val_df, "actual", run_name,
                                 title=f"{run_name} — val ({zw[0]:%Y-%m-%d}–{zw[1]:%Y-%m-%d})",
                                 zoom=zw)
    else:
        axes[1, 0].set_axis_off(); axes[1, 1].set_axis_off()

    plt.tight_layout()
    plt.show()


# Render every figure: ML runs (sorted by test_mae) + baselines + LEAR.
ml_runs_sorted = (
    results_df[results_df["feature_variant"].isin(ACTIVE_FS)]
    .sort_values("test_mae")["run_name"].tolist()
)
baseline_runs = list(baseline_val_predictions.keys())  # includes LEAR

print(f"Rendering {len(ml_runs_sorted)} ML figures + {len(baseline_runs)} "
      f"baseline figures = {len(ml_runs_sorted) + len(baseline_runs)} figures total")
for run in ml_runs_sorted:
    plot_run_4panel(run)
for run in baseline_runs:
    plot_run_4panel(run)
print("Done.")


## 11b. rMAE — every model vs every baseline

Relative MAE: `rMAE_b = test_mae(row) / test_mae(b)`. Values < 1 mean the row beats baseline `b` on the held-out test set. Five reference rows: the three persistence baselines (`naive_d1`, `naive_w1`, `epf_hybrid`), the hour-of-week climatology, and LEAR (the canonical EPF benchmark). Sorted by absolute test MAE.


In [ ]:
# §11b — rMAE table: every row vs each baseline / benchmark.
# rMAE_b = test_mae(row) / test_mae(b). < 1 means the row beats baseline b.
BASELINES_FOR_RMAE = [
    "price_baseline_naive_d1",
    "price_baseline_naive_w1",
    "price_baseline_naive_epf_hybrid",
    "climatology_how",
    "lear",
]
SHORT_RMAE_NAMES = {
    "price_baseline_naive_d1":          "rMAE_naive_d1",
    "price_baseline_naive_w1":          "rMAE_naive_w1",
    "price_baseline_naive_epf_hybrid":  "rMAE_epf_hybrid",
    "climatology_how":                  "rMAE_climatology",
    "lear":                             "rMAE_lear",
}

baseline_mae_lookup: dict[str, float] = {}
for b in BASELINES_FOR_RMAE:
    sub = results_df[results_df["run_name"] == b]
    if not sub.empty:
        baseline_mae_lookup[b] = float(sub["test_mae"].iloc[0])

print("Baseline test MAEs:")
for b, m in baseline_mae_lookup.items():
    print(f"  {b:36s}  {m:8.3f}")

rmae_rows: list[dict] = []
for _, row in results_df.iterrows():
    out = {
        "run_name": row["run_name"],
        "family": row["family"],
        "feature_variant": row["feature_variant"],
        "test_mae": float(row["test_mae"]),
    }
    for b, mae_b in baseline_mae_lookup.items():
        out[SHORT_RMAE_NAMES[b]] = float(row["test_mae"]) / mae_b
    rmae_rows.append(out)

rmae_df = pd.DataFrame(rmae_rows).sort_values("test_mae").reset_index(drop=True)

print(f"\nrMAE table ({len(rmae_df)} rows):")
print(rmae_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Persist for downstream consumption.
RMAE_PARQUET = PACKAGE_ROOT / "_build/cache/da_experiments_rmae.parquet"
RMAE_PARQUET.parent.mkdir(parents=True, exist_ok=True)
rmae_df.to_parquet(RMAE_PARQUET)
print(f"\nWrote {RMAE_PARQUET}")

# Color-styled table for visual scan in Jupyter.
try:
    rmae_cols = [SHORT_RMAE_NAMES[b] for b in baseline_mae_lookup]
    styled = (
        rmae_df.style
               .background_gradient(subset=rmae_cols, cmap="RdYlGn_r",
                                     vmin=0.5, vmax=1.5)
               .format(precision=3)
    )
    display(styled)
except Exception as e:
    print(f"(Styler unavailable: {e})")


## 11c. Ablation comparison

Compare test MAE across the three ablation modes for every (model, FS variant) pair. Two views:

- Δ vs. **fixed** — does Optuna feature selection beat a hand-picked intuition set?
- Δ vs. **no_penalty** — does the FS regularizer (group_penalty=0.3, feature_penalty=0.02) actually help test MAE, or just shrink the selected feature count?

In [ ]:
# §11c — Ablation comparison: test MAE by (model, FS variant) × ablation mode.
#
# Three pivot views:
#   1) raw test MAE per cell
#   2) Δ vs. fixed-features baseline (negative = Optuna search beats intuition)
#   3) Δ vs. no_penalty (negative = penalty improves test MAE)
# A heatmap of (1) is rendered for the full results_df.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Build the ablation-eligible subset. Naive/climatology/LEAR baselines have no
# ablation_mode, and TabNet only ran under "no_penalty" — exclude both from
# the head-to-head pivots so each row has all three modes for fair Δ math.
ABLATION_FAMILIES_FOR_TABLE = {"tree", "linear"}
abl_df = results_df[
    results_df["family"].isin(ABLATION_FAMILIES_FOR_TABLE)
    & results_df["ablation_mode"].isin(ABLATION_MODES)
].copy()

if abl_df.empty:
    print("[11c] No ablation rows in results_df — did the §10 run loop execute?")
else:
    # Helper for grouped index
    abl_df["model_fs"] = abl_df["model"] + "_" + abl_df["feature_variant"]

    pivot_mae = abl_df.pivot_table(
        index=["family", "model", "feature_variant"],
        columns="ablation_mode",
        values="test_mae",
        aggfunc="first",
    )[ABLATION_MODES]   # consistent column order
    pivot_grp = abl_df.pivot_table(
        index=["family", "model", "feature_variant"],
        columns="ablation_mode",
        values="n_selected_groups",
        aggfunc="first",
    )[ABLATION_MODES]
    pivot_feat = abl_df.pivot_table(
        index=["family", "model", "feature_variant"],
        columns="ablation_mode",
        values="n_selected_features",
        aggfunc="first",
    )[ABLATION_MODES]

    # Δ vs. fixed (negative = Optuna FS beat the intuition set)
    pivot_delta_vs_fixed = pivot_mae.sub(pivot_mae["fixed"], axis=0)
    # Δ vs. no_penalty (negative = penalty improved test MAE)
    pivot_delta_vs_np = pivot_mae.sub(pivot_mae["no_penalty"], axis=0)

    print("=" * 80)
    print("§11c — Test MAE by ablation mode (EUR/MWh)")
    print("=" * 80)
    print(pivot_mae.round(3).to_string())

    print("\n" + "=" * 80)
    print("Δ test MAE vs. 'fixed' (intuition feature set)")
    print("  negative = Optuna search beats intuition")
    print("=" * 80)
    print(pivot_delta_vs_fixed.round(3).to_string())

    print("\n" + "=" * 80)
    print("Δ test MAE vs. 'no_penalty' (Optuna without FS regularization)")
    print("  negative = adding penalty improved test MAE")
    print("=" * 80)
    print(pivot_delta_vs_np.round(3).to_string())

    print("\n" + "=" * 80)
    print("Selected groups by ablation mode")
    print("=" * 80)
    print(pivot_grp.round(0).astype("Int64").to_string())

    print("\n" + "=" * 80)
    print("Selected features by ablation mode")
    print("=" * 80)
    print(pivot_feat.round(0).astype("Int64").to_string())

    # Heatmap of test MAE per cell.
    _M = pivot_mae.values
    _idx_labels = [f"{m}·{v}" for (_, m, v) in pivot_mae.index]
    fig, ax = plt.subplots(figsize=(7, max(4, 0.35 * len(_idx_labels))))
    im = ax.imshow(_M, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(ABLATION_MODES)))
    ax.set_xticklabels(ABLATION_MODES)
    ax.set_yticks(range(len(_idx_labels)))
    ax.set_yticklabels(_idx_labels, fontsize=8)
    for i in range(_M.shape[0]):
        for j in range(_M.shape[1]):
            v = _M[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        color="white" if v > np.nanmedian(_M) else "black",
                        fontsize=7)
    ax.set_title("§11c · Test MAE by ablation mode (EUR/MWh)")
    fig.colorbar(im, ax=ax, label="test MAE")
    fig.tight_layout()
    plt.show()

    # Aggregate summary: mean Δ across (model, FS) for each mode pair.
    print("\n" + "=" * 80)
    print("Aggregate effect (mean Δ test MAE across all rows)")
    print("=" * 80)
    print(f"  no_penalty  − fixed:      {pivot_delta_vs_fixed['no_penalty'].mean():+.3f} EUR/MWh")
    print(f"  with_penalty − fixed:     {pivot_delta_vs_fixed['with_penalty'].mean():+.3f} EUR/MWh")
    print(f"  with_penalty − no_penalty: {pivot_delta_vs_np['with_penalty'].mean():+.3f} EUR/MWh")
    print("(negative = the latter mode beat the former on average)")


## 12. Feature-group selection frequency

Aggregate `selected_groups` across model runs within each feature-set variant. Cells in the heatmap show the fraction of runs in that FS that picked the group. With many trials the picks should converge toward the genuinely informative groups; in a smoke (`N_TRIALS=2`) the picks are noisy and the chart serves as scaffolding.


In [ ]:
# §12 — Feature-group selection frequency by FS.
# For each feature_variant, count how often each candidate group was picked
# across model runs. Normalize by run count to get a fraction in [0, 1].
# With many trials this should converge to the genuinely informative groups;
# at smoke trial counts the picks are noisy and the chart serves as scaffolding.
import matplotlib.pyplot as plt
from collections import Counter

_freq_rows: list[dict] = []
for fs in ACTIVE_FS:
    sub = results_df[results_df["feature_variant"] == fs]
    sub = sub[sub["selected_groups"].apply(lambda x: isinstance(x, list))]
    if sub.empty:
        continue
    n_runs = len(sub)
    counter: Counter[str] = Counter()
    for groups in sub["selected_groups"]:
        for g in groups:
            counter[g] += 1
    for group, n in counter.items():
        _freq_rows.append({
            "feature_variant": fs, "group": group,
            "freq": n / n_runs, "n_runs": n_runs, "n_picked": n,
        })

freq_df = pd.DataFrame(_freq_rows)
print(f"Feature-group selection rows: {len(freq_df)}")

if freq_df.empty:
    print("No selected_groups recorded yet — skipping plot.")
else:
    pivot = freq_df.pivot_table(
        index="group", columns="feature_variant", values="freq", aggfunc="first",
    )
    pivot = pivot.reindex(columns=[fs for fs in ACTIVE_FS if fs in pivot.columns]).fillna(0.0)
    # Sort groups by total frequency (descending) for legibility.
    pivot = (
        pivot.assign(total=pivot.sum(axis=1))
             .sort_values("total", ascending=False)
             .drop(columns="total")
    )

    fig, ax = plt.subplots(figsize=(8, max(4, 0.27 * len(pivot.index))))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax.text(
                j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v < 0.5 else "black", fontsize=7,
            )
    fig.colorbar(im, ax=ax, label="fraction of model runs in FS that selected group")
    ax.set_title("Feature-group selection frequency by FS variant")
    plt.tight_layout()
    plt.show()


## 13. SHAP analysis (best tree model)

Compute SHAP values for the best tree model on the held-out test set using `shap.TreeExplainer` (fast for tree models). The summary beeswarm ranks features by mean absolute SHAP magnitude and shows directionality; the bar chart is a compact alternative when the beeswarm is crowded.


In [ ]:
# §13 — SHAP analysis for the best tree model.
# Pick the lowest-test-MAE tree run, recover its trained model from the stash,
# and compute SHAP values on the test set with TreeExplainer (fast for trees).
import shap
from sklearn.pipeline import Pipeline

_tree_runs = results_df[results_df["family"] == "tree"]
if _tree_runs.empty:
    print("No tree runs to SHAP — skipping.")
else:
    best = _tree_runs.sort_values("test_mae").iloc[0]
    best_name = best["run_name"]
    print(f"Best tree run: {best_name}  (test_mae={best['test_mae']:.4f})")

    final_model = trained_models.get(best_name)
    selected_features = list(best["selected_features"]) if isinstance(best["selected_features"], list) else []

    if final_model is None or not selected_features:
        print(f"Missing final_model or selected_features for {best_name} — skipping SHAP.")
    else:
        # Restrict X_test to the columns the model was actually trained on.
        X_test = exp["X_test"].reindex(columns=selected_features)

        # Some specs wrap the booster in a sklearn Pipeline (e.g. with a scaler).
        # TreeExplainer needs the raw booster + the post-scaler X.
        if isinstance(final_model, Pipeline):
            booster = final_model.named_steps["model"]
            if "scaler" in final_model.named_steps:
                X_for_shap = pd.DataFrame(
                    final_model.named_steps["scaler"].transform(X_test),
                    columns=selected_features, index=X_test.index,
                )
            else:
                X_for_shap = X_test
        else:
            booster = final_model
            X_for_shap = X_test

        explainer = shap.TreeExplainer(booster)
        shap_values = explainer.shap_values(X_for_shap)
        # Some explainers return a list (multi-output); take first if so.
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
        print(f"SHAP values shape: {getattr(shap_values, 'shape', None)}")

        # Beeswarm summary (top 20 features)
        plt.figure()
        shap.summary_plot(shap_values, X_for_shap, max_display=20, show=False)
        plt.title(f"SHAP beeswarm — {best_name}")
        plt.tight_layout()
        plt.show()

        # Bar chart of mean |SHAP|
        plt.figure()
        shap.summary_plot(shap_values, X_for_shap, plot_type="bar",
                          max_display=20, show=False)
        plt.title(f"SHAP mean |value| — {best_name}")
        plt.tight_layout()
        plt.show()


In [ ]:
# Persist results next to the notebook
test_predictions_df.to_parquet(PREDICTIONS_PARQUET)
results_df.drop(columns=["selected_features", "selected_groups", "model_params"]).to_parquet(RESULTS_PARQUET)

print(f"Wrote {PREDICTIONS_PARQUET} ({test_predictions_df.shape})")
print(f"Wrote {RESULTS_PARQUET} ({results_df.shape})")